# CEO Turnover and Executive Pay Dispersion in Europe
**Data & AI in Economics | TU Dortmund**

This notebook develops an analysis-ready panel for studying whether CEO turnover changes CEO compensation and pay dispersion in European firms. It begins with the STOXX Europe 600 universe, retrieves and validates BoardEx and Compustat data, identifies firm-level CEO spells and turnover events, matches CEOs to remuneration records, and combines the resulting CEO pay panel with firm fundamentals, market capitalization, executive characteristics, sector, and country information.

> **Team size:** 2 students  
> **Deliverable:** Jupyter Notebook, extended from proposal to final analysis


<a id="toc"></a>
## Table of Contents

1. [Team](#team)
2. [Research Question](#research-question)
3. [Data Sources and Variables](#data-sources-and-variables)
4. [Setup and Helper Functions](#helper-functions)
5. [Exploratory Data Analysis and Data Preparation of Final Analysis Panel and Executive Pay Dispersion](#exploratory-data-analysis-and-data-preparation-of-final-analysis-panel-and-executive-pay-dispersion)
   - [5.1 Data Collection and Profiling](#data-collection-and-profiling)
      - [5.1.1 STOXX 600 Company Universe](#stoxx-600-company-universe)
      - [5.1.2 Individual Employment Records](#individual-employment-records)
      - [5.1.3 Individuals Profile Details](#individuals-profile-details)
      - [5.1.4 Individual Remuneration Records](#individual-remuneration-records)
      - [5.1.5 Annual Firms Financial Fundamentals](#compustat-fundamentals)
      - [5.1.6 Monthly Security Prices](#monthly-security-prices)
      - [5.1.7 Daily Exchange Rates](#daily-exchange-rates)
   - [5.2 Construction of CEO Turnover and Pay Panel](#construction-of-ceo-turnover-and-pay-panel)
      - [5.2.1 CEO Turnover Events](#ceo-turnover-events)
      - [5.2.2 Prepare Annual Remuneration Records for Identified CEOs](#prepare-annual-remuneration-records)
      - [5.2.3 Match CEO Remuneration to Valid CEO Spells](#ceo-remuneration-matching)
      - [5.2.4 Firm-Year CEO Pay Panel](#firm-year-ceo-pay-panel)
   - [5.3 Firm Financial Fundamentals and Financial Ratios](#firm-financial-fundamentals-and-financial-ratios)
      - [5.3.1 Convert Annual Fundamentals to Nominal EUR](#fundamentals-eur-conversion)
      - [5.3.2 Convert Security Prices to EUR and Merge with Fundamentals](#market-eur-merge)
      - [5.3.3 Calculate Financial Ratios](#financial-ratios)
   - [5.4 Final Analysis Panel](#final-analysis-panel)
      - [5.4.1 Merge CEO Pay, Firm Fundamentals, and Profile Details](#construct-final-panel)
      - [5.4.2 CPI Adjustment and Real 2015 EUR Variables](#cpi-adjustment)
      - [5.4.3 Winsorization of CEO Pay](#winsorization)
      - [5.4.4 Construct CEO-to-Executive Pay Dispersion](#pay-dispersion-construction)
      - [5.4.5 Save the Clean Analysis Panel](#save-panel)
6. [Data Visualizations](#data-visualizations)
   - [6.1 Target Distribution](#target-distribution)
   - [6.2 Annual Pay Trend and Coverage](#annual-pay-trend)
   - [6.3 Pay and Dispersion Around CEO Turnover](#turnover-event-visualization)
   - [6.4 Exploratory Relationships](#exploratory-relationships)
   - [6.5 Mean and Median Pay Across CEO and Firm Groups](#grouped-pay-comparisons)
7. [Causal Inference](#causal-inference)
   - [7.1 Causal Inference: CEO Pay](#causal-inference-results)
   - [7.2 Causal Inference: Pay Dispersion](#causal-inference-pay-dispersion-results)
8. [Supervised Learning](#supervised-learning)
   - [8.1 Prediction Target](#supervised-learning-target)
   - [8.2 Model Sample and Features](#supervised-learning-sample)
   - [8.3 Train/Test Split](#supervised-learning-split)
   - [8.4 Preprocessing Pipeline: Imputation, Encoding, and Scaling](#supervised-learning-preprocessing)
   - [8.5 Tuning Cache](#supervised-learning-tuning-cache)
   - [8.6 CEO Pay Regression Models](#supervised-learning-pay-regression)
   - [8.7 CEO Turnover Classification](#supervised-learning-turnover-classification)
   - [8.8 Pay Dispersion Regression](#supervised-learning-dispersion-regression)
9. [Unsupervised Learning](#unsupervised-learning)
   - [9.1 Objective](#unsupervised-learning-objective)
   - [9.2 Clustering Sample](#unsupervised-learning-sample)
   - [9.3 Preprocessing](#unsupervised-learning-preprocessing)
   - [9.4 PCA Dimensionality Reduction](#unsupervised-learning-pca)
   - [9.5 Choosing the Number of Clusters](#unsupervised-learning-cluster-selection)
   - [9.6 Final Clusters and Profiles](#unsupervised-learning-final-clusters)
   - [9.7 Cluster Interpretation](#unsupervised-learning-interpretation)
   - [9.8 Robustness: Without Turnover Indicators](#unsupervised-learning-robustness)
10. [Results and Discussion](#results-and-discussion)
   - [10.1 Discussion & Conclusion](#discussion-conclusion)
11. [Evaluation Strategy](#evaluation-strategy)
12. [Work Plan](#work-plan)

**Reading guide:** run the notebook from top to bottom. The main empirical dataset is saved in Section 5.4.5 and then reused by the visualization, causal-inference, supervised-learning, and unsupervised-learning sections.


<a id="team"></a>
## 1. Team


| Role | Name | Student ID |
|------|------|------------|
| Lead |Achmad Rizky Akbar| |
| Member | Kajetan Zduńczyk| |


<a id="research-question"></a>
## 2. Mission Title & Research Question


**Title:** *CEO Turnover and Executive Pay Dispersion in Europe: Evidence from Leadership Changes*

**Research question:** *Does CEO turnover causally change CEO compensation levels and pay dispersion within European firms?*

**Why it matters:** *By focusing on leadership changes observed in BoardEx, we can estimate how compensation responds to a governance shock without hand‑collecting policy data. This helps investors and regulators understand whether turnover acts as a disciplining mechanism on executive pay and internal pay gaps.*

<a id="data-sources-and-variables"></a>
## 3. Data Sources and Variables


**Source(s):**

1. Firms in Stoxx 600 Index

    STOXX 600 is a major stock index representing the performance of 600 large-, mid-, and small-capitalization companies across 17 developed European countries.

    https://www.stoxx.com/selection-lists

2. BoardEx - Individual Profile Employment **(boardex.eur_wrds_dir_profile_emp)**

    Includes data about Individual Profile Employment for executives in the BoardEx (Europe) universe.

    https://wrds-www.wharton.upenn.edu/pages/get-data/boardex/boardex-europe/individual-profile/individual-profile-employment/

3. BoardEx - Individual Profile Details **(boardex.eur_dir_profile_details)**

    Includes data about Individual Profile Details for executives in the BoardEx (Europe) universe.

    https://wrds-www.wharton.upenn.edu/pages/get-data/boardex/boardex-europe/individual-profile/individual-profile-details/

4. BoardEx - Annual Remuneration **(boardex.eur_dir_standard_remun)**

    Contains data such as salary, bonus, and other cash compensation
    
    https://wrds-www.wharton.upenn.edu/pages/get-data/boardex/boardex-europe/compensation-analysis/annual-remuneration/

5. Compustat Global – Fundamentals Annual **(comp_global_daily.g_funda)**

    Provides firm-year controls.

    https://wrds-www.wharton.upenn.edu/pages/get-data/compustat-capital-iq-standard-poors/compustat/global-daily/fundamentals-annual/

6. Compustat Global - Security Monthly **(comp.g_secm)**

    Includes monthly security prices from various exchanges.

    https://wrds-www.wharton.upenn.edu/pages/get-data/compustat-capital-iq-standard-poors/compustat/global-daily/security-monthly/#dataDictionaryTab

7. Compustat Global - daily updates **(comp.g_exrt_dly)**

    Contains daily exchange rates data.

    https://wrds-www.wharton.upenn.edu/dynamic-query-form/comp_global_daily/g_exrt_dly/
    

**Table grains:** Employment data are role-spell records, remuneration data are executive-year records, fundamentals are firm-year records, and market prices are firm-security-month records. The final analysis table is reduced to one selected CEO observation per firm-year.

**Key variables:**

| Variable | Type | Role (feature / target / instrument / ...) | Description |
|----------|------|---------------------------------------------|-------------|
| Total CEO compensation | Numeric | **Target** | Total annual compensation retained in reported currency, nominal EUR, and real 2015 EUR |
| Pay dispersion | Numeric | **Target / Outcome** | CEO pay relative to top-executive team (e.g., CEO-to-top-5 ratio) |
| CEO turnover indicator | Binary | **Treatment** | Flag for CEO change in a given year (from BoardEx role start/end) |
| Post‑turnover period | Binary | Feature | Indicator for years after turnover (event window) |
| Firm size (log assets / market cap) | Numeric | Feature | Scale and visibility of firm |
| Profitability (ROA / EBIT margin) | Numeric | Feature | Performance controls |
| Leverage | Numeric | Feature | Capital structure |
| Industry & country fixed effects | Categorical | Feature | Sector and institutional context |

**Outcomes and controls:**

The preparation pipeline constructs CEO compensation, pay dispersion, and turnover variables, converts monetary levels to nominal EUR, and deflates them to country-specific real 2015 EUR. The final modelling stages use CEO pay and pay dispersion as complementary compensation outcomes.

**Potential data quality issues:**  
- **Missing compensation components:** use multiple imputation or restrict to firms with complete pay breakdowns; report sensitivity to this choice.
- **Turnover date ambiguity:** define CEO change using role start/end dates and validate with overlapping roles; conduct robustness with alternative windows.
- **Reporting bias / top-coding:** winsorize extreme pay values; compare distributions by country to detect systematic reporting differences.
- **Selection bias in BoardEx coverage:** include a Stoxx 600 filter and check representativeness vs. population benchmarks.
- **Timing misalignment:** align fiscal-year fundamentals with compensation year; drop or lag inconsistent observations.
- **Currency and inflation effects:** implemented through date-aligned EUR exchange rates followed by headquarters-country CPI rebased to 2015 = 100; observations outside CPI coverage remain missing.

---

<a id="helper-functions"></a>
## 4. Setup and Helper Functions

This section defines reusable display, timing, connection, and caching utilities. Keeping these operations in one place makes the data pipeline easier to rerun and audit.


In [53]:
# Core packages and project paths
import wrds
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import time
import json
import lightgbm as lgb
import xgboost as xgb
import optuna
import joblib
import re
import pycountry
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.tree import DecisionTreeRegressor, ExtraTreeRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

optuna.logging.set_verbosity(optuna.logging.WARNING)



Matplotlib is building the font cache; this may take a moment.


In [2]:
import warnings

def fxn():
    warnings.warn("deprecated", DeprecationWarning)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    fxn()

# Or if you are using > Python 3.11:
with warnings.catch_warnings(action="ignore"):
    fxn()


In [3]:
def get_wrds_connection():
    """Create the WRDS connection only when a server query is actually needed."""
    global conn

    try:
        conn
        print("Using existing WRDS connection.")
    except NameError:
        conn = wrds.Connection()
        print("Created new WRDS connection.")

    return conn

In [4]:
# show all columns and rows in DataFrame outputs
pd.set_option('display.max_columns', None)

In [5]:
def summarize_dataframe(df_input):
    """Display and return a dtype-safe health summary for a DataFrame or data file.

    Supports CSV, Excel, pickle, and Parquet files. Categorical statistics are
    calculated explicitly so pandas `string`, `object`, and `category` dtypes
    are handled consistently.
    """
    if isinstance(df_input, (str, Path)):
        path = Path(df_input)
        readers = {
            ".csv": pd.read_csv,
            ".xlsx": pd.read_excel,
            ".xls": pd.read_excel,
            ".pkl": pd.read_pickle,
            ".pickle": pd.read_pickle,
            ".parquet": pd.read_parquet,
        }
        if path.suffix.lower() not in readers:
            raise ValueError(
                f"Unsupported file extension: {path.suffix or '<none>'}. "
                f"Supported extensions: {sorted(readers)}"
            )
        df = readers[path.suffix.lower()](path)
        print(f"Loaded: {path}")
    elif isinstance(df_input, pd.DataFrame):
        df = df_input
    else:
        raise TypeError("df_input must be a pandas DataFrame or a supported file path.")

    if df.empty:
        print(f"Empty DataFrame with {df.shape[1]} columns.")
        return {"overview": pd.DataFrame(), "table_overview": pd.DataFrame(), "missing": pd.DataFrame()}

    try:
        duplicate_rows = int(df.duplicated().sum())
    except TypeError:
        duplicate_rows = pd.NA

    overview = pd.DataFrame({
        "value": [
            len(df),
            df.shape[1],
            duplicate_rows,
            f"{df.memory_usage(deep=True).sum() / 1024**2:.2f} MB",
        ]
    }, index=["rows", "columns", "duplicate rows", "memory usage"])

    table_overview = df.head(3)

    missing = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(dropna=True),
    }).sort_values(["missing_pct", "unique"], ascending=[False, False])

    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    categorical_cols = df.select_dtypes(
        include=["object", "string", "category", "bool"]
    ).columns.tolist()
    datetime_cols = df.select_dtypes(
        include=["datetime", "datetimetz"]
    ).columns.tolist()

    numeric_summary = (
        df[numeric_cols].describe().T if numeric_cols else pd.DataFrame()
    )

    categorical_rows = []
    for col in categorical_cols:
        counts = df[col].value_counts(dropna=True)
        categorical_rows.append({
            "column": col,
            "count": int(df[col].notna().sum()),
            "missing": int(df[col].isna().sum()),
            "unique": int(df[col].nunique(dropna=True)),
            "top": counts.index[0] if not counts.empty else pd.NA,
            "top_frequency": int(counts.iloc[0]) if not counts.empty else 0,
        })
    categorical_summary = (
        pd.DataFrame(categorical_rows).set_index("column")
        if categorical_rows else pd.DataFrame()
    )

    datetime_summary = pd.DataFrame()
    if datetime_cols:
        datetime_summary = pd.DataFrame({
            "min": df[datetime_cols].min(),
            "max": df[datetime_cols].max(),
            "missing": df[datetime_cols].isna().sum(),
        })

    print("### Dataset overview")
    display(overview)
    print("### Table overview")
    display(table_overview)
    print("### Column coverage and dtypes")
    display(missing)
    if not numeric_summary.empty:
        print("### Numeric columns")
        display(numeric_summary)
    if not categorical_summary.empty:
        print("### Categorical columns")
        display(categorical_summary)
    if not datetime_summary.empty:
        print("### Datetime columns")
        display(datetime_summary)

    return {
        "overview": overview,
        "table_overview": table_overview,
        "missing": missing,
        "numeric": numeric_summary,
        "categorical": categorical_summary,
        "datetime": datetime_summary,
    }


---

<a id="exploratory-data-analysis-and-data-preparation-of-final-analysis-panel-and-executive-pay-dispersion"></a>
# 5. Exploratory Data Analysis and Data Preparation of Final Analysis Panel and Executive Pay Dispersion

<a id="data-collection-and-profiling"></a>
## 5.1 Data Collection and Profiling

The data pipeline follows a common STOXX 600 sample frame:

1. STOXX 600 ISINs define the firm universe.
2. BoardEx employment records provide CEO roles and turnover dates.
3. BoardEx executive details provide demographic characteristics.
4. BoardEx remuneration records provide annual compensation.
5. Compustat annual fundamentals provide firm-year controls.
6. Compustat monthly security data provide prices for market capitalization.
7. Compustat daily exchange rates provide the currency bridge to EUR.

Each query selects only columns used by the preparation or modelling pipeline. This reduces WRDS transfer time, cache size, and in-memory DataFrame size while keeping identifiers needed for validation. The sources are combined only after their identifiers, dates, currencies, and table grains have been checked.


<a id="stoxx-600-company-universe"></a>
### 5.1.1 STOXX 600 Company Universe


Before querying BoardEx, we define the project sample. The analysis is restricted to firms in the STOXX Europe 600. We use the `stoxx600_clean.csv` file as the sample frame and extract its ISINs as the main identifier for WRDS queries.

In [6]:
# Define paths to the project data folder and STOXX 600 company file
DATA_DIR = Path("data")
STOXX_FILE = DATA_DIR / "stoxx600_clean.csv"

In [7]:
# Read STOXX Europe 600 constituents
df_sxxp = pd.read_csv(STOXX_FILE, sep=";", usecols=["ISIN", "Instrument_Name", "Country"])

# Summary statistics
_ = summarize_dataframe(df_sxxp)

### Dataset overview


,value
rows,600
columns,3
duplicate rows,0
memory usage,0.10 MB


### Table overview


,ISIN,Instrument_Name,Country
0,NL0010273215,ASML HLDG,NL
1,GB0005405286,HSBC,GB
2,GB0009895292,ASTRAZENECA,GB


### Column coverage and dtypes


,dtype,non_null,missing,missing_pct,unique
ISIN,object,600,0,0.0,600
Instrument_Name,object,600,0,0.0,600
Country,object,600,0,0.0,17


### Categorical columns


,count,missing,unique,top,top_frequency
column,,,,,
ISIN,600,0,600,NL0010273215,1
Instrument_Name,600,0,600,ASML HLDG,1
Country,600,0,17,GB,128


| Column | Precise description | WRDS/pandas representation |
|---|---|---|
| `ISIN` | Twelve-character International Securities Identification Number used to restrict all server queries to the STOXX Europe 600 universe. | String |
| `Instrument_Name` | Constituent Name | String |
| `Country` | Constituent ISO currency code | String |

**Insights:**

- The summary shows that there exists no duplicated ISIN. Hence, it will not create repeated query keys.


These ISINs are the bridge from the index universe to BoardEx. All following BoardEx employment and remuneration pulls should be filtered using this STOXX 600 firm list. The values are standardized before they are passed to WRDS.

<a id="individual-employment-records"></a>
### 5.1.2 Individual Employment Records


The STOXX 600 file defines the company universe for the project. We use the ISINs from `df_sxxp` as the filter for BoardEx Europe employment records, so all later CEO turnover and compensation variables are built only for firms that are members of the STOXX 600 sample.

This query retains:

- the individual,
- company,
- role-spell,
- and firm-context fields.

Role start and end dates define CEO spells; `directorid` identifies CEO changes; `companyid` and `isin` provide merge keys; and sector/country fields become final-panel controls.

In [8]:
# Get connection to WRDS
conn = get_wrds_connection()

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Created new WRDS connection.


In [9]:
# Clean ISIN codes so they match correctly in later queries and merges
isin_tuple = tuple(df_sxxp["ISIN"].astype(str).str.strip().str.upper())

# Get employment records
df_emp = conn.raw_sql(
    """
    SELECT
    companyid, companyname, directorid, directorname,
    datestartrole, dateendrole, rolename,
    hocountryname, sector, isin
    FROM boardex.eur_wrds_dir_profile_emp
    WHERE isin IN %(isins)s
    ORDER BY companyname, directorname, datestartrole, rolename
    """,
    params={"isins": isin_tuple}
)

# Summary statistics
_ = summarize_dataframe(df_emp)

### Dataset overview


,value
rows,182894
columns,10
duplicate rows,0
memory usage,93.51 MB


### Table overview


,companyid,companyname,directorid,directorname,datestartrole,dateendrole,rolename,hocountryname,sector,isin
0,294.0,3I GROUP PLC,1348931.0,Aaron Church,2013-07-01,2022-04-28,Director - Infrastructure,United Kingdom - England,Private Equity,GB00B1YW4409
1,294.0,3I GROUP PLC,1348931.0,Aaron Church,2022-04-01,9000-01-01,Partner,United Kingdom - England,Private Equity,GB00B1YW4409
2,294.0,3I GROUP PLC,772614.0,Adam Workman,2000-09-01,2002-01-28,Investment Associate,United Kingdom - England,Private Equity,GB00B1YW4409


### Column coverage and dtypes


,dtype,non_null,missing,missing_pct,unique
directorid,Float64,182894,0,0.0,89240
directorname,string,182894,0,0.0,88043
rolename,string,182894,0,0.0,11102
datestartrole,string,182894,0,0.0,7793
dateendrole,string,182894,0,0.0,7405
companyname,string,182894,0,0.0,586
companyid,Float64,182894,0,0.0,567
isin,string,182894,0,0.0,567
sector,string,182894,0,0.0,39
hocountryname,string,182894,0,0.0,27


### Numeric columns


,count,mean,std,min,25%,50%,75%,max
companyid,182894.0,227796.831126,640732.817089,294.0,9074.0,22335.0,30558.0,4066835.0
directorid,182894.0,1388871.352833,909953.143927,28.0,555497.0,1325964.5,2151240.0,3360920.0


### Categorical columns


,count,missing,unique,top,top_frequency
column,,,,,
companyname,182894,0,586,DEUTSCHE BANK AG,4896
directorname,182894,0,88043,Doctor Roland Busch,31
datestartrole,182894,0,7793,1900-01-01,16724
dateendrole,182894,0,7405,9000-01-01,20149
rolename,182894,0,11102,Various Positions,13442
hocountryname,182894,0,27,Germany,33983
sector,182894,0,39,Banks,35790
isin,182894,0,567,DE0005140008,4896


**Table description:** `df_emp` has one row per BoardEx employment-role spell. It contains the minimum fields required to identify firm-level CEOs, order their spells through time, construct turnover events, link people and firms to other sources, and attach sector and headquarters-country controls.


| Column | Precise description | WRDS type |
|---|---|---|
| `companyid` | BoardEx company identifier used to group CEO spells and merge remuneration records (`boardid`). | Float |
| `companyname` | BoardEx company name retained for readable diagnostics and outputs. | Char(128) |
| `directorid` | Stable BoardEx individual identifier used to identify consecutive CEOs and merge executive/remuneration records. | Float |
| `directorname` | Individual name retained for readable CEO-spell and transition checks. | Char(255) |
| `datestartrole` | First date of the employment-role spell; determines CEO ordering and turnover year. | Date |
| `dateendrole` | Last date of the role spell; open-ended sentinel dates are cleaned later. | Date |
| `rolename` | Reported role title used to identify and refine firm-level CEO positions. | Char(100) |
| `hocountryname` | Company headquarters country used as a country control in the final panel. | Char(255) |
| `sector` | BoardEx company-sector classification used as an industry control. | Char(100) |
| `isin` | International Securities Identification Number linking BoardEx firms to the STOXX and Compustat samples. | Char(12) |

**Insights:**
- The employment table is complete: all 140,892 records have director, role, company, sector, country, and ISIN information.

- There are no duplicate rows, so the raw BoardEx employment extract is clean at the role-record level.

- The table covers 64,458 unique directors across 567 matched firms, giving broad coverage of executive and board role histories in the STOXX 600 sample.

- Role names are highly granular, with 9,961 unique values, so CEO identification requires keyword-based cleaning rather than relying on one standardized title.

- Several date values are placeholders rather than true calendar dates, especially `1900-01-01` for role starts and `9000-01-01` for ongoing roles, so these dates need cleaning before constructing CEO spells or turnover events.

- Coverage is uneven across firms and sectors: Deutsche Bank and the banking sector have especially many role records, which likely reflects larger boards, richer reporting, or longer employment histories.

- BoardEx matches slightly fewer firms than the 600 STOXX constituents.

- The employment table is the backbone of the turnover design: it supplies `directorid`, `companyid`, role titles, and start/end dates.

- The most important quality check here is whether CEO-like role titles are too broad; later filters narrow regional or divisional CEO roles into firm-level CEO spells.


<a id="individuals-profile-details"></a>
### 5.1.3 Individuals Profile Details

For the individual profile details, only `directorid`, `age`, and `gender` will be utilized. Date of birth and death are retained for diagnostic purposes.

In [10]:
# Clean directorid so they match correctly in later queries and merges
directorid_tuple = tuple(df_emp["directorid"].dropna().astype(int).unique().tolist())

# Get individual profile records
df_profile = conn.raw_sql(
    """
    SELECT directorid, age, gender, dob, dod
    FROM boardex.eur_dir_profile_details
    WHERE directorid IN %(directorids)s
    ORDER BY directorname
    """,
    params={"directorids": directorid_tuple}
)

# Summary statistics
_ = summarize_dataframe(df_profile)

### Dataset overview


,value
rows,89240
columns,5
duplicate rows,0
memory usage,19.57 MB


### Table overview


,directorid,age,gender,dob,dod
0,2012627.0,43,M,1981-01-01,9999-12-31
1,2640395.0,<NA>,M,1900-01-01,9999-12-31
2,15806.0,90,M,1936-05-28,9999-12-31


### Column coverage and dtypes


,dtype,non_null,missing,missing_pct,unique
age,string,53984,35256,39.51,79
directorid,Float64,89240,0,0.00,89240
dob,string,89240,0,0.00,11606
dod,string,89240,0,0.00,1302
gender,string,89237,3,0.00,2


### Numeric columns


,count,mean,std,min,25%,50%,75%,max
directorid,89240.0,1555563.743277,913774.019343,28.0,835907.75,1541364.0,2311178.75,3360920.0


### Categorical columns


,count,missing,unique,top,top_frequency
column,,,,,
age,53984,35256,79,60,2209
gender,89237,3,2,M,70166
dob,89240,0,11606,1900-01-01,35257
dod,89240,0,1302,9999-12-31,87788


| Column | Precise description | WRDS type |
|---|---|---|
| `directorid` | BoardEx individual identifier used to merge the profile onto CEO records. | Float |
| `age` | Director age reported by BoardEx at the time of data extraction. It is treated as an executive characteristic, not a historical age-by-year measure. | Char(3) |
| `gender` | One-character gender classification reported by BoardEx. | Char(1) |
|`dob` | Date of birth | Date |
|`dod` | Date of death | Date |

<a id="individual-remuneration-records"></a>
### 5.1.4 Individual Remuneration Records

We retrieve remuneration for all matched individuals rather than CEOs alone. The selected fields cover identifiers, reporting date and currency, core cash compensation, and the equity/performance measures retained in the CEO-pay panel. The broader individual population is also used to construct the within-firm pay-dispersion outcome.

In [11]:
# Clean directorid so they match correctly in later queries and merges
directorid_tuple = tuple(df_emp["directorid"].dropna().astype(int).unique().tolist())

# Get individual remuneration records
df_remun = conn.raw_sql(
    """
    SELECT
    boardid, directorid,
    annualreportdate, currency, salary, bonus, totalcompensation,
    valtoteqheld, valltipheld, valeqaward, ltipvalue, toteqatrisk, perftotal
    FROM boardex.eur_dir_standard_remun
    WHERE directorid IN %(directorids)s
    ORDER BY boardid, directorid, annualreportdate
    """,
    params={"directorids": directorid_tuple}
)

# Convert data type annualreportdate to date for df_remun
df_remun["annualreportdate"] = pd.to_datetime(df_remun["annualreportdate"], errors="coerce")

# Summary statistics
_ = summarize_dataframe(df_remun)

### Dataset overview


/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/3834996771.py:19: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_remun["annualreportdate"] = pd.to_datetime(df_remun["annualreportdate"], errors="coerce")


,value
rows,299420
columns,13
duplicate rows,2224
memory usage,45.40 MB


### Table overview


,boardid,directorid,annualreportdate,currency,salary,bonus,totalcompensation,valtoteqheld,valltipheld,valeqaward,ltipvalue,toteqatrisk,perftotal
0,296.0,511799.0,2005-12-01,USD,21.0,<NA>,21.0,584.0,<NA>,<NA>,<NA>,<NA>,<NA>
1,296.0,511799.0,2006-12-01,USD,24.0,<NA>,24.0,811.0,<NA>,<NA>,<NA>,<NA>,<NA>
2,296.0,511799.0,2007-12-01,USD,27.0,<NA>,27.0,779.0,<NA>,<NA>,<NA>,<NA>,<NA>


...

,count,mean,std,min,25%,50%,75%,max
boardid,299420.0,505189.147956,901358.064293,296.0,12154.0,26246.0,640371.0,4066835.0
directorid,299420.0,683560.084199,741873.053435,28.0,24256.0,484845.5,1193409.0,3358973.0
salary,56745.0,304.432532,521.81514,0.0,54.0,108.0,276.0,19076.0
bonus,14797.0,768.075488,1035.961397,0.0,82.0,367.0,1101.0,17870.0
totalcompensation,77379.0,370.129234,904.810615,0.0,0.0,77.0,195.0,20085.0
valtoteqheld,32042.0,51104.210786,1224533.981376,0.0,39.0,150.0,933.0,79338242.0
valltipheld,8503.0,6861.641773,25849.821999,0.0,623.0,2132.0,6141.5,1163693.0
valeqaward,380.0,689.157895,2508.624256,0.0,77.0,177.5,261.5,34572.0
ltipvalue,8180.0,2204.070905,7759.554126,0.0,238.0,931.5,2278.25,347474.0
toteqatrisk,9718.0,2260.593229,7472.281063,0.0,219.0,902.0,2328.0,347474.0


### Categorical columns


,count,missing,unique,top,top_frequency
column,,,,,
currency,299420,0,1,USD,299420


### Datetime columns


,min,max,missing
annualreportdate,1996-12-01,2026-06-01,17070


| Column | Precise description | WRDS type / unit |
|---|---|---|
| `boardid` | BoardEx company identifier; matched to employment `companyid`. | Float |
| `directorid` | BoardEx individual identifier linking remuneration to CEO spells. | Float |
| `annualreportdate` | Annual-report date associated with the remuneration observation; its year defines the pay-panel year. | Date |
| `currency` | Three-character currency code supplied by BoardEx for the monetary fields. | Char(3) |
| `salary` | Annual base salary reported by BoardEx. | Float; USD thousands in the current extract |
| `bonus` | Annual cash bonus reported by BoardEx. | Float; USD thousands in the current extract |
| `totalcompensation` | Main CEO-pay outcome: total annual compensation reported by BoardEx. | Float; thousands |
| `perftotal` | LTIP value divided by total awards for the remuneration period. | Float ratio |
| `valtoteqheld` | Total value of equity held by the executive. | Float; thousands |
| `valltipheld` | Value of long-term incentive plans held. | Float; thousands |
| `valeqaward` | Value of equity awarded during the latest year. | Float; thousands |
| `ltipvalue` | Value of LTIP awards granted during the latest year. | Float; thousands |
| `toteqatrisk` | Total value of stock, option, and LTIP awards at risk. | Float; thousands |

**Insights:**
- The remuneration table is large, with 275,806 records, but it is not complete for all compensation variables.

- There are 2,008 duplicate rows, so duplicate handling is needed before constructing the CEO compensation panel.

- The key outcome for this study, `totalcompensation`, is available for 67,440 records, meaning about one quarter of the raw remuneration records contain usable total pay information.

- `salary` and `bonus` are much less complete; this supports using total compensation as the main pay measure.

- Equity- and long-term incentive variables have very high missingness, often above 90%, so they are not reliable as core outcomes in this study.

- All remuneration values are reported in one raw currency field (`USD`).

<a id="compustat-fundamentals"></a>
### 5.1.5 Annual Firms Financial Fundamentals

Compustat fundamentals add the annual firm controls used in `df_panel_final`. The compact query retains identifiers and dates, accounting currency, the numerators and denominators required for profitability and leverage ratios, employment, and shares outstanding for market capitalization.

In [12]:
# Get annual firms financial fundamentals records using isin tuple
df_firm = conn.raw_sql(
    """
    SELECT
    gvkey, isin, datadate, curcd,
    at, sale, ebit, dltt, nicon, emp, cshoi
    FROM comp_global_daily.g_funda
    WHERE isin IN %(isins)s
    ORDER BY isin, fyear, datadate
    """,
    params={"isins": isin_tuple}
)

# Summary statistics
_ = summarize_dataframe(df_firm)

### Dataset overview


,value
rows,16350
columns,11
duplicate rows,0
memory usage,4.52 MB


### Table overview


,gvkey,isin,datadate,curcd,at,sale,ebit,dltt,nicon,emp,cshoi
0,272817,AT0000606306,2003-12-31,EUR,20060.255,<NA>,332.687,620.895,178.733,17.299,50.0
1,272817,AT0000606306,2004-12-31,EUR,28907.122,<NA>,414.062,826.594,210.462,22.851,62.5
2,272817,AT0000606306,2005-12-31,EUR,40695.0,<NA>,621.0,1340.0,382.0,43.6,142.58


### Column coverage and dtypes


,dtype,non_null,missing,missing_pct,unique
sale,Float64,12113,4237,25.91,11728
nicon,Float64,13406,2944,18.01,10544
emp,Float64,14977,1373,8.40,11969
cshoi,Float64,15245,1105,6.76,11742
ebit,Float64,16164,186,1.14,13506
dltt,Float64,16179,171,1.05,13855
at,Float64,16311,39,0.24,16040
curcd,string,16336,14,0.09,22
gvkey,string,16350,0,0.00,557
isin,string,16350,0,0.00,557


### Numeric columns


,count,mean,std,min,25%,50%,75%,max
at,16311.0,716968.992262,11346843.026168,0.0,3159.05,11864.0,54919.7,643358816.0
sale,12113.0,146854.538712,2642424.55592,-3518.0,1931.6,6185.9,21526.0,100117000.0
ebit,16164.0,19956.228761,312519.218501,-1652000.0,194.84725,659.772,2637.225,10426000.0
dltt,16179.0,78420.583106,1554747.60725,0.0,308.471,1891.9,7796.8,101852807.0
nicon,13406.0,4100.687279,109105.680251,-37500.0,96.0,355.9025,1332.0,7748000.0
emp,14977.0,39.828477,68.458763,0.0,3.707,14.0,45.95,684.025
cshoi,15245.0,1149.31836,9191.041952,0.0,55.065,205.394,728.835,523438.445


### Categorical columns


,count,missing,unique,top,top_frequency
column,,,,,
gvkey,16350,0,557,208331,44
isin,16350,0,557,GB00B62G9D36,44
datadate,16350,0,397,2025-12-31,489
curcd,16336,14,22,EUR,6751


| Column | Precise description | Compustat representation |
|---|---|---|
| `gvkey` | Stable Compustat company identifier used to connect annual fundamentals to monthly security prices. | Char |
| `isin` | Security identifier used to connect Compustat fundamentals to the STOXX/BoardEx firm sample. | Char |
| `datadate` | Fiscal reporting date used to attach the nearest monthly market price within 31 days. | Date |
| `curcd` | Currency in which the annual accounting values are reported. | Char |
| `at` | Total assets; denominator for ROA and leverage and input to log assets. | Numeric |
| `sale` | Net sales/turnover; denominator for EBIT margin. | Numeric |
| `ebit` | Earnings before interest and taxes; numerator for EBIT margin. | Numeric |
| `dltt` | Long-term debt; numerator for the leverage ratio. | Numeric |
| `nicon` | Consolidated net income/loss; numerator for ROA. | Numeric |
| `emp` | Number of employees, generally reported in thousands by Compustat. | Numeric |
| `cshoi` | Common shares outstanding for the issue, used with monthly price to calculate market capitalization. | Numeric |

**Insights:**

- The firm fundamentals table is compact and clean, with 16,350 firm-year records and no duplicate rows.

- The table covers 557 matched firms.

- Core identifiers and timing variables are complete: `gvkey`, `isin`, `datadate`, have no missing values.

- Balance sheet and profitability variables are mostly complete, especially `at`, `ebit`, and `dltt`, which makes them suitable as firm-level controls.

- Sales has the largest missingness among the main accounting variables at 25.91%, so it should be used carefully or treated as a secondary control.

- Firm size and financial variables are highly skewed, with very large maximum values, so log transformations and winsorization are useful before modeling.

- The table reports values in 17 currencies, with EUR most common, so currency conversion is required before comparing firms across countries.

<a id="monthly-security-prices"></a>
### 5.1.6 Monthly Security Prices

Monthly security prices are retrieved for the `gvkey` values present in annual financial fundamentals. The compact table keeps only the firm and issue identifiers, observation date, closing price, price currency, exchange code, and exchange description needed to select one listing and construct market capitalization. Its grain is one security issue per month.

In [13]:
# Clean gvkey
gvkey_tuple = tuple(
    df_firm["gvkey"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
    .unique()
    .tolist()
)

# Select one exchange and one security issue per gvkey using price coverage
df_market = conn.raw_sql(
    """
    WITH market_raw AS (
        SELECT
            gvkey, iid, datadate, prccm, curcdm, exchg
        FROM comp.g_secm
        WHERE gvkey IN %(gvkeys)s
    ),

    exchange_quality AS (
        SELECT
            gvkey,
            exchg,
            COUNT(*) AS n_rows,
            COUNT(prccm) AS non_missing_prccm,
            SUM(CASE WHEN prccm IS NULL THEN 1 ELSE 0 END) AS missing_prccm,
            MAX(datadate) AS last_datadate,
            ROW_NUMBER() OVER (
                PARTITION BY gvkey
                ORDER BY
                    SUM(CASE WHEN prccm IS NULL THEN 1 ELSE 0 END) ASC,
                    COUNT(prccm) DESC,
                    MAX(datadate) DESC,
                    COUNT(*) DESC,
                    exchg ASC
            ) AS exchange_rank
        FROM market_raw
        GROUP BY gvkey, exchg
    ),

    best_exchange AS (
        SELECT gvkey, exchg
        FROM exchange_quality
        WHERE exchange_rank = 1
    ),

    market_best_exchange AS (
        SELECT a.*
        FROM market_raw a
        INNER JOIN best_exchange e
            ON a.gvkey = e.gvkey
           AND a.exchg = e.exchg
    ),

    iid_quality AS (
        SELECT
            gvkey,
            iid,
            COUNT(*) AS n_rows,
            COUNT(prccm) AS non_missing_prccm,
            SUM(CASE WHEN prccm IS NULL THEN 1 ELSE 0 END) AS missing_prccm,
            MIN(datadate) AS first_datadate,
            MAX(datadate) AS last_datadate,
            ROW_NUMBER() OVER (
                PARTITION BY gvkey
                ORDER BY
                    SUM(CASE WHEN prccm IS NULL THEN 1 ELSE 0 END) ASC,
                    COUNT(prccm) DESC,
                    MAX(datadate) DESC,
                    COUNT(*) DESC,
                    iid ASC
            ) AS iid_rank
        FROM market_best_exchange
        GROUP BY gvkey, iid
    ),

    best_iid AS (
        SELECT gvkey, iid
        FROM iid_quality
        WHERE iid_rank = 1
    )

    SELECT
        a.gvkey,
        a.iid,
        a.datadate,
        a.prccm,
        a.curcdm,
        a.exchg,
        b.exchgdesc
    FROM market_best_exchange a
    INNER JOIN best_iid i
        ON a.gvkey = i.gvkey
       AND a.iid = i.iid
    LEFT JOIN comp_na_daily_all.r_ex_codes b
        ON a.exchg = b.exchgcd
    ORDER BY a.gvkey, a.datadate
    """,
    params={"gvkeys": gvkey_tuple}
)

# Summary statistics
_ = summarize_dataframe(df_market)

### Dataset overview


,value
rows,113152
columns,7
duplicate rows,0
memory usage,32.67 MB


### Table overview


,gvkey,iid,datadate,prccm,curcdm,exchg,exchgdesc
0,001166,01W,2007-01-31,17.58,EUR,104,NYSE Euronext Amsterdam
1,001166,01W,2007-02-28,17.43,EUR,104,NYSE Euronext Amsterdam
2,001166,01W,2007-03-31,16.65,EUR,104,NYSE Euronext Amsterdam


### Column coverage and dtypes


,dtype,non_null,missing,missing_pct,unique
prccm,Float64,112997,155,0.14,33593
gvkey,string,113152,0,0.00,553
datadate,string,113152,0,0.00,233
exchg,Int64,113152,0,0.00,24
exchgdesc,string,113152,0,0.00,24
iid,string,113152,0,0.00,20
curcdm,string,113152,0,0.00,10


### Numeric columns


,count,mean,std,min,25%,50%,75%,max
prccm,112997.0,132.113003,540.767052,0.0154,10.35,33.05,95.33,22580.0
exchg,113152.0,194.144885,54.519416,104.0,154.0,192.0,209.0,349.0


### Categorical columns


,count,missing,unique,top,top_frequency
column,,,,,
gvkey,113152,0,553,001166,233
iid,113152,0,20,01W,67444
datadate,113152,0,233,2026-05-31,549
curcdm,113152,0,10,EUR,65139
exchgdesc,113152,0,24,London Stock Exchange,23895


| Column | Precise description | Compustat representation |
|---|---|---|
| `gvkey` | Compustat company identifier linking monthly prices to annual fundamentals. | Char |
| `iid` | Compustat issue identifier distinguishing multiple securities issued by the same company. | Char |
| `datadate` | Month-end date of the security-price observation. | Date |
| `prccm` | Monthly closing price in the currency identified by `curcdm`. | Numeric |
| `curcdm` | ISO currency code applicable to `prccm`. | Char |
| `exchg` | Compustat exchange code used to select one primary price history per `gvkey`. | Integer |
| `exchgdesc` | Exchange description | Char |

**Insights:**

- The market table contains 125,246 security-month observations for 553 firms and has no duplicate rows.

- Price coverage is very strong: only 191 prccm values are missing (0.15%), so monthly market prices are usable for constructing market capitalization.

- Market prices are reported in 11 currencies, with EUR most common, so currency conversion is still required before comparing firms across countries.

- Raw share prices are highly skewed, ranging from 0.001 to 22,580, so later analysis should use market capitalization or transformed variables rather than raw prccm levels.

<a id="daily-exchange-rates"></a>
### 5.1.7 Daily Exchange Rates

The analysis requires a common currency for monetary levels. Source currencies are derived directly from remuneration, annual fundamentals, and monthly prices. Compustat stores these observations as GBP-based cross rates, so the query retrieves only GBP-to-required-currency series and the function derives each source-currency-to-EUR rate as `(GBP→EUR) / (GBP→source currency)`. This avoids loading the full cross-currency matrix. The resulting table grain is one source-currency/date conversion rate to EUR.

In [14]:
# Retrieve only currencies observed in the monetary source tables
currency_list = sorted(
    set(df_remun["currency"].dropna().astype(str))
    | set(df_firm["curcd"].dropna().astype(str))
    | set(df_market["curcdm"].dropna().astype(str))
)

target_currencies = tuple(sorted(set(currency_list) | {"EUR", "GBP"}))

# Convert Compustat GBP cross-rates into EUR rates
df_fx = conn.raw_sql(
    """
    WITH gbp_rates AS (
        SELECT
            datadate,
            tocurd,
            exratd
        FROM comp.g_exrt_dly
        WHERE fromcurd = 'GBP'
          AND tocurd IN %(currencies)s
    ),

    eur_reference AS (
        SELECT
            datadate,
            exratd AS gbp_to_eur
        FROM gbp_rates
        WHERE tocurd = 'EUR'
    )

    SELECT
        r.datadate,
        r.tocurd AS fromcurd,
        'EUR' AS tocurd,
        e.gbp_to_eur / r.exratd AS exratd
    FROM gbp_rates r
    INNER JOIN eur_reference e
        ON r.datadate = e.datadate
    ORDER BY r.datadate, r.tocurd
    """,
    params={"currencies": target_currencies}
)

# Convert data type datadate to date for df_fx
df_fx["datadate"] = pd.to_datetime(df_fx["datadate"], errors="coerce")

# Summary statistics
_ = summarize_dataframe(df_fx)

### Dataset overview


/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/3762959625.py:45: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_fx["datadate"] = pd.to_datetime(df_fx["datadate"], errors="coerce")


,value
rows,272827
columns,4
duplicate rows,0
memory usage,31.48 MB


### Table overview


,datadate,fromcurd,tocurd,exratd
0,1985-12-31,ATS,EUR,0.065127
1,1985-12-31,BEF,EUR,0.022244
2,1985-12-31,CHF,EUR,0.54396


...

,count,mean,std,min,25%,50%,75%,max
exratd,272827.0,0.355913,0.416184,0.000429,0.063609,0.13467,0.5113,1.7525


### Categorical columns


,count,missing,unique,top,top_frequency
column,,,,,
fromcurd,272827,0,23,HKD,13677
tocurd,272827,0,1,EUR,272827


### Datetime columns


,min,max,missing
datadate,1985-12-31,2026-06-28,0


| Column | Precise description | Compustat representation |
|---|---|---|
| `datadate` | Calendar date to which the daily exchange rate applies. | Date |
| `fromcurd` | ISO code of the monetary variable's original currency. | Char |
| `tocurd` | Target currency; fixed to EUR by this query. | Char |
| `exratd` | Units of EUR obtained for one unit of `fromcurd`; multiply the original amount by this rate to express it in EUR. | Numeric |

**Insights:**

- The FX table contains **272,791 daily currency-rate observations** and has **no duplicate rows**, so it is clean at the currency-date level.

- All key fields are complete: `datadate`, `fromcurd`, `tocurd`, and `exratd` have **0 missing values**.

- The table covers **23 source currencies** converted into EUR, with `tocurd` always equal to `EUR`.

- The date coverage is broad, with **13,674 unique dates**, giving enough historical coverage for converting compensation, accounting, and market-price variables into euros.

- `exratd` represents the value of **1 unit of each source currency in EUR**. For example, an `exratd` of `0.54396` for CHF means 1 CHF equals about 0.544 EUR on that date.

---

---

<a id="construction-of-ceo-turnover-and-pay-panel"></a>
## 5.2 Construction of CEO Turnover and Pay Panel

This section converts raw role and remuneration records into a firm-year CEO panel.

The sequence is as follows:

1. identify valid CEO spells,
2. detect changes between consecutive CEOs,
3. match pay to the correct person and firm,
4. finally construct treatment and outcome variables.


<a id="ceo-turnover-events"></a>
### 5.2.1 CEO Turnover Events


#### Identify broad CEO candidates from role names

In [15]:
# Define keywords that identify broad CEO-like roles in BoardEx role names.
CEO_PATTERN = r"chief executive|(?<!\w)ceo(?!\w)|chief executive officer|managing director"

# Work on a copy so the original employment table remains unchanged.
df_emp = df_emp.copy()

# Convert role start dates to datetime for later spell construction.
df_emp["datestartrole"] = pd.to_datetime(df_emp["datestartrole"], errors="coerce")

# Replace BoardEx placeholder end dates and string-like missing values with proper missing values.
df_emp["dateendrole_clean"] = (
    df_emp["dateendrole"]
    .astype(str)
    .replace(["9000-01-01", "9999-12-31", "NaT", "nan", "None"], pd.NA)
)

# Convert cleaned role end dates to datetime.
df_emp["dateendrole_clean"] = pd.to_datetime(
    df_emp["dateendrole_clean"],
    errors="coerce"
)

# Standardize role names before searching for CEO-related terms.
df_emp["rolename_clean"] = df_emp["rolename"].fillna("").str.strip()

# Flag records whose role name indicates a CEO or CEO-like position.
df_emp["is_ceo"] = (
    df_emp["rolename_clean"]
    .str.lower()
    .str.contains(CEO_PATTERN, na=False, regex=True)
)

# Keep only broad CEO candidate records for the CEO-spell construction step.
df_emp_ceo = df_emp[df_emp["is_ceo"]].copy()

# Summary statistics
_ = summarize_dataframe(df_emp_ceo)

/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/2059738519.py:8: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_emp["datestartrole"] = pd.to_datetime(df_emp["datestartrole"], errors="coerce")


### Dataset overview


/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/2059738519.py:11: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_emp["dateendrole_clean"] = (
/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/2059738519.py:18:

,value
rows,10226
columns,13
duplicate rows,0
memory usage,5.52 MB


### Table overview


...

,count,mean,std,min,25%,50%,75%,max
companyid,10226.0,276248.569137,698923.228447,294.0,9863.0,24350.0,32240.0,4066835.0
directorid,10226.0,1181456.605711,858394.356014,28.0,494450.0,1120837.5,1767649.75,3359052.0


### Categorical columns


,count,missing,unique,top,top_frequency
column,,,,,
companyname,10226,0,568,SIEMENS AG,232
directorname,10226,0,7526,Albern Murty,8
dateendrole,10226,0,2688,9000-01-01,1910
rolename,10226,0,628,Division CEO,3452
hocountryname,10226,0,25,France,2523
sector,10226,0,39,Banks,1701
isin,10226,0,554,DE0007236101,232
rolename_clean,10226,0,628,Division CEO,3452
is_ceo,10226,0,1,True,10226


### Datetime columns


,min,max,missing
datestartrole,1900-01-01,2026-06-23,0
dateendrole_clean,1966-12-28,2026-12-28,1940


**Insights:**

- The CEO-candidate subset is much smaller than the full employment table: 8,009 records compared with 140,892 employment records, or about 5.7% of all role records.

- The filter keeps 5,856 unique directors and 534 firms, compared with 64,458 directors and 567 firms in the full employment table, so most firms remain covered after focusing on CEO-like roles.

- The subset has no duplicate rows and no missing values in the key identifiers, company information, role names, sectors, or countries.

- Compared with the full employment table, role names are much more focused: 550 unique CEO-like role names instead of 9,961 general role names.

- `dateendrole_clean` has 22.9% missing values because ongoing-role placeholders such as `9000-01-01` were converted to missing values under `dateendrole_clean`; this is expected and helps separate active roles from completed CEO spells.

- “Division CEO” is the most frequent matched role, which shows that the broad keyword filter captures CEO-like roles below the group CEO level and may require later refinement when constructing firm-level CEO spells.

- The CEO-candidate subset still spans 39 sectors and 25 countries, so the filtered sample preserves broad sector and geographic coverage from the original employment data.

#### Refine candidates to firm-level CEO spells

In this step, we refine the broad CEO-candidate records into **firm-level CEO spells**. The earlier keyword filter captures many CEO-like roles, including division-level and regional titles, so we now focus on identifying the executives who plausibly served as the main CEO of each firm.

In [16]:
# The broad CEO keyword search also captures regional, divisional, acting, and advisory roles.
df_emp_ceo = df_emp_ceo.copy()

# Main firm-level CEO keywords
has_main_ceo_title = df_emp_ceo["rolename_clean"].str.contains(
    r"\bCEO\b|Chief Executive|Chief Executive Officer|Group CEO|President/CEO|Chairman/CEO|Chair/CEO|MD/CEO|CEO/MD|Managing Director",
    case=False,
    regex=True,
    na=False
)

# Exclude business-unit / regional / lower-level CEO roles
exclude_unit_roles = df_emp_ceo["rolename_clean"].str.contains(
    r"Regional|Division|Divisional|Country|Branch|Sector|Zonal|Area|Business Unit|Market|Subsidiary|Segment",
    case=False,
    regex=True,
    na=False
)

# Exclude non-actual CEO roles
exclude_non_actual = df_emp_ceo["rolename_clean"].str.contains(
    r"Designate|Elect|in-Residence|Delegate|Office|Adviser|Advisor|Honorary|Shareholder Representative",
    case=False,
    regex=True,
    na=False
)

# Exclude subordinate CEO roles
exclude_subordinate = df_emp_ceo["rolename_clean"].str.contains(
    r"Deputy CEO|Vice CEO|Associate CEO|Assistant CEO",
    case=False,
    regex=True,
    na=False
)

# Interim / acting CEOs are excluded from the strict baseline definition
is_interim_acting = df_emp_ceo["rolename_clean"].str.contains(
    r"Interim|Acting",
    case=False,
    regex=True,
    na=False
)

# Co-CEO / joint CEO can be included in a robustness definition
is_co_ceo = df_emp_ceo["rolename_clean"].str.contains(
    r"Co-CEO|Joint CEO",
    case=False,
    regex=True,
    na=False
)

# Baseline CEO role definition
df_emp_ceo["firm_level_ceo_role"] = (
    has_main_ceo_title
    & ~exclude_unit_roles
    & ~exclude_non_actual
    & ~exclude_subordinate
)

# Strict baseline: exclude interim/acting and co-CEO roles
df_emp_ceo["firm_level_ceo_strict"] = (
    df_emp_ceo["firm_level_ceo_role"]
    & ~is_interim_acting
    & ~is_co_ceo
)

# Robustness definition: include co-CEOs but exclude interim/acting CEOs
df_emp_ceo["firm_level_ceo_with_coceo"] = (
    df_emp_ceo["firm_level_ceo_role"]
    & ~is_interim_acting
)

print(f"Strict firm-level CEO spells: {df_emp_ceo['firm_level_ceo_strict'].sum():,}")
print(f"Firm-level CEO spells including co-CEOs: {df_emp_ceo['firm_level_ceo_with_coceo'].sum():,}")
display(df_emp_ceo.head(5))

Strict firm-level CEO spells: 2,399
Firm-level CEO spells including co-CEOs: 2,479


/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/2721745304.py:53: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_emp_ceo["firm_level_ceo_role"] = (
/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/2721745304.

,companyid,companyname,directorid,directorname,datestartrole,dateendrole,rolename,hocountryname,sector,isin,dateendrole_clean,rolename_clean,is_ceo,firm_level_ceo_role,firm_level_ceo_strict,firm_level_ceo_with_coceo
77,294.0,3I GROUP PLC,1854.0,Brian Larcombe,1997-07-09,2004-07-07,CEO,United Kingdom - England,Private Equity,GB00B1YW4409,2004-07-07,CEO,True,True,True,True
133,294.0,3I GROUP PLC,7259.0,David Marlow,1988-04-01,1992-12-31,CEO,United Kingdom - England,Private Equity,GB00B1YW4409,1992-12-31,CEO,True,True,True,True
180,294.0,3I GROUP PLC,1233.0,Ewen Macpherson,1992-01-02,1997-07-09,CEO,United Kingdom - England,Private Equity,GB00B1YW4409,1997-07-09,CEO,True,True,True,True
222,294.0,3I GROUP PLC,1069.0,Hugh Foulds,1978-01-02,1988-12-31,CEO,United Kingdom - England,Private Equity,GB00B1YW4409,1988-12-31,CEO,True,True,True,True
262,294.0,3I GROUP PLC,1121118.0,Jeremy Ghose,2011-02-01,2017-03-03,Division CEO,United Kingdom - England,Private Equity,GB00B1YW4409,2017-03-03,Division CEO,True,False,False,False


#### Construct the baseline CEO-spell table

After refining the broad CEO-candidate records into firm-level CEO spells, we convert the refined records into a baseline CEO-spell table. Each row represents a director’s CEO tenure at a matched firm, with cleaned start and end dates, firm identifiers, and role information. This spell-level structure is the bridge between raw BoardEx employment records and the later firm-year turnover panel, because it allows us to identify when a CEO starts, leaves, or remains in office.

In [17]:
# Keep only strict firm-level CEOs, excluding interim/acting and co-CEO roles.
df_ceo_main = (
    df_emp_ceo[df_emp_ceo["firm_level_ceo_strict"]]
    .dropna(subset=["companyid", "directorid", "datestartrole"]) # removes any rows where critical identifiers are missing.
    .sort_values(["companyid", "datestartrole", "dateendrole_clean", "directorid"]) 
    .copy()
)

# Display a compact quality check for the resulting CEO-role sample.
_ = summarize_dataframe(df_ceo_main)

### Dataset overview


,value
rows,2399
columns,16
duplicate rows,0
memory usage,1.29 MB


### Table overview


,companyid,companyname,directorid,directorname,datestartrole,dateendrole,rolename,hocountryname,sector,isin,dateendrole_clean,rolename_clean,is_ceo,firm_level_ceo_role,firm_level_ceo_strict,firm_level_ceo_with_coceo
222,294.0,3I GROUP PLC,1069.0,Hugh Foulds,1978-01-02,1988-12-31,CEO,United Kingdom - England,Private Equity,GB00B1YW4409,1988-12-31,CEO,True,True,True,True
133,294.0,3I GROUP PLC,7259.0,David Marlow,1988-04-01,1992-12-31,CEO,United Kingdom - England,Private Equity,GB00B1YW4409,1992-12-31,CEO,True,True,True,True
180,294.0,3I GROUP PLC,1233.0,Ewen Macpherson,1992-01-02,1997-07-09,CEO,United Kingdom - England,Private Equity,GB00B1YW4409,1997-07-09,CEO,True,True,True,True


### Column coverage and dtypes


...

,count,mean,std,min,25%,50%,75%,max
companyid,2399.0,352196.636932,787338.244999,294.0,11984.0,25331.0,33505.0,4066835.0
directorid,2399.0,657257.745311,734813.968882,28.0,17187.0,484396.0,1131999.5,3359032.0


### Categorical columns


,count,missing,unique,top,top_frequency
column,,,,,
companyname,2399,0,555,ZURICH INSURANCE GROUP AG,15
directorname,2399,0,1965,Jan Jenisch,4
dateendrole,2399,0,1380,9000-01-01,493
rolename,2399,0,115,CEO,880
hocountryname,2399,0,25,France,337
sector,2399,0,39,Banks,322
isin,2399,0,546,DK0010311471,17
rolename_clean,2399,0,115,CEO,880
is_ceo,2399,0,1,True,2399


### Datetime columns


,min,max,missing
datestartrole,1900-01-01,2026-06-01,0
dateendrole_clean,1966-12-28,2026-06-17,512


**Insights:**

- The strict CEO-role sample contains 2,029 CEO spells across 511 firms, with no duplicate rows and complete coverage for the key identifiers (`companyid`, `directorid`, `isin`) and start dates.

- All observations satisfy the strict CEO definition: `is_ceo`, `firm_level_ceo_role`, `firm_level_ceo_strict`, and `firm_level_ceo_with_coceo` are always `True` in this filtered table. This confirms that interim/acting and co-CEO roles have already been removed.

- `dateendrole_clean` is missing for 460 rows, mostly because ongoing CEO roles are coded with `9000-01-01` in the raw `dateendrole` field. These should be interpreted as active/open-ended roles rather than ordinary missing values.

- The table covers a broad European firm sample: 23 headquarters countries and 39 sectors. France and Banks are the most frequent country and sector in this CEO-spell sample.

- Some firms and directors appear multiple times, which is expected because the table is at the CEO-spell level rather than one row per firm or one row per person.

#### Identify CEO turnover events

**CEO turnover** is defined when the current firm-level CEO's `directorid` differs from the previous observed CEO's `directorid` within the same company.

The first observed CEO spell per company is not coded as turnover, because we do not observe a prior CEO inside our sample window.

In [18]:
# Store the previous CEO's director ID within each firm to identify leadership transitions.
df_ceo_main["previous_ceo_directorid"] = (
    df_ceo_main
    .groupby("companyid")["directorid"]
    .shift()
)

# Flag CEO turnover when the current CEO differs from the previous CEO in the same firm.
# Using .ne(...).fillna(False) avoids pandas NA-to-integer conversion errors.
df_ceo_main["ceo_turnover"] = (
    df_ceo_main["directorid"]
    .ne(df_ceo_main["previous_ceo_directorid"])
    .fillna(False)
    .astype(int)
)

# The first observed CEO spell per company has no predecessor, so it is not counted as turnover.
df_ceo_main.loc[
    df_ceo_main["previous_ceo_directorid"].isna(),
    "ceo_turnover"
] = 0

# Extract the CEO start year as the turnover year.
df_ceo_main["turnover_year"] = df_ceo_main["datestartrole"].dt.year

print(f"CEO role spells: {len(df_ceo_main):,}")
print(f"Unique companies with CEO spells: {df_ceo_main['companyid'].nunique():,}")
print(f"CEO turnover events: {df_ceo_main['ceo_turnover'].sum():,}")

df_ceo_main.head()

CEO role spells: 2,399
Unique companies with CEO spells: 546
CEO turnover events: 1,535


/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/1685197352.py:2: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_ceo_main["previous_ceo_directorid"] = (
/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/1685197

,companyid,companyname,directorid,directorname,datestartrole,dateendrole,rolename,hocountryname,sector,isin,dateendrole_clean,rolename_clean,is_ceo,firm_level_ceo_role,firm_level_ceo_strict,firm_level_ceo_with_coceo,previous_ceo_directorid,ceo_turnover,turnover_year
222,294.0,3I GROUP PLC,1069.0,Hugh Foulds,1978-01-02,1988-12-31,CEO,United Kingdom - England,Private Equity,GB00B1YW4409,1988-12-31,CEO,True,True,True,True,<NA>,0,1978
133,294.0,3I GROUP PLC,7259.0,David Marlow,1988-04-01,1992-12-31,CEO,United Kingdom - England,Private Equity,GB00B1YW4409,1992-12-31,CEO,True,True,True,True,1069.0,1,1988
180,294.0,3I GROUP PLC,1233.0,Ewen Macpherson,1992-01-02,1997-07-09,CEO,United Kingdom - England,Private Equity,GB00B1YW4409,1997-07-09,CEO,True,True,True,True,7259.0,1,1992
77,294.0,3I GROUP PLC,1854.0,Brian Larcombe,1997-07-09,2004-07-07,CEO,United Kingdom - England,Private Equity,GB00B1YW4409,2004-07-07,CEO,True,True,True,True,1233.0,1,1997
455,294.0,3I GROUP PLC,2439.0,Phil Yea,2004-07-07,2009-01-27,CEO,United Kingdom - England,Private Equity,GB00B1YW4409,2009-01-27,CEO,True,True,True,True,1854.0,1,2004


#### Collapse turnover events to the firm-year level

This step collapses the CEO-spell data into one row per company-year. A company-year is marked as a turnover year when at least one new strict CEO starts in that year. The first observed CEO for each firm is not treated as a turnover event, because there is no prior CEO in the dataset to compare against.

The resulting table can then be merged into the final firm-year panel as a binary indicator of whether CEO turnover occurred in that company-year.

In [19]:
ceo_turnover_firm_year = (
    df_ceo_main[df_ceo_main["ceo_turnover"] == 1] # Keep only CEO spells that represent an actual turnover event.
    .groupby(["companyid", "turnover_year"]) # Collapse events to one row per company-year.
    .size() # Count how many CEO turnover events occurred in each company-year.
    .reset_index(name="num_ceo_turnovers") # Store the count in a clear column name.
)

ceo_turnover_firm_year["ceo_turnover_dummy"] = 1
_ = summarize_dataframe(ceo_turnover_firm_year)

### Dataset overview


,value
rows,1485
columns,4
duplicate rows,0
memory usage,0.04 MB


### Table overview


,companyid,turnover_year,num_ceo_turnovers,ceo_turnover_dummy
0,294.0,1988,1,1
1,294.0,1992,1,1
2,294.0,1997,1,1


### Column coverage and dtypes


,dtype,non_null,missing,missing_pct,unique
companyid,Float64,1485,0,0.0,464
turnover_year,int32,1485,0,0.0,50
num_ceo_turnovers,int64,1485,0,0.0,3
ceo_turnover_dummy,int64,1485,0,0.0,1


### Numeric columns


,count,mean,std,min,25%,50%,75%,max
companyid,1485.0,266845.995286,655949.280645,294.0,11062.0,24013.0,32235.0,3729546.0
turnover_year,1485.0,2011.430976,11.589281,1900.0,2005.0,2013.0,2020.0,2026.0
num_ceo_turnovers,1485.0,1.03367,0.191315,1.0,1.0,1.0,1.0,3.0
ceo_turnover_dummy,1485.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0


The resulting `ceo_turnover_firm_year` table is the firm-year turnover bridge that is merged into the remuneration panel by `companyid` and fiscal/report year.

<a id="prepare-annual-remuneration-records"></a>
### 5.2.2 Prepare Annual Remuneration Records for Identified CEOs


#### Filter remuneration to identified CEOs

In [20]:
# Clean ceo directorid from df_ceo_main
ceo_directorid_list = (df_ceo_main["directorid"].dropna().astype(int).unique().tolist())

# Obtain remuneration for the selected ceo
df_remun_ceo = df_remun[df_remun["directorid"].isin(ceo_directorid_list)].copy()



_ = summarize_dataframe(df_remun_ceo)

### Dataset overview


,value
rows,33965
columns,13
duplicate rows,575
memory usage,5.41 MB


### Table overview


,boardid,directorid,annualreportdate,currency,salary,bonus,totalcompensation,valtoteqheld,valltipheld,valeqaward,ltipvalue,toteqatrisk,perftotal
129,384.0,33284.0,2001-12-01,USD,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
130,384.0,33284.0,2002-12-01,USD,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
131,384.0,33284.0,2003-12-01,USD,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


### Column coverage and dtypes


...

,count,mean,std,min,25%,50%,75%,max
boardid,33965.0,359838.233093,778620.190102,384.0,10957.0,24495.0,35183.0,4066835.0
directorid,33965.0,320896.504195,482556.429515,28.0,11474.0,27881.0,510814.0,3266204.0
salary,8932.0,625.74026,805.280774,0.0,75.0,207.0,1050.0,19076.0
bonus,3277.0,1338.144034,1333.428204,0.0,358.0,1029.0,1903.0,14286.0
totalcompensation,11262.0,885.65317,1523.529163,0.0,29.0,124.0,1214.0,19076.0
valtoteqheld,6268.0,44761.517869,462105.520969,0.0,81.0,438.0,3886.0,12581879.0
valltipheld,2439.0,12393.153752,44166.658826,0.0,1369.5,4172.0,11055.5,1163693.0
valeqaward,94.0,1016.595745,1901.020633,0.0,98.75,208.5,468.5,7527.0
ltipvalue,2157.0,3697.444135,11756.149892,0.0,705.0,1896.0,3749.0,347474.0
toteqatrisk,2534.0,3885.81689,11291.757962,0.0,705.0,1914.0,3900.75,347474.0


### Categorical columns


,count,missing,unique,top,top_frequency
column,,,,,
currency,33965,0,1,USD,33965


### Datetime columns


,min,max,missing
annualreportdate,1997-03-01,2026-05-01,1499


**Insights:**

- The CEO filtered remuneration table is broad but sparse: it contains 31,645 rows, but most detailed pay components have very high missingness.

- `totalcompensation` has the best coverage among the monetary remuneration variables, with around 33% coverage.

- We therefore keep `totalcompensation` as the main CEO pay measure because it is the most complete and comparable compensation variable across firms and years.

#### Clean Remuneration and Convert Nominal Values to EUR

This step keeps total compensation as the main CEO pay measure and converts nominal remuneration values into EUR. Because compensation is reported at the annual-report level, we match each record to the average annual USD-to-EUR exchange rate rather than to a single daily rate. This reduces unnecessary missing FX matches and better reflects the annual nature of the pay variable.

After conversion, we keep the key identifiers, report date, reporting currency, original compensation, average FX rate, and EUR-denominated compensation. Exact duplicate rows are removed before the remuneration data are matched to valid CEO spells.

In [21]:
# Keep only totalcompensation as pay measure
df_remun_ceo = df_remun_ceo[["boardid", "directorid", "annualreportdate", "currency", "totalcompensation"]].copy()

# Extract year because compensation is annual and exact daily FX matches can be missing
df_remun_ceo["annualreportyear"] = df_remun_ceo["annualreportdate"].dt.year
df_fx["year"] = df_fx["datadate"].dt.year

# Calculate average annual FX rate by currency-year
df_fx_annual = (df_fx.groupby(["fromcurd", "year"], as_index=False).agg(fx_rate_avg_to_eur=("exratd", "mean")))

# Merge annual average FX rate based on currency and annual report year
df_remun_ceo = df_remun_ceo.merge(df_fx_annual, left_on=["currency", "annualreportyear"], right_on=["fromcurd", "year"],how="left",validate="many_to_one")

# Convert total compensation into EUR
df_remun_ceo["totalcompensation_eur"] = (df_remun_ceo["totalcompensation"] * df_remun_ceo["fx_rate_avg_to_eur"])

# Remove exact duplicate remuneration rows after cleaning and FX conversion
df_remun_ceo = df_remun_ceo.drop_duplicates().copy()

# Retain the necessary columns
df_remun_ceo = df_remun_ceo[["boardid","directorid","annualreportdate", "annualreportyear", "currency", "totalcompensation", "fx_rate_avg_to_eur", "totalcompensation_eur"]].copy()

_ = summarize_dataframe(df_remun_ceo)

### Dataset overview


/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/4016243209.py:5: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_remun_ceo["annualreportyear"] = df_remun_ceo["annualreportdate"].dt.year
/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9p

,value
rows,33390
columns,8
duplicate rows,0
memory usage,3.85 MB


### Table overview


,boardid,directorid,annualreportdate,annualreportyear,currency,totalcompensation,fx_rate_avg_to_eur,totalcompensation_eur
0,384.0,33284.0,2001-12-01,2001.0,USD,<NA>,1.11768,<NA>
1,384.0,33284.0,2002-12-01,2002.0,USD,<NA>,1.060608,<NA>
2,384.0,33284.0,2003-12-01,2003.0,USD,<NA>,0.88509,<NA>


...

,count,mean,std,min,25%,50%,75%,max
boardid,33390.0,362161.982839,781367.398951,384.0,10957.0,24523.0,35420.0,4066835.0
directorid,33390.0,321344.771908,482864.194434,28.0,11474.0,27793.0,513292.0,3266204.0
annualreportyear,31905.0,2012.452186,7.345277,1997.0,2007.0,2013.0,2019.0,2026.0
totalcompensation,11123.0,869.14034,1510.294152,0.0,28.0,121.0,1162.5,19076.0
fx_rate_avg_to_eur,31905.0,0.850931,0.102062,0.683517,0.75526,0.8477,0.904276,1.11768
totalcompensation_eur,10796.0,740.308198,1258.215894,0.0,28.648534,106.356651,1033.679577,14407.34153


### Categorical columns


,count,missing,unique,top,top_frequency
column,,,,,
currency,33390,0,1,USD,33390


### Datetime columns


,min,max,missing
annualreportdate,1997-03-01,2026-05-01,1485


**Insights:**

- After removing 516 exact duplicate rows, the cleaned CEO remuneration dataset contains 31,129 observations and 8 columns, including the annual report year, the annual average USD-to-EUR exchange rate, and EUR-converted compensation.

- Using annual average FX rates improves the currency conversion step. `fx_rate_avg_to_eur` is available for every row with a valid `annualreportdate`, so remaining missing FX values are driven by missing annual report dates rather than failed exact date matches.

- The main limitation is still compensation coverage. `totalcompensation` is missing for 21,467 rows, which means most missing `totalcompensation_eur` values come from missing pay values rather than the FX merge.

- Duplicate removal is applied before matching remuneration records to CEO spells, so repeated BoardEx remuneration rows do not enter the CEO-pay panel.

- All remuneration records are reported in USD, and the FX variable converts USD into EUR using yearly average rates. This is appropriate because CEO compensation is an annual measure rather than a daily transaction.

<a id="ceo-remuneration-matching"></a>
### 5.2.3 Match CEO Remuneration to Valid CEO Spells


The merge links remuneration and employment by `directorid` and `boardid == companyid` so compensation cannot be assigned to the same person at another firm. We retain annual-report dates inside the CEO spell with a 90-day boundary tolerance. The tolerance accommodates reporting dates close to a transition, but it can allow both outgoing and incoming CEOs to appear in the same firm-year; the next section therefore applies an explicit one-record selection rule.

In [22]:
ceo_spells = df_ceo_main[["companyid", "isin", "companyname", "hocountryname", "sector", "directorid", "directorname", "rolename", "datestartrole", "dateendrole_clean", "ceo_turnover", "turnover_year"]].copy()

# Ensure merge keys are comparable
ceo_spells["companyid"] = pd.to_numeric(ceo_spells["companyid"], errors="coerce")
df_remun_ceo["boardid"] = pd.to_numeric(df_remun_ceo["boardid"], errors="coerce")

# Merge on both Director and Company IDs simultaneously
df_ceo_remun_matched = df_remun_ceo.merge(ceo_spells, left_on=["directorid", "boardid"], right_on=["directorid", "companyid"], how="inner")

# Apply tolerance window directly using in-place vectorized conditions
df_ceo_remun_matched = df_ceo_remun_matched.loc[
    (df_ceo_remun_matched["annualreportdate"] >= (df_ceo_remun_matched["datestartrole"] - pd.Timedelta(days=90))) &
    (df_ceo_remun_matched["dateendrole_clean"].isna() | 
     (df_ceo_remun_matched["annualreportdate"] <= (df_ceo_remun_matched["dateendrole_clean"] + pd.Timedelta(days=90))))
].copy()

# Keep valid dated CEO-pay observations with positive EUR compensation
df_ceo_remun_matched = df_ceo_remun_matched.loc[
    df_ceo_remun_matched["annualreportdate"].notna()
    & (df_ceo_remun_matched["totalcompensation_eur"] > 0)
].copy()

# Extract annual report year from annualreportdate
df_ceo_remun_matched["annualreportyear"] = df_ceo_remun_matched["annualreportdate"].dt.year

# Add log-transform EUR CEO compensation
df_ceo_remun_matched["log_totalcompensation_eur"] = np.log(df_ceo_remun_matched["totalcompensation_eur"])

# Keep only columns needed for the CEO pay panel
df_ceo_remun_matched = df_ceo_remun_matched[
    [
    "companyid", "isin", "companyname", "hocountryname", "sector",
    "directorid", "directorname", "rolename", "datestartrole", "dateendrole_clean",
    "annualreportdate", "annualreportyear", "currency",
    "totalcompensation", "fx_rate_avg_to_eur", "totalcompensation_eur", "log_totalcompensation_eur",
    "ceo_turnover","turnover_year",
    ]
].copy()

_ = summarize_dataframe(df_ceo_remun_matched)


### Dataset overview


/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/2860181394.py:4: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  ceo_spells["companyid"] = pd.to_numeric(ceo_spells["companyid"], errors="coerce")
/var/folders/wx/b_h8zt7d5ps2y7dh

,value
rows,2316
columns,19
duplicate rows,0
memory usage,1.22 MB


### Table overview


,companyid,isin,companyname,hocountryname,sector,directorid,directorname,rolename,datestartrole,dateendrole_clean,annualreportdate,annualreportyear,currency,totalcompensation,fx_rate_avg_to_eur,totalcompensation_eur,log_totalcompensation_eur,ceo_turnover,turnover_year
33,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,4611.0,Fred Kindle,President/CEO,2005-01-01,2008-02-13,2004-12-01,2004,USD,351.0,0.804927,282.529315,5.643782,1,2005
34,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,4611.0,Fred Kindle,President/CEO,2005-01-01,2008-02-13,2005-12-01,2005,USD,1299.0,0.804784,1045.414149,6.952168,1,2005
35,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,4611.0,Fred Kindle,President/CEO,2005-01-01,2008-02-13,2006-12-01,2006,USD,2689.0,0.796772,2142.520336,7.669738,1,2005


...

,count,mean,std,min,25%,50%,75%,max
companyid,2316.0,116657.768998,419922.255025,422.0,8710.0,16036.0,27916.0,3486568.0
directorid,2316.0,333215.760794,453294.099046,61.0,12364.0,31498.0,515002.0,2703829.0
annualreportyear,2316.0,2013.175302,7.129315,1997.0,2007.0,2013.0,2019.0,2025.0
totalcompensation,2316.0,2723.61399,1740.245116,2.0,1523.0,2405.5,3608.0,13591.0
fx_rate_avg_to_eur,2316.0,0.845941,0.098932,0.683517,0.75526,0.8477,0.904276,1.11768
totalcompensation_eur,2316.0,2277.900263,1434.534744,1.848976,1258.808102,2019.701307,2972.532566,10297.6674
log_totalcompensation_eur,2316.0,7.499908,0.784145,0.614632,7.137921,7.610705,7.99717,9.239673
ceo_turnover,2316.0,0.694732,0.46062,0.0,0.0,1.0,1.0,1.0
turnover_year,2316.0,2008.523316,8.503277,1976.0,2003.0,2009.0,2015.0,2025.0


### Categorical columns


,count,missing,unique,top,top_frequency
column,,,,,
isin,2316,0,137,IE00BD1RP616,30
companyname,2316,0,137,BANK OF IRELAND GROUP PLC,30
hocountryname,2316,0,15,France,749
sector,2316,0,32,Banks,253
directorname,2316,0,388,Michael O'Leary,26
rolename,2316,0,40,Chairman/CEO,967
currency,2316,0,1,USD,2316


### Datetime columns


,min,max,missing
datestartrole,1976-01-02,2025-07-01,0
dateendrole_clean,1998-03-30,2026-05-31,529
annualreportdate,1997-11-01,2025-12-01,0


**Insights:**

- After matching remuneration records to valid CEO spells, the dataset contains 2,111 CEO-pay observations across 137 companies and 348 CEOs and **no duplicates**.

- All retained observations have valid annual report dates and positive EUR-denominated compensation. This means the table is now suitable for constructing the baseline CEO-pay panel.

- The matching step greatly reduces the sample from the broader remuneration table because it keeps only records linked to the same person, the same firm, and a valid CEO spell window.

- Missing values remain only in `dateendrole_clean`, affecting 528 rows. This is expected because some CEO spells are ongoing or have no recorded end date.

<a id="firm-year-ceo-pay-panel"></a>
### 5.2.4 Firm-Year CEO Pay Panel


The matched table can contain more than one CEO record for a company-year when outgoing and incoming CEOs both receive compensation, when roles overlap, or when the 90-day tolerance admits records near both spells. For this first analysis panel, we keep the latest annual-report observation within each `companyid` and `annualreportyear`. If annual-report dates tie, the current sort uses `directorid` only as a deterministic tie-breaker; that tie should not be interpreted economically.

In [23]:
ceo_pay_panel = df_ceo_remun_matched.copy()

# Keep one record per firm-year: latest report date, then directorid as a deterministic tie-breaker.
ceo_pay_panel = (
    ceo_pay_panel
    .sort_values(["companyid", "annualreportyear", "annualreportdate", "directorid"])
    .drop_duplicates(subset=["companyid", "annualreportyear"], keep="last")
    .sort_values(["companyid", "annualreportyear"])
    .reset_index(drop=True)
)

ceo_pay_panel.head()

,companyid,isin,companyname,hocountryname,sector,directorid,directorname,rolename,datestartrole,dateendrole_clean,annualreportdate,annualreportyear,currency,totalcompensation,fx_rate_avg_to_eur,totalcompensation_eur,log_totalcompensation_eur,ceo_turnover,turnover_year
0,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,14884.0,Jörgen Centerman,President/CEO,2000-12-31,2002-09-05,2001-12-01,2001,USD,1787.0,1.11768,1997.294162,7.599549,0,2000
1,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2002-12-01,2002,USD,1419.0,1.060608,1505.002572,7.31655,1,2002
2,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2003-12-01,2003,USD,2414.0,0.88509,2136.607095,7.666974,1,2002
3,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2004-12-01,2004,USD,3383.0,0.804927,2723.067443,7.909514,1,2002
4,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,4611.0,Fred Kindle,President/CEO,2005-01-01,2008-02-13,2005-12-01,2005,USD,1299.0,0.804784,1045.414149,6.952168,1,2005


Next, we merge in the CEO turnover firm-year indicator created from the employment data. `ceo_turnover_dummy` equals one in years where a new CEO spell starts at the firm. Missing values are set to zero, meaning no observed CEO turnover in that firm-year.

In [24]:
# Merge firm-year CEO turnover indicator into the pay panel
ceo_pay_panel = ceo_pay_panel.merge(
    ceo_turnover_firm_year,
    left_on=["companyid", "annualreportyear"],
    right_on=["companyid", "turnover_year"],
    how="left",
    suffixes=("", "_event")
)

ceo_pay_panel["num_ceo_turnovers"] = ceo_pay_panel["num_ceo_turnovers"].fillna(0).astype(int)
ceo_pay_panel["ceo_turnover_dummy"] = ceo_pay_panel["ceo_turnover_dummy"].fillna(0).astype(int)

ceo_pay_panel.head()

/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/982465494.py:10: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  ceo_pay_panel["num_ceo_turnovers"] = ceo_pay_panel["num_ceo_turnovers"].fillna(0).astype(int)
/var/folders/wx/b_h8

,companyid,isin,companyname,hocountryname,sector,directorid,directorname,rolename,datestartrole,dateendrole_clean,annualreportdate,annualreportyear,currency,totalcompensation,fx_rate_avg_to_eur,totalcompensation_eur,log_totalcompensation_eur,ceo_turnover,turnover_year,turnover_year_event,num_ceo_turnovers,ceo_turnover_dummy
0,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,14884.0,Jörgen Centerman,President/CEO,2000-12-31,2002-09-05,2001-12-01,2001,USD,1787.0,1.11768,1997.294162,7.599549,0,2000,NaN,0,0
1,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2002-12-01,2002,USD,1419.0,1.060608,1505.002572,7.31655,1,2002,2002.0,1,1
2,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2003-12-01,2003,USD,2414.0,0.88509,2136.607095,7.666974,1,2002,NaN,0,0
3,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2004-12-01,2004,USD,3383.0,0.804927,2723.067443,7.909514,1,2002,NaN,0,0
4,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,4611.0,Fred Kindle,President/CEO,2005-01-01,2008-02-13,2005-12-01,2005,USD,1299.0,0.804784,1045.414149,6.952168,1,2005,2005.0,1,1


***For event-style analysis, each company is assigned its first observed turnover year.*** `treated_firm` identifies companies that experience at least one observed turnover, while `event_time` measures years relative to that first turnover and `post_turnover` identifies the turnover year and later observations.

In [25]:
first_turnover_year = (
    ceo_turnover_firm_year
    .groupby("companyid")["turnover_year"]
    .min()
    .rename("first_turnover_year")
    .reset_index()
)
print(f"Firms with an observed turnover: {len(first_turnover_year):,}")
print(
    f"Observed first-turnover range: "
    f"{first_turnover_year['first_turnover_year'].min()}-"
    f"{first_turnover_year['first_turnover_year'].max()}"
)
first_turnover_year.head()

Firms with an observed turnover: 464
Observed first-turnover range: 1900-2026


,companyid,first_turnover_year
0,294.0,1988
1,384.0,2012
2,422.0,2002
3,595.0,2016
4,598.0,2006


In [26]:
# Create event-time variables around the first observed CEO turnover per firm.
ceo_pay_panel = ceo_pay_panel.merge(
    first_turnover_year,
    on="companyid",
    how="left"
)

ceo_pay_panel["treated_firm"] = ceo_pay_panel["first_turnover_year"].notna().astype(int)
ceo_pay_panel["event_time"] = ceo_pay_panel["annualreportyear"] - ceo_pay_panel["first_turnover_year"]
ceo_pay_panel["post_turnover"] = (
    ceo_pay_panel["treated_firm"].eq(1)
    & ceo_pay_panel["event_time"].ge(0)
).astype(int)

# Useful event-window flags for later robustness checks
ceo_pay_panel["event_window_3yr"] = ceo_pay_panel["event_time"].between(-3, 3).fillna(False).astype(int)
ceo_pay_panel["event_window_5yr"] = ceo_pay_panel["event_time"].between(-5, 5).fillna(False).astype(int)

# Sort based on companyid and annualreportyear to improve readability
ceo_pay_panel = ceo_pay_panel.sort_values(["companyid", "annualreportyear"]).copy()

ceo_pay_panel.head()

/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/1453383241.py:8: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  ceo_pay_panel["treated_firm"] = ceo_pay_panel["first_turnover_year"].notna().astype(int)
/var/folders/wx/b_h8zt7d5

,companyid,isin,companyname,hocountryname,sector,directorid,directorname,rolename,datestartrole,dateendrole_clean,annualreportdate,annualreportyear,currency,totalcompensation,fx_rate_avg_to_eur,totalcompensation_eur,log_totalcompensation_eur,ceo_turnover,turnover_year,turnover_year_event,num_ceo_turnovers,ceo_turnover_dummy,first_turnover_year,treated_firm,event_time,post_turnover,event_window_3yr,event_window_5yr
0,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,14884.0,Jörgen Centerman,President/CEO,2000-12-31,2002-09-05,2001-12-01,2001,USD,1787.0,1.11768,1997.294162,7.599549,0,2000,NaN,0,0,2002.0,1,-1.0,0,1,1
1,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2002-12-01,2002,USD,1419.0,1.060608,1505.002572,7.31655,1,2002,2002.0,1,1,2002.0,1,0.0,1,1,1
2,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2003-12-01,2003,USD,2414.0,0.88509,2136.607095,7.666974,1,2002,NaN,0,0,2002.0,1,1.0,1,1,1
3,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2004-12-01,2004,USD,3383.0,0.804927,2723.067443,7.909514,1,2002,NaN,0,0,2002.0,1,2.0,1,1,1
4,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,4611.0,Fred Kindle,President/CEO,2005-01-01,2008-02-13,2005-12-01,2005,USD,1299.0,0.804784,1045.414149,6.952168,1,2005,2005.0,1,1,2002.0,1,3.0,1,1,1


At this stage, compensation is comparable across currencies but remains nominal. Pay growth, logarithms, winsorization, and dispersion are deliberately postponed until after the final panel receives its CPI adjustment.

---

---

<a id="firm-financial-fundamentals-and-financial-ratios"></a>
## 5.3 Firm Financial Fundamentals and Financial Ratios

Section 5.2 creates the CEO-pay firm-year backbone. This subsection prepares the firm-side variables that will be merged onto that backbone: accounting fundamentals, month-end market prices, market capitalization, profitability, leverage, and log size measures. The result is `df_funda`, a firm-year table with monetary values converted to nominal EUR and ratios computed before the final CPI adjustment.

<a id="fundamentals-eur-conversion"></a>
### 5.3.1 Convert Annual Fundamentals to Nominal EUR

The annual Compustat fundamentals are reported in local accounting currencies. We extract the fiscal/report year from `datadate`, attach the average annual EUR exchange rate for each currency-year, and convert the main accounting variables into nominal EUR. Ratios are not computed here yet, because profitability and leverage should use same-currency reported values.

In [27]:
df_funda = df_firm.copy()

# Convert data type datadate into datetime format
df_funda["datadate"] = pd.to_datetime(df_funda["datadate"], errors="coerce")

# Extract annual report year from annualreportdate
df_funda["annualreportyear"] = df_funda["datadate"].dt.year

# Merge annual average FX rate based on curcd and annual report year
df_funda = df_funda.merge(df_fx_annual, left_on=["curcd", "annualreportyear"], right_on=["fromcurd", "year"],how="left",validate="many_to_one").drop(columns=["fromcurd", "year"])

fundamental_cols = ["at", "sale", "ebit", "dltt", "nicon"]

# Add converted fundamentals into df_funda
for col in fundamental_cols:
    df_funda[f"{col}_eur"] = df_funda[col] * df_funda["fx_rate_avg_to_eur"]

_ = summarize_dataframe(df_funda)

### Dataset overview


/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/2073289850.py:4: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_funda["datadate"] = pd.to_datetime(df_funda["datadate"], errors="coerce")
/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9

,value
rows,16350
columns,18
duplicate rows,0
memory usage,4.63 MB


### Table overview


,gvkey,isin,datadate,curcd,at,sale,ebit,dltt,nicon,emp,cshoi,annualreportyear,fx_rate_avg_to_eur,at_eur,sale_eur,ebit_eur,dltt_eur,nicon_eur
0,272817,AT0000606306,2003-12-31,EUR,20060.255,<NA>,332.687,620.895,178.733,17.299,50.0,2003,1.0,20060.255,<NA>,332.687,620.895,178.733
1,272817,AT0000606306,2004-12-31,EUR,28907.122,<NA>,414.062,826.594,210.462,22.851,62.5,2004,1.0,28907.122,<NA>,414.062,826.594,210.462
2,272817,AT0000606306,2005-12-31,EUR,40695.0,<NA>,621.0,1340.0,382.0,43.6,142.58,2005,1.0,40695.0,<NA>,621.0,1340.0,382.0


...

,count,mean,std,min,25%,50%,75%,max
at,16311.0,716968.992262,11346843.026168,0.0,3159.05,11864.0,54919.7,643358816.0
sale,12113.0,146854.538712,2642424.55592,-3518.0,1931.6,6185.9,21526.0,100117000.0
ebit,16164.0,19956.228761,312519.218501,-1652000.0,194.84725,659.772,2637.225,10426000.0
dltt,16179.0,78420.583106,1554747.60725,0.0,308.471,1891.9,7796.8,101852807.0
nicon,13406.0,4100.687279,109105.680251,-37500.0,96.0,355.9025,1332.0,7748000.0
emp,14977.0,39.828477,68.458763,0.0,3.707,14.0,45.95,684.025
cshoi,15245.0,1149.31836,9191.041952,0.0,55.065,205.394,728.835,523438.445
annualreportyear,16350.0,2009.486972,10.151982,1987.0,2002.0,2010.0,2018.0,2026.0
fx_rate_avg_to_eur,16333.0,0.815488,0.426998,0.000465,0.506247,1.0,1.0,1.641654
at_eur,16308.0,61898.090687,209718.812492,0.0,2103.912755,7301.819,28215.718274,3021625.996208


### Categorical columns


,count,missing,unique,top,top_frequency
column,,,,,
gvkey,16350,0,557,208331,44
isin,16350,0,557,GB00B62G9D36,44
curcd,16336,14,22,EUR,6751


### Datetime columns


,min,max,missing
datadate,1987-06-30,2026-03-31,0


#### Check whether fundamentals have multiple report dates within the same firm-year

In [28]:
# Check whether fundamentals have multiple report dates within the same firm-year
duplicate_funda_years = (
    df_funda
    .groupby(["isin", "annualreportyear"])
    .agg(
        n_rows=("datadate", "size"),
        n_datadates=("datadate", "nunique"),
        first_datadate=("datadate", "min"),
        last_datadate=("datadate", "max"),
    )
    .reset_index()
    .query("n_rows > 1")
)

duplicate_funda_years.head()

,isin,annualreportyear,n_rows,n_datadates,first_datadate,last_datadate
283,BE0003565737,1992,2,2,1992-03-31,1992-12-31
565,BE0003851681,2020,2,2,2020-06-30,2020-12-31
2936,DE0006602006,2002,2,2,2002-09-30,2002-12-31
3111,DE0007037129,2001,2,2,2001-06-30,2001-12-31
3503,DE0008430026,1998,2,2,1998-06-30,1998-12-31


**Insights:**

- If a firm has more than one fundamentals record within the same `isin` and `annualreportyear`, then `df_funda` is not unique at the future merge key. This can create duplicate rows when merging fundamentals into the CEO-pay panel. Therefore, before the final merge, we need to keep one fundamentals record per firm-year, which the latest `datadate`.

In [29]:
# Keep one fundamentals record per firm-year before merging into CEO pay panel
df_funda = (
    df_funda
    .sort_values(["isin", "annualreportyear", "datadate"])
    .drop_duplicates(subset=["isin", "annualreportyear"], keep="last")
    .copy()
)

# Check that fundamentals are now unique at the merge grain
print(
    "Duplicate firm-year fundamentals:",
    df_funda.duplicated(["isin", "annualreportyear"]).sum()
)

Duplicate firm-year fundamentals: 0


<a id="market-eur-merge"></a>
### 5.3.2 Convert Security Prices to EUR and Merge with Fundamentals

The market table has already been reduced to one exchange and one security issue (`iid`) per `gvkey`, so `gvkey` and `datadate` form a unique market-price key. We convert monthly closing prices to EUR and merge them into the annual fundamentals table on the shared month-end report date.

In [30]:
# Convert data type datadate into datetime format
df_market["datadate"] = pd.to_datetime(df_market["datadate"], errors="coerce")
df_fx["datadate"] = pd.to_datetime(df_fx["datadate"], errors="coerce")

# Merge df_market and df_fx to get EUR exchange rate
df_market_eur = df_market.merge(
    df_fx,
    left_on=["datadate", "curcdm",],
    right_on=["datadate", "fromcurd"],
    how="left").drop(columns=["iid", "exchg", "exchgdesc", "fromcurd", "tocurd", "year"]).rename(columns={"exratd":"fx_eur", "prccm":"close"})

# Add close_eur
df_market_eur["close_eur"] = df_market_eur["close"] * df_market_eur["fx_eur"]

# Merge df_funda and df_market_eur to include close_eur
df_funda = df_funda.merge(
    df_market_eur[["gvkey", "datadate", "close_eur"]],
    on=["gvkey", "datadate"],
    how="left",
    validate="many_to_one"
)

_ = summarize_dataframe(df_funda)

/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/339067132.py:2: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_market["datadate"] = pd.to_datetime(df_market["datadate"], errors="coerce")
/var/folders/wx/b_h8zt7d5ps2y7dhc2qj

### Dataset overview


,value
rows,16310
columns,19
duplicate rows,0
memory usage,4.76 MB


### Table overview


,gvkey,isin,datadate,curcd,at,sale,ebit,dltt,nicon,emp,cshoi,annualreportyear,fx_rate_avg_to_eur,at_eur,sale_eur,ebit_eur,dltt_eur,nicon_eur,close_eur
0,272817,AT0000606306,2003-12-31,EUR,20060.255,<NA>,332.687,620.895,178.733,17.299,50.0,2003,1.0,20060.255,<NA>,332.687,620.895,178.733,<NA>
1,272817,AT0000606306,2004-12-31,EUR,28907.122,<NA>,414.062,826.594,210.462,22.851,62.5,2004,1.0,28907.122,<NA>,414.062,826.594,210.462,<NA>
2,272817,AT0000606306,2005-12-31,EUR,40695.0,<NA>,621.0,1340.0,382.0,43.6,142.58,2005,1.0,40695.0,<NA>,621.0,1340.0,382.0,<NA>


...

,count,mean,std,min,25%,50%,75%,max
at,16288.0,717611.613955,11354809.938691,0.0,3161.175,11864.5,54919.55,643358816.0
sale,12100.0,146997.494399,2643840.117932,-3518.0,1932.72525,6184.95,21517.0,100117000.0
ebit,16142.0,19971.611069,312729.35619,-1652000.0,195.0,660.7685,2637.975,10426000.0
dltt,16156.0,78450.76288,1555827.756528,0.0,308.58925,1892.2,7800.83225,101852807.0
nicon,13385.0,4106.756872,109191.133543,-37500.0,96.0,356.4,1335.0,7748000.0
emp,14960.0,39.826213,68.472436,0.0,3.70775,14.0,45.95075,684.025
cshoi,15221.0,1150.189179,9198.218855,0.0,55.067,205.394,728.75,523438.445
annualreportyear,16310.0,2009.485285,10.151915,1987.0,2002.0,2010.0,2018.0,2026.0
fx_rate_avg_to_eur,16307.0,0.815244,0.426935,0.000465,0.506247,1.0,1.0,1.641654
at_eur,16285.0,61910.695714,209808.585411,0.0,2104.886261,7305.028221,28203.8,3021625.996208


### Categorical columns


,count,missing,unique,top,top_frequency
column,,,,,
gvkey,16310,0,557,015181,39
isin,16310,0,557,ES0113211835,39
curcd,16310,0,22,EUR,6744


### Datetime columns


,min,max,missing
datadate,1987-06-30,2026-03-31,0


**Insights:**

- The merged fundamentals-market dataset contains 16,350 firm-year observations for 557 firms, with no duplicate rows. This confirms that the market-price merge did not create row duplication.

- Currency conversion coverage is very strong. `fx_rate_avg_to_eur` is available for 99.9% of observations, so most missing EUR-denominated fundamentals are driven by missing original Compustat values rather than missing FX rates.

- Balance-sheet variables have high coverage: total assets (`at_eur`) and long-term debt (`dltt_eur`) are available for more than 98% of rows. These variables are suitable as firm-size and leverage inputs for the final panel.

- Sales and net income have more missing values, with `sale_eur` missing for 25.9% and `nicon_eur` missing for 18.0% of rows. These variables may need careful treatment in later modeling or robustness checks.

- The market-price variable `close_eur` is available for 9,315 observations, or about 57% of the dataset. The remaining missing values reflect firm-years without a matched month-end market price after selecting one exchange and one security issue per firm.

- The dataset spans annual report years from 1987 to 2026, but the final CEO-pay analysis will likely use a smaller time window depending on where CEO remuneration, turnover, and fundamentals overlap.

- The table is now structurally ready for calculating market capitalization in EUR using `close_eur` and `cshoi`.

<a id="financial-ratios"></a>
### 5.3.3 Calculate Financial Ratios

The final step creates the firm controls used later in the analysis panel. Market capitalization is calculated from EUR closing price and shares outstanding. ROA, EBIT margin, and leverage are computed from reported same-currency accounting values, which keeps ratios currency-invariant. Log size variables are only calculated for strictly positive values to avoid invalid logarithms.

In [31]:
# Compute market capitalization in EUR
df_funda["market_cap_eur"] = df_funda["close_eur"] * df_funda["cshoi"]

# Ratios are currency-invariant, so compute them from same-currency reported values
df_funda["roa"] = df_funda["nicon"] / df_funda["at"].replace(0, np.nan)
df_funda["ebit_margin"] = df_funda["ebit"] / df_funda["sale"].replace(0, np.nan)
df_funda["leverage"] = df_funda["dltt"] / df_funda["at"].replace(0, np.nan)

# Log-transform only strictly positive values
df_funda["log_assets_eur_nominal"] = np.nan
df_funda.loc[df_funda["at_eur"] > 0, "log_assets_eur_nominal"] = np.log(
    df_funda.loc[df_funda["at_eur"] > 0, "at_eur"].astype(float)
)

df_funda["log_market_cap_eur_nominal"] = np.nan
df_funda.loc[df_funda["market_cap_eur"] > 0, "log_market_cap_eur_nominal"] = np.log(
    df_funda.loc[df_funda["market_cap_eur"] > 0, "market_cap_eur"].astype(float)
)

df_funda.head()

/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/2790409271.py:2: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_funda["market_cap_eur"] = df_funda["close_eur"] * df_funda["cshoi"]
/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc000

,gvkey,isin,datadate,curcd,at,sale,ebit,dltt,nicon,emp,cshoi,annualreportyear,fx_rate_avg_to_eur,at_eur,sale_eur,ebit_eur,dltt_eur,nicon_eur,close_eur,market_cap_eur,roa,ebit_margin,leverage,log_assets_eur_nominal,log_market_cap_eur_nominal
0,272817,AT0000606306,2003-12-31,EUR,20060.255,<NA>,332.687,620.895,178.733,17.299,50.0,2003,1.0,20060.255,<NA>,332.687,620.895,178.733,<NA>,<NA>,0.00891,<NA>,0.030952,9.906496,NaN
1,272817,AT0000606306,2004-12-31,EUR,28907.122,<NA>,414.062,826.594,210.462,22.851,62.5,2004,1.0,28907.122,<NA>,414.062,826.594,210.462,<NA>,<NA>,0.007281,<NA>,0.028595,10.271843,NaN
2,272817,AT0000606306,2005-12-31,EUR,40695.0,<NA>,621.0,1340.0,382.0,43.6,142.58,2005,1.0,40695.0,<NA>,621.0,1340.0,382.0,<NA>,<NA>,0.009387,<NA>,0.032928,10.613861,NaN
3,272817,AT0000606306,2006-12-31,EUR,55867.0,<NA>,1082.0,1677.0,1182.0,52.732,142.508,2006,1.0,55867.0,<NA>,1082.0,1677.0,1182.0,<NA>,<NA>,0.021157,<NA>,0.030018,10.930729,NaN
4,272817,AT0000606306,2007-12-31,EUR,72742.805,<NA>,3411.209,16964.595,841.258,58.365,153.841,2007,1.0,72742.805,<NA>,3411.209,16964.595,841.258,103.6,15937.9276,0.011565,<NA>,0.233213,11.194685,9.676457


**Section output:** `df_funda` now contains the firm-year controls needed for the final analysis panel: nominal EUR fundamentals, market capitalization, profitability, leverage, and log size variables. The next section merges these firm controls with `ceo_pay_panel` and executive characteristics, then applies CPI adjustment to express monetary variables in real 2015 EUR.

---

<a id="final-analysis-panel"></a>
## 5.4 Final Analysis Panel

<a id="construct-final-panel"></a>
### 5.4.1 Merge CEO pay panel with firm fundamentals and individuals profile details

In [32]:
df_panel_final = (
    ceo_pay_panel
    .merge(
        df_funda.drop(columns="fx_rate_avg_to_eur", errors="ignore"),
        on=["isin", "annualreportyear"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        df_profile.drop(columns=["dob", "dod"], errors="ignore"),
        on="directorid",
        how="left",
        validate="many_to_one"
    )
).sort_values(
    ["companyid", "annualreportyear"]
).reset_index(drop=True)
df_panel_final


_ = summarize_dataframe(df_panel_final)

### Dataset overview


,value
rows,2258
columns,52
duplicate rows,0
memory usage,2.16 MB


### Table overview


,companyid,isin,companyname,hocountryname,sector,directorid,directorname,rolename,datestartrole,dateendrole_clean,annualreportdate,annualreportyear,currency,totalcompensation,fx_rate_avg_to_eur,totalcompensation_eur,log_totalcompensation_eur,ceo_turnover,turnover_year,turnover_year_event,num_ceo_turnovers,ceo_turnover_dummy,first_turnover_year,treated_firm,event_time,post_turnover,event_window_3yr,event_window_5yr,gvkey,datadate,curcd,at,sale,ebit,dltt,nicon,emp,cshoi,at_eur,sale_eur,ebit_eur,dltt_eur,nicon_eur,close_eur,market_cap_eur,roa,ebit_margin,leverage,log_assets_eur_nominal,log_market_cap_eur_nominal,age,gender
0,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,14884.0,Jörgen Centerman,President/CEO,2000-12-31,2002-09-05,2001-12-01,2001,USD,1787.0,1.11768,1997.294162,7.599549,0,2000,NaN,0,0,2002.0,1,-1.0,0,1,1,210418,2001-12-31,USD,32344.0,23726.0,466.0,5043.0,-130.0,156.865,1113.133,36150.24196,26518.07571,520.838881,5636.460246,-145.2984,<NA>,<NA>,-0.004019,0.019641,0.155918,10.495439,NaN,77,M
1,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2002-12-01,2002,USD,1419.0,1.060608,1505.002572,7.31655,1,2002,2002.0,1,1,2002.0,1,0.0,1,1,1,210418,2002-12-31,USD,29533.0,18295.0,598.0,5376.0,97.0,139.051,1113.179,31322.932317,19403.821039,634.243508,5701.827926,102.878964,<NA>,<NA>,0.003284,0.032687,0.182034,10.352106,NaN,86,M
2,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2003-12-01,2003,USD,2414.0,0.88509,2136.607095,7.666974,1,2002,NaN,0,0,2002.0,1,1.0,1,1,1,210418,2003-12-31,USD,30413.0,18795.0,656.0,6290.0,86.0,116.464,2028.405,26918.240097,16635.265269,580.618995,5567.215671,76.117734,<NA>,<NA>,0.002828,0.034903,0.206819,10.200559,NaN,86,M


### Column coverage and dtypes


...

,count,mean,std,min,25%,50%,75%,max
companyid,2258.0,117401.582374,420311.176641,422.0,8824.0,16475.0,27916.0,3486568.0
directorid,2258.0,335920.517272,454200.13843,61.0,12502.0,31875.0,515002.0,2703829.0
annualreportyear,2258.0,2013.233835,7.099787,1997.0,2007.0,2014.0,2019.0,2025.0
totalcompensation,2258.0,2741.381311,1743.349026,2.0,1544.0,2413.5,3630.75,13591.0
fx_rate_avg_to_eur,2258.0,0.845363,0.098536,0.683517,0.75526,0.8477,0.904276,1.11768
totalcompensation_eur,2258.0,2292.066388,1436.137886,1.848976,1271.483507,2032.09101,2992.921005,10297.6674
log_totalcompensation_eur,2258.0,7.508506,0.780977,0.614632,7.14794,7.616821,8.004005,9.239673
ceo_turnover,2258.0,0.695306,0.46038,0.0,0.0,1.0,1.0,1.0
turnover_year,2258.0,2008.553587,8.513328,1976.0,2003.0,2009.0,2015.0,2025.0
turnover_year_event,278.0,2012.467626,7.183819,1997.0,2006.25,2012.0,2019.0,2025.0


### Categorical columns


,count,missing,unique,top,top_frequency
column,,,,,
isin,2258,0,137,SE0000115446,27
companyname,2258,0,137,VOLVO AB,27
hocountryname,2258,0,15,France,726
sector,2258,0,32,Banks,241
directorname,2258,0,383,Michael O'Leary,26
rolename,2258,0,38,Chairman/CEO,949
currency,2258,0,1,USD,2258
gvkey,2258,0,137,011217,27
curcd,2258,0,10,EUR,1827


### Datetime columns


,min,max,missing
datestartrole,1976-01-02,2025-07-01,0
dateendrole_clean,1998-03-30,2026-05-31,528
annualreportdate,1997-11-01,2025-12-01,0
datadate,1997-11-30,2025-12-31,0


#### Confirm that the final table still has exactly one observation per firm-year

In [33]:
duplicate_firm_years = df_panel_final.duplicated(
    subset=["companyid", "annualreportyear"],
    keep=False,
).sum()

print(f"Final panel shape: {df_panel_final.shape}")
print(f"Unique companies: {df_panel_final['companyid'].nunique():,}")
print(f"Duplicate firm-year rows: {duplicate_firm_years:,}")

Final panel shape: (2258, 52)
Unique companies: 137
Duplicate firm-year rows: 0


**Insights:**

- The final merged panel contains 2,258 firm-year CEO observations across 137 firms, with no duplicate firm-year rows. This confirms that the CEO-pay, fundamentals, market-price, and profile merges preserve the intended one-row-per-firm-year structure.

- CEO compensation coverage is complete in the retained panel: `totalcompensation_eur` and `log_totalcompensation_eur` have no missing values. Average CEO compensation is about 2,292 thousand EUR, with a median of about 2,032 thousand EUR.

- Firm fundamentals merge well overall. EBIT and CEO-pay variables are complete, while sales-based ratios remain less complete because some firm-years do not have all accounting inputs needed for those measures.

- Market capitalization is available for 1,690 observations, or about 75% of the panel. The remaining missing values mainly reflect missing matched market prices or shares outstanding before the market-capitalization step.

- Turnover variables are structurally complete: `ceo_turnover_dummy`, `num_ceo_turnovers`, `treated_firm`, `post_turnover`, and event-window indicators have no missing values. CEO-turnover years account for about 12.3% of firm-year observations.

- `dateendrole_clean` is missing for 528 rows, which is expected because ongoing CEO spells have no observed end date.

- The panel is strongly male-dominated: 2,203 of 2,258 observations are male CEOs. France and Banks are the most frequent country and sector groups, so later analysis should be aware of this sample composition.

---

---

<a id="cpi-adjustment"></a>
### 5.4.2 Convert Nominal EUR to Real 2015 EUR

The previous steps make monetary variables comparable across currencies by converting them to EUR. This subsection adds the second comparability layer: inflation adjustment. Because a euro amount from 2005 and a euro amount from 2020 do not represent the same purchasing power, we deflate all nominal-EUR monetary variables with headquarters-country CPI data.

The workflow is: map each firm headquarters country to an ISO3 code, retrieve monthly CPI from `comp.g_ecind_mth`, aggregate CPI to country-year averages, rebase each country to 2015 = 100, and merge the resulting CPI index onto the firm-year panel. Monetary variables are then converted into real EUR values expressed in 2015 price levels.

**Interpretation:** a real-EUR value measures purchasing value in 2015 prices. Original local-currency and nominal-EUR columns remain in the panel, so later analysis can distinguish currency conversion from inflation adjustment.

#### Map headquarters country names to CPI ISO3 codes

CPI is stored by country code in Compustat Global, while the panel stores headquarters countries as names. This step creates `econiso`, an ISO3 country code used only for the CPI lookup, without overwriting the original headquarters-country label.

In [34]:

def country_to_iso3(country_name):
    """
    Convert headquarters country name to ISO3 code without modifying
    the original hocountryname column.

    Uses temporary lookup rules for non-standard country-name formats.
    """
    if pd.isna(country_name):
        return None

    country_name = str(country_name).strip()

    # Temporary lookup name used only for pycountry
    lookup_name = country_name

    # General rule: detect any United Kingdom variant
    if "United Kingdom" in country_name:
        lookup_name = "United Kingdom"

    # General rule: detect Republic of Ireland variant
    elif "Ireland" in country_name:
        lookup_name = "Ireland"

    try:
        return pycountry.countries.lookup(lookup_name).alpha_3
    except LookupError:
        return None

# Set econiso by applying the function country_to_iso3 based on hocountryname
df_panel_final["econiso"] = df_panel_final["hocountryname"].apply(country_to_iso3)

# Retrieved iso3
df_panel_final.econiso.unique()

/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/4201274103.py:30: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_panel_final["econiso"] = df_panel_final["hocountryname"].apply(country_to_iso3)


array(['CHE', 'FRA', 'DEU', 'NLD', 'IRL', 'GBR', 'SWE', 'ESP', 'ITA',
       'LUX', 'FIN', 'BEL', 'NOR', 'IMN', 'JEY'], dtype=object)

#### Retrieve monthly CPI for panel countries

The CPI query is restricted to the countries that actually appear in `df_panel_final`. This keeps the WRDS pull small and makes the later CPI merge easier to validate.

In [35]:
iso3_tuple = tuple(df_panel_final.econiso.unique().tolist())

df_econ = conn.raw_sql(
    """
    SELECT
        datadate,
        econiso,
        cpi1
    FROM comp.g_ecind_mth
    WHERE econiso IN %(needed_iso3)s
    """,
    params={"needed_iso3": iso3_tuple}
)

# Convert date column to datetime format
df_econ["datadate"] = pd.to_datetime(df_econ["datadate"], errors="coerce")

df_econ = df_econ.sort_values(by=["econiso", "datadate"]).reset_index(drop=True)

df_econ.head()

/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/4052613415.py:16: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_econ["datadate"] = pd.to_datetime(df_econ["datadate"], errors="coerce")


,datadate,econiso,cpi1
0,1981-01-31,BEL,45.8535
1,1981-02-28,BEL,45.8535
2,1981-03-31,BEL,45.8535
3,1981-04-30,BEL,45.8535
4,1981-05-31,BEL,45.8535


The CPI is first rebased so that each country has the same base year, with 2015 set equal to 100:

$$
CPI^{2015base}_{c,t}
=
\frac{CPI_{c,t}}{CPI_{c,2015}}
\times 100
$$

where $CPI_{c,t}$ is the annual average CPI of country $c$ in year $t$, and $CPI_{c,2015}$ is the annual average CPI of the same country in the 2015 base year.

Nominal monetary values are then deflated using the rebased CPI:

$$
RealValue^{2015}_{i,t}
=
\frac{NominalValue_{i,t}}{CPI^{2015base}_{c,t}}
\times 100
$$

This expresses each monetary value in 2015 price terms.

#### Build annual CPI averages and the 2015 index

Monthly CPI observations are collapsed to annual country-year averages because the analysis panel is annual. The 2015 CPI value is then broadcast within each country and used to construct the normalized CPI index.

In [36]:
# Collapse monthly CPI observations into annual average CPI by country-year
df_cpi_annual = (
    df_econ
    .dropna(subset=["econiso", "datadate", "cpi1"])
    .assign(year=lambda x: x["datadate"].dt.year)
    .groupby(["econiso", "year"], as_index=False)
    .agg(annual_avg_cpi=("cpi1", "mean"))
)

# Attach each country's 2015 CPI average directly within the grouped CPI table
df_cpi_annual["annual_avg_cpi_2015"] = (
    df_cpi_annual["annual_avg_cpi"]
    .where(df_cpi_annual["year"].eq(2015))
    .groupby(df_cpi_annual["econiso"])
    .transform("first")
)

# Normalize CPI to each country's 2015 price level
df_cpi_annual["annual_avg_cpi_index_2015"] = (
    df_cpi_annual["annual_avg_cpi"]
    .div(df_cpi_annual["annual_avg_cpi_2015"])
    .mul(100)
)

df_cpi_annual.head(3)

/Users/kajetan/Projects/TUdortmund/dai-mission-group-Q/.venv/lib/python3.14/site-packages/pandas/core/frame.py:5239: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  data[k] = com.apply_if_callable(v, data)
/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9

,econiso,year,annual_avg_cpi,annual_avg_cpi_2015,annual_avg_cpi_index_2015
0,BEL,1981,45.8535,92.7947,49.413921
1,BEL,1982,49.8548,92.7947,53.725913
2,BEL,1983,53.6775,92.7947,57.845437


#### Validate CPI country coverage and merge the CPI index

Before merging CPI into the panel, we check whether any panel countries are absent from the CPI pull. Small financial centers and crown dependencies can require proxy CPI countries because they are not always reported as separate CPI economies in `comp.g_ecind_mth`.

In [37]:
# Iso3 codes that cannot be matched in df_econ
missing_in_df_econ = (set(df_panel_final["econiso"].dropna().unique()) - set(df_econ["econiso"].dropna().unique()))

missing_in_df_econ

{'IMN', 'JEY', 'LUX'}

**Insights:**

- The CPI pull does not directly cover `IMN`, `JEY`, and `LUX`. Rather than dropping these firm-years, the panel uses transparent proxy rules: Isle of Man and Jersey use the United Kingdom (`GBR`), while Luxembourg uses Belgium (`BEL`).

- These proxies affect only the CPI lookup country in `cpi_proxy_econiso`. The original headquarters ISO3 code remains in `econiso`, so the panel still preserves the firm location used elsewhere in the analysis.

In [38]:
# Define CPI proxy countries for places without a directly usable CPI series
# Luxembourg uses Belgium; Isle of Man and Jersey use the United Kingdom
CPI_PROXY_ISO3 = {
    "LUX": "BEL",
    "IMN": "GBR",
    "JEY": "GBR"
}

# Create the CPI lookup country:
# use the original firm country unless it appears in the proxy dictionary above
df_panel_final["cpi_proxy_econiso"] = (
    df_panel_final["econiso"].replace(CPI_PROXY_ISO3)
)

# Merge annual CPI information into the firm-year panel
# Match on proxy country and annual report year, because CPI is measured by country-year
df_panel_final = df_panel_final.merge(
    df_cpi_annual,
    left_on=["cpi_proxy_econiso", "annualreportyear"],
    right_on=["econiso", "year"],
    how="left",
    validate="many_to_one",
    suffixes=("", "_cpi")
)

# Remove duplicate CPI-side country/year columns after the merge
df_panel_final = df_panel_final.drop(columns=["econiso_cpi", "year"], errors="ignore")

# Preview the updated panel with CPI variables attached
df_panel_final.head()

/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/562665556.py:11: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_panel_final["cpi_proxy_econiso"] = (
/Users/kajetan/Projects/TUdortmund/dai-mission-group-Q/.venv/lib/python3.1

,companyid,isin,companyname,hocountryname,sector,directorid,directorname,rolename,datestartrole,dateendrole_clean,annualreportdate,annualreportyear,currency,totalcompensation,fx_rate_avg_to_eur,totalcompensation_eur,log_totalcompensation_eur,ceo_turnover,turnover_year,turnover_year_event,num_ceo_turnovers,ceo_turnover_dummy,first_turnover_year,treated_firm,event_time,post_turnover,event_window_3yr,event_window_5yr,gvkey,datadate,curcd,at,sale,ebit,dltt,nicon,emp,cshoi,at_eur,sale_eur,ebit_eur,dltt_eur,nicon_eur,close_eur,market_cap_eur,roa,ebit_margin,leverage,log_assets_eur_nominal,log_market_cap_eur_nominal,age,gender,econiso,cpi_proxy_econiso,annual_avg_cpi,annual_avg_cpi_2015,annual_avg_cpi_index_2015
0,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,14884.0,Jörgen Centerman,President/CEO,2000-12-31,2002-09-05,2001-12-01,2001,USD,1787.0,1.11768,1997.294162,7.599549,0,2000,NaN,0,0,2002.0,1,-1.0,0,1,1,210418,2001-12-31,USD,32344.0,23726.0,466.0,5043.0,-130.0,156.865,1113.133,36150.24196,26518.07571,520.838881,5636.460246,-145.2984,<NA>,<NA>,-0.004019,0.019641,0.155918,10.495439,NaN,77,M,CHE,CHE,93.0786,98.6186,94.382398
1,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2002-12-01,2002,USD,1419.0,1.060608,1505.002572,7.31655,1,2002,2002.0,1,1,2002.0,1,0.0,1,1,1,210418,2002-12-31,USD,29533.0,18295.0,598.0,5376.0,97.0,139.051,1113.179,31322.932317,19403.821039,634.243508,5701.827926,102.878964,<NA>,<NA>,0.003284,0.032687,0.182034,10.352106,NaN,86,M,CHE,CHE,93.6769,98.6186,94.989079
2,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2003-12-01,2003,USD,2414.0,0.88509,2136.607095,7.666974,1,2002,NaN,0,0,2002.0,1,1.0,1,1,1,210418,2003-12-31,USD,30413.0,18795.0,656.0,6290.0,86.0,116.464,2028.405,26918.240097,16635.265269,580.618995,5567.215671,76.117734,<NA>,<NA>,0.002828,0.034903,0.206819,10.200559,NaN,86,M,CHE,CHE,94.2748,98.6186,95.595354
3,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2004-12-01,2004,USD,3383.0,0.804927,2723.067443,7.909514,1,2002,NaN,0,0,2002.0,1,2.0,1,1,1,210418,2004-12-31,USD,24677.0,20721.0,1087.0,4901.0,448.0,102.537,2028.405,19863.179216,16678.888704,874.955457,3944.946361,360.607217,<NA>,<NA>,0.018155,0.052459,0.198606,9.896623,NaN,86,M,CHE,CHE,95.0317,98.6186,96.362856
4,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,4611.0,Fred Kindle,President/CEO,2005-01-01,2008-02-13,2005-12-01,2005,USD,1299.0,0.804784,1045.414149,6.952168,1,2005,2005.0,1,1,2002.0,1,3.0,1,1,1,210418,2005-12-31,USD,22276.0,22442.0,1798.0,3933.0,883.0,103.5,2035.111,17927.3638,18060.957909,1447.001262,3165.214663,710.62409,<NA>,<NA>,0.039639,0.080118,0.176558,9.794084,NaN,67,M,CHE,CHE,96.1455,98.6186,97.492258


#### Deflate nominal-EUR variables into real 2015 EUR

Using the merged CPI index, nominal-EUR monetary variables are converted to real 2015 EUR. The transformation is applied consistently to CEO compensation, accounting fundamentals, debt, profitability, and market capitalization. Log variables are created only where the corresponding real value is positive.

In [39]:
# Monetary variables
nominal_eur_cols = [
    "totalcompensation_eur",
    "at_eur",
    "sale_eur",
    "ebit_eur",
    "dltt_eur",
    "nicon_eur",
    "market_cap_eur",
]

for col in nominal_eur_cols:
    real_col = f"real_{col.replace('_eur', '_eur_2015')}"
    df_panel_final[real_col] = (
        df_panel_final[col] / df_panel_final["annual_avg_cpi_index_2015"] * 100
    )

# Log-transform positive real values for the main scale variables
df_panel_final["log_real_totalcompensation_eur_2015"] = np.nan
df_panel_final.loc[
    df_panel_final["real_totalcompensation_eur_2015"] > 0,
    "log_real_totalcompensation_eur_2015"
] = np.log(
    df_panel_final.loc[
        df_panel_final["real_totalcompensation_eur_2015"] > 0,
        "real_totalcompensation_eur_2015"
    ].astype(float)
)

df_panel_final["log_real_assets_eur_2015"] = np.nan
df_panel_final.loc[
    df_panel_final["real_at_eur_2015"] > 0,
    "log_real_assets_eur_2015"
] = np.log(
    df_panel_final.loc[
        df_panel_final["real_at_eur_2015"] > 0,
        "real_at_eur_2015"
    ].astype(float)
)

df_panel_final["log_real_market_cap_eur_2015"] = np.nan
df_panel_final.loc[
    df_panel_final["real_market_cap_eur_2015"] > 0,
    "log_real_market_cap_eur_2015"
] = np.log(
    df_panel_final.loc[
        df_panel_final["real_market_cap_eur_2015"] > 0,
        "real_market_cap_eur_2015"
    ].astype(float)
)

_ = summarize_dataframe(df_panel_final)

### Dataset overview


/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/2304612686.py:14: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_panel_final[real_col] = (
/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/2304612686.py:14: Fu

,value
rows,2258
columns,67
duplicate rows,0
memory usage,2.63 MB


### Table overview


,companyid,isin,companyname,hocountryname,sector,directorid,directorname,rolename,datestartrole,dateendrole_clean,annualreportdate,annualreportyear,currency,totalcompensation,fx_rate_avg_to_eur,totalcompensation_eur,log_totalcompensation_eur,ceo_turnover,turnover_year,turnover_year_event,num_ceo_turnovers,ceo_turnover_dummy,first_turnover_year,treated_firm,event_time,post_turnover,event_window_3yr,event_window_5yr,gvkey,datadate,curcd,at,sale,ebit,dltt,nicon,emp,cshoi,at_eur,sale_eur,ebit_eur,dltt_eur,nicon_eur,close_eur,market_cap_eur,roa,ebit_margin,leverage,log_assets_eur_nominal,log_market_cap_eur_nominal,age,gender,econiso,cpi_proxy_econiso,annual_avg_cpi,annual_avg_cpi_2015,annual_avg_cpi_index_2015,real_totalcompensation_eur_2015,real_at_eur_2015,real_sale_eur_2015,real_ebit_eur_2015,real_dltt_eur_2015,real_nicon_eur_2015,real_market_cap_eur_2015,log_real_totalcompensation_eur_2015,log_real_assets_eur_2015,log_real_market_cap_eur_2015
0,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,14884.0,Jörgen Centerman,President/CEO,2000-12-31,2002-09-05,2001-12-01,2001,USD,1787.0,1.11768,1997.294162,7.599549,0,2000,NaN,0,0,2002.0,1,-1.0,0,1,1,210418,2001-12-31,USD,32344.0,23726.0,466.0,5043.0,-130.0,156.865,1113.133,36150.24196,26518.07571,520.838881,5636.460246,-145.2984,<NA>,<NA>,-0.004019,0.019641,0.155918,10.495439,NaN,77,M,CHE,CHE,93.0786,98.6186,94.382398,2116.17229,38301.889498,28096.420672,551.838997,5971.940043,-153.946501,<NA>,7.657364,10.553255,NaN
1,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2002-12-01,2002,USD,1419.0,1.060608,1505.002572,7.31655,1,2002,2002.0,1,1,2002.0,1,0.0,1,1,1,210418,2002-12-31,USD,29533.0,18295.0,598.0,5376.0,97.0,139.051,1113.179,31322.932317,19403.821039,634.243508,5701.827926,102.878964,<NA>,<NA>,0.003284,0.032687,0.182034,10.352106,NaN,86,M,CHE,CHE,93.6769,98.6186,94.989079,1584.39537,32975.298425,20427.423041,667.701502,6002.614172,108.306096,<NA>,7.367958,10.403514,NaN
2,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2003-12-01,2003,USD,2414.0,0.88509,2136.607095,7.666974,1,2002,NaN,0,0,2002.0,1,1.0,1,1,1,210418,2003-12-31,USD,30413.0,18795.0,656.0,6290.0,86.0,116.464,2028.405,26918.240097,16635.265269,580.618995,5567.215671,76.117734,<NA>,<NA>,0.002828,0.034903,0.206819,10.200559,NaN,86,M,CHE,CHE,94.2748,98.6186,95.595354,2235.053275,28158.523305,17401.750749,607.371561,5823.730365,79.62493,<NA>,7.712020,10.245605,NaN


...

,count,mean,std,min,25%,50%,75%,max
companyid,2258.0,117401.582374,420311.176641,422.0,8824.0,16475.0,27916.0,3486568.0
directorid,2258.0,335920.517272,454200.13843,61.0,12502.0,31875.0,515002.0,2703829.0
annualreportyear,2258.0,2013.233835,7.099787,1997.0,2007.0,2014.0,2019.0,2025.0
totalcompensation,2258.0,2741.381311,1743.349026,2.0,1544.0,2413.5,3630.75,13591.0
fx_rate_avg_to_eur,2258.0,0.845363,0.098536,0.683517,0.75526,0.8477,0.904276,1.11768
totalcompensation_eur,2258.0,2292.066388,1436.137886,1.848976,1271.483507,2032.09101,2992.921005,10297.6674
log_totalcompensation_eur,2258.0,7.508506,0.780977,0.614632,7.14794,7.616821,8.004005,9.239673
ceo_turnover,2258.0,0.695306,0.46038,0.0,0.0,1.0,1.0,1.0
turnover_year,2258.0,2008.553587,8.513328,1976.0,2003.0,2009.0,2015.0,2025.0
turnover_year_event,278.0,2012.467626,7.183819,1997.0,2006.25,2012.0,2019.0,2025.0


### Categorical columns


,count,missing,unique,top,top_frequency
column,,,,,
isin,2258,0,137,SE0000115446,27
companyname,2258,0,137,VOLVO AB,27
hocountryname,2258,0,15,France,726
sector,2258,0,32,Banks,241
directorname,2258,0,383,Michael O'Leary,26
rolename,2258,0,38,Chairman/CEO,949
currency,2258,0,1,USD,2258
gvkey,2258,0,137,011217,27
curcd,2258,0,10,EUR,1827


### Datetime columns


,min,max,missing
datestartrole,1976-01-02,2025-07-01,0
dateendrole_clean,1998-03-30,2026-05-31,528
annualreportdate,1997-11-01,2025-12-01,0
datadate,1997-11-30,2025-12-31,0


**Insights:**

- CPI adjustment is complete for the main CEO-pay outcome: `real_totalcompensation_eur_2015` and `log_real_totalcompensation_eur_2015` are available for all 2,258 firm-year observations in the saved output.

- The CPI index is close to 100 on average because the sample is centered around the 2015 base year, but it still corrects both earlier and later nominal-EUR values for country-specific price levels.

- Real market capitalization has the same coverage as nominal market capitalization: 1,690 observations. CPI adjustment does not create new market data; it only converts available nominal-EUR values into 2015 price terms.

- After this step, the panel contains both nominal-EUR and real-2015-EUR monetary variables. Later sections use the real and log-real variables for CEO-pay analysis so that changes over time are not driven mechanically by inflation.

---

<a id="winsorization"></a>
### 5.4.3 Winsorization of CEO Pay (p1 / p99)

The real CEO-pay variable is now comparable across currencies and over time, but extreme observations can still dominate averages, regressions, and prediction models. We therefore winsorize real CEO compensation at the 1st and 99th percentiles.

The winsorized level variable, `w_real_totalcompensation_eur_2015`, keeps the original 2015-EUR interpretation while limiting the influence of the most extreme tails. Its log transformation, `log_w_real_pay`, becomes the main CEO-pay outcome used in later causal-inference and supervised-learning sections.

In [40]:
# Winsorize real compensation at p1/p99 while preserving missing values
def winsorize_series(s, lo=0.01, hi=0.99):
    """Return a clipped series and the lower/upper cutoffs used."""
    q_lo, q_hi = s.dropna().quantile([lo, hi])
    return s.clip(lower=q_lo, upper=q_hi), q_lo, q_hi

# Optional pay components are included only when they exist in the current panel
real_pay_mapping = {
    "real_totalcompensation_eur_2015": "w_real_totalcompensation_eur_2015",
    "real_salary_eur_2015": "w_real_salary_eur_2015",
    "real_bonus_eur_2015": "w_real_bonus_eur_2015",
    "real_totaldirectcomp_eur_2015": "w_real_totaldirectcomp_eur_2015",
}

winsor_cutoffs = {}
for src_col, dst_col in real_pay_mapping.items():
    if src_col in df_panel_final.columns:
        df_panel_final[dst_col], q_lo, q_hi = winsorize_series(df_panel_final[src_col])
        winsor_cutoffs[src_col] = {"p1": q_lo, "p99": q_hi}

# Primary CEO-pay outcome: log of winsorized real total compensation
df_panel_final["log_w_real_pay"] = np.log(
    df_panel_final["w_real_totalcompensation_eur_2015"].where(
        df_panel_final["w_real_totalcompensation_eur_2015"] > 0
    )
)

# Compact validation table for the main pay outcome
winsor_summary = df_panel_final[[
    "real_totalcompensation_eur_2015",
    "w_real_totalcompensation_eur_2015",
    "log_w_real_pay",
]].describe().T

main_p1 = winsor_cutoffs["real_totalcompensation_eur_2015"]["p1"]
main_p99 = winsor_cutoffs["real_totalcompensation_eur_2015"]["p99"]
n_low_clip = df_panel_final["real_totalcompensation_eur_2015"].lt(main_p1).sum()
n_high_clip = df_panel_final["real_totalcompensation_eur_2015"].gt(main_p99).sum()

display(winsor_summary)
print(f"CEO pay winsorization cutoffs: p1={main_p1:,.0f}, p99={main_p99:,.0f} thousand EUR at 2015 prices")
print(f"Rows clipped at lower tail: {n_low_clip:,}")
print(f"Rows clipped at upper tail: {n_high_clip:,}")

/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/3503888584.py:18: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_panel_final[dst_col], q_lo, q_hi = winsorize_series(df_panel_final[src_col])
/var/folders/wx/b_h8zt7d5ps2y7dhc

,count,mean,std,min,25%,50%,75%,max
real_totalcompensation_eur_2015,2258.0,2296.072096,1430.566292,1.499228,1299.190853,2032.285651,2993.278584,11181.360376
w_real_totalcompensation_eur_2015,2258.0,2283.753913,1371.869279,135.794919,1299.190853,2032.285651,2993.278584,7459.528786
log_w_real_pay,2258.0,7.528935,0.707059,4.911146,7.169497,7.616916,8.004125,8.917248


CEO pay winsorization cutoffs: p1=136, p99=7,460 thousand EUR at 2015 prices
Rows clipped at lower tail: 23
Rows clipped at upper tail: 23


**Insights:**

- Winsorization is applied after CPI adjustment, so the clipping thresholds are defined in real 2015 EUR rather than nominal currency units.

- Only the tails are clipped. The median and most observations remain unchanged, while very small and very large CEO-pay observations are limited before modelling.

- `log_w_real_pay` is the primary later-stage CEO-pay outcome. Keeping both the raw real value and the winsorized/log version makes the modelling choice transparent instead of overwriting the original compensation measure.

<a id="pay-dispersion-construction"></a>
### 5.4.4 Construct CEO-to-Executive Pay Dispersion

The final panel contains one CEO observation per firm-year, but the second outcome in this project is internal pay dispersion. We define pay dispersion as the ratio between CEO total compensation and the median total compensation of other eligible senior executives in the same firm-year. Formally:

$$\text{pay\_ratio}_{it} = \frac{\text{CEO total comp}_{it}}{\text{median(other exec total comp)}_{it}}$$

This subsection follows the same matching logic used in Section 5.2 for CEO pay: start from role spells, keep the relevant role population, merge to remuneration records by director and firm, keep compensation reports that fall inside the active role window, convert pay to EUR, and then collapse to one firm-year denominator.

The ratio is built in nominal EUR because CEO pay and non-CEO executive pay are observed in the same firm-year. A country-year CPI adjustment would scale both numerator and denominator by the same factor, so it would not change the ratio.

#### Filter companies and clean role names

The denominator should be based only on firms that appear in the final CEO panel. We first subset the employment-spell table to those firms and standardize role names before applying inclusion and exclusion rules.

In [41]:
# Restrict employment roles to companies that appear in the final CEO panel
company_list = df_panel_final["companyid"].dropna().unique().tolist()

df_emp_filtered = df_emp.loc[df_emp["companyid"].isin(company_list)].copy()

# Standardize role names before applying keyword rules
df_emp_filtered["rolename_clean"] = (df_emp_filtered["rolename"].fillna("").str.strip())

print(f"Employment role spells in panel firms: {len(df_emp_filtered):,}")
print(f"Unique role names in panel firms: {df_emp_filtered['rolename_clean'].nunique():,}")

Employment role spells in panel firms: 94,183
Unique role names in panel firms: 7,482


/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/3961073599.py:7: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_emp_filtered["rolename_clean"] = (df_emp_filtered["rolename"].fillna("").str.strip())


#### Keep senior non-CEO operational roles

Mirroring the Section 5.2 CEO-role construction, we build transparent boolean rules instead of manually selecting individual role names. The denominator keeps senior operational roles and removes CEO, board-only, supervisory, representative, advisory, junior, and support roles.

In [42]:
# Broad senior operational roles that can enter the pay-ratio denominator
has_senior_exec_title = df_emp_filtered["rolename_clean"].str.contains(
    r"\bchief\b|"
    r"\bcfo\b|\bcoo\b|\bcio\b|\bcto\b|\bcmo\b|\bchro\b|"
    r"\bexecutive\b|\bofficer\b|"
    r"\bpresident\b|\bvice president\b|\bvp\b|\bsvp\b|\bevp\b|"
    r"\bmd\b|managing director|general manager|"
    r"\bhead of\b|global head|group head|regional head|division head|co-head|"
    r"\bdirector\b|\bpartner\b|\bprincipal\b|"
    r"\btreasurer\b|\bcontroller\b|financial controller|"
    r"general counsel|legal counsel|\bfd\b|\bgfd\b|\bgfc\b",
    case=False,
    na=False,
    regex=True,
)

# Exclude CEO roles because the denominator should contain non-CEO executives
exclude_ceo_roles = df_emp_filtered["rolename_clean"].str.contains(
    r"\bceo\b|chief executive",
    case=False,
    na=False,
    regex=True,
)

# Exclude board-only, supervisory, representative, and non-executive roles
exclude_board_or_supervisory_roles = df_emp_filtered["rolename_clean"].str.contains(
    r"independent|\bned\b|non[- ]?executive|supervisory|"
    r"shareholder|employee representative|government representative|"
    r"board member|committee|council|"
    r"\bchair\b|chairman|chairwoman|chairperson|vice chair|deputy chair",
    case=False,
    na=False,
    regex=True,
)

# Exclude advisory, audit-only, junior, and support roles
exclude_non_operational_roles = df_emp_filtered["rolename_clean"].str.contains(
    r"\bauditor\b|statutory auditor|"
    r"advisory|advisor|consultant|analyst|"
    r"intern|trainee|apprentice|junior|"
    r"honorary|representative director|"
    r"associate|assistant(?!.*(general manager|chief|executive))",
    case=False,
    na=False,
    regex=True,
)

# Final denominator-role flag
df_emp_filtered["pay_ratio_denominator_role"] = (
    has_senior_exec_title
    & ~exclude_ceo_roles
    & ~exclude_board_or_supervisory_roles
    & ~exclude_non_operational_roles
)

# Keep only employment spells relevant for the non-CEO executive denominator
df_emp_pay_ratio_roles = df_emp_filtered.loc[df_emp_filtered["pay_ratio_denominator_role"]].copy()

print(f"Eligible pay-ratio role spells: {len(df_emp_pay_ratio_roles):,}")
print(f"Eligible unique role names: {df_emp_pay_ratio_roles['rolename_clean'].nunique():,}")

Eligible pay-ratio role spells: 51,882
Eligible unique role names: 5,512


/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/3488838521.py:37: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  exclude_non_operational_roles = df_emp_filtered["rolename_clean"].str.contains(
/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/3488838521.py:49: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensur

#### Match eligible role spells to remuneration records

This step mirrors the CEO-remuneration matching logic in Section 5.2. We merge remuneration records to eligible role spells using both person and firm identifiers, then keep reports whose annual report date falls within the role spell, allowing the same 90-day tolerance around start and end dates.

In [43]:
# Prepare role spells that are eligible for the pay-ratio denominator
pay_ratio_role_spells = df_emp_pay_ratio_roles[[
    "companyid",
    "directorid",
    "directorname",
    "rolename",
    "rolename_clean",
    "datestartrole",
    "dateendrole_clean",
]].copy()

# Ensure merge keys and dates are comparable
pay_ratio_role_spells["companyid"] = pd.to_numeric(
    pay_ratio_role_spells["companyid"],
    errors="coerce",
)
pay_ratio_role_spells["directorid"] = pd.to_numeric(
    pay_ratio_role_spells["directorid"],
    errors="coerce",
)
pay_ratio_role_spells["datestartrole"] = pd.to_datetime(
    pay_ratio_role_spells["datestartrole"],
    errors="coerce",
)
pay_ratio_role_spells["dateendrole_clean"] = pd.to_datetime(
    pay_ratio_role_spells["dateendrole_clean"],
    errors="coerce",
)

# Prepare remuneration records for the same director-firm matching logic used in Section 5.2
df_remun_pay_ratio = df_remun[["boardid","directorid","annualreportdate","currency","totalcompensation",]].copy()

df_remun_pay_ratio["boardid"] = pd.to_numeric(
    df_remun_pay_ratio["boardid"],
    errors="coerce",
)
df_remun_pay_ratio["directorid"] = pd.to_numeric(
    df_remun_pay_ratio["directorid"],
    errors="coerce",
)
df_remun_pay_ratio["annualreportdate"] = pd.to_datetime(
    df_remun_pay_ratio["annualreportdate"],
    errors="coerce",
)
df_remun_pay_ratio["totalcompensation"] = pd.to_numeric(
    df_remun_pay_ratio["totalcompensation"],
    errors="coerce",
)

# Merge remuneration to eligible non-CEO executive role spells
df_pay_ratio_remun_matched = df_remun_pay_ratio.merge(
    pay_ratio_role_spells,
    left_on=["directorid", "boardid"],
    right_on=["directorid", "companyid"],
    how="inner",
)

# Keep remuneration records whose annual report date falls inside the role spell
df_pay_ratio_remun_matched = df_pay_ratio_remun_matched.loc[
    (
        df_pay_ratio_remun_matched["annualreportdate"]
        >= df_pay_ratio_remun_matched["datestartrole"] - pd.Timedelta(days=90)
    )
    & (
        df_pay_ratio_remun_matched["dateendrole_clean"].isna()
        | (
            df_pay_ratio_remun_matched["annualreportdate"]
            <= df_pay_ratio_remun_matched["dateendrole_clean"] + pd.Timedelta(days=90)
        )
    )
].copy()

# Keep valid dated records with positive compensation
df_pay_ratio_remun_matched = df_pay_ratio_remun_matched.loc[
    df_pay_ratio_remun_matched["annualreportdate"].notna()
    & df_pay_ratio_remun_matched["totalcompensation"].gt(0)
].copy()

# Add firm-year key for later aggregation
df_pay_ratio_remun_matched["annualreportyear"] = (
    df_pay_ratio_remun_matched["annualreportdate"].dt.year
)

print(f"Matched non-CEO executive remuneration records: {len(df_pay_ratio_remun_matched):,}")

Matched non-CEO executive remuneration records: 5,432


/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/3366419220.py:13: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  pay_ratio_role_spells["companyid"] = pd.to_numeric(
/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_77

#### Convert matched executive pay to EUR

The matched non-CEO executive remuneration records are converted using the same annual-average FX table used for CEO compensation. This keeps numerator and denominator on the same nominal-EUR basis before calculating the ratio.

In [44]:
# Merge annual average FX rate based on remuneration currency and annual report year
df_pay_ratio_remun_matched = df_pay_ratio_remun_matched.merge(
    df_fx_annual,
    left_on=["currency", "annualreportyear"],
    right_on=["fromcurd", "year"],
    how="left",
    validate="many_to_one",
)

# Convert non-CEO executive compensation into EUR
df_pay_ratio_remun_matched["totalcompensation_eur"] = (
    df_pay_ratio_remun_matched["totalcompensation"]
    * df_pay_ratio_remun_matched["fx_rate_avg_to_eur"]
)

# Keep only valid positive EUR compensation
df_pay_ratio_remun_matched = df_pay_ratio_remun_matched.loc[
    df_pay_ratio_remun_matched["totalcompensation_eur"].gt(0)
].copy()

print(f"Matched records with positive EUR pay: {len(df_pay_ratio_remun_matched):,}")

Matched records with positive EUR pay: 5,432


/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/4194075848.py:11: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_pay_ratio_remun_matched["totalcompensation_eur"] = (


#### Aggregate the firm-year executive-pay denominator

Each firm-year denominator is the median total compensation of eligible non-CEO executives. We also keep the number of unique executives used in the denominator as a coverage-quality variable.

In [45]:
# Build firm-year denominator for CEO-to-executive pay ratio
exec_pay_agg = (
    df_pay_ratio_remun_matched
    .sort_values(["companyid", "directorid", "annualreportyear", "annualreportdate"])
    .drop_duplicates(
        subset=["companyid", "directorid", "annualreportyear"],
        keep="last",
    )
    .groupby(["companyid", "annualreportyear"], as_index=False)
    .agg(
        median_exec_pay_eur=("totalcompensation_eur", "median"),
        n_exec_in_ratio=("directorid", "nunique"),
    )
)

exec_pay_agg.head()

,companyid,annualreportyear,median_exec_pay_eur,n_exec_in_ratio
0,422.0,2002,1070.153344,3
1,422.0,2003,881.992117,4
2,422.0,2004,827.464774,3
3,422.0,2005,788.688118,5
4,422.0,2006,797.170545,8


#### Merge the denominator and construct the pay ratio

The denominator is merged onto the CEO-level panel at the firm-year level. The pay ratio compares CEO total compensation with median eligible non-CEO executive compensation in the same firm-year.

In [46]:
# Make repeated notebook runs safe by removing previously generated dispersion columns
pay_dispersion_cols = ["median_exec_pay_eur", "n_exec_in_ratio", "pay_ratio", "w_pay_ratio", "log_pay_dispersion"]

df_panel_final = df_panel_final.drop(
    columns=pay_dispersion_cols,
    errors="ignore",
)

# Ensure merge keys are comparable
df_panel_final["companyid"] = pd.to_numeric(
    df_panel_final["companyid"],
    errors="coerce",
)
exec_pay_agg["companyid"] = pd.to_numeric(
    exec_pay_agg["companyid"],
    errors="coerce",
)

# Merge one executive-pay denominator onto each CEO firm-year observation
df_panel_final = df_panel_final.merge(
    exec_pay_agg,
    on=["companyid", "annualreportyear"],
    how="left",
    validate="many_to_one",
)

# CEO-to-executive pay ratio
df_panel_final["pay_ratio"] = (
    df_panel_final["totalcompensation_eur"]
    / df_panel_final["median_exec_pay_eur"]
).replace([np.inf, -np.inf], np.nan)

df_panel_final.head()

/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/1587975870.py:10: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_panel_final["companyid"] = pd.to_numeric(
/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/1587

,companyid,isin,companyname,hocountryname,sector,directorid,directorname,rolename,datestartrole,dateendrole_clean,annualreportdate,annualreportyear,currency,totalcompensation,fx_rate_avg_to_eur,totalcompensation_eur,log_totalcompensation_eur,ceo_turnover,turnover_year,turnover_year_event,num_ceo_turnovers,ceo_turnover_dummy,first_turnover_year,treated_firm,event_time,post_turnover,event_window_3yr,event_window_5yr,gvkey,datadate,curcd,at,sale,ebit,dltt,nicon,emp,cshoi,at_eur,sale_eur,ebit_eur,dltt_eur,nicon_eur,close_eur,market_cap_eur,roa,ebit_margin,leverage,log_assets_eur_nominal,log_market_cap_eur_nominal,age,gender,econiso,cpi_proxy_econiso,annual_avg_cpi,annual_avg_cpi_2015,annual_avg_cpi_index_2015,real_totalcompensation_eur_2015,real_at_eur_2015,real_sale_eur_2015,real_ebit_eur_2015,real_dltt_eur_2015,real_nicon_eur_2015,real_market_cap_eur_2015,log_real_totalcompensation_eur_2015,log_real_assets_eur_2015,log_real_market_cap_eur_2015,w_real_totalcompensation_eur_2015,log_w_real_pay,median_exec_pay_eur,n_exec_in_ratio,pay_ratio
0,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,14884.0,Jörgen Centerman,President/CEO,2000-12-31,2002-09-05,2001-12-01,2001,USD,1787.0,1.11768,1997.294162,7.599549,0,2000,NaN,0,0,2002.0,1,-1.0,0,1,1,210418,2001-12-31,USD,32344.0,23726.0,466.0,5043.0,-130.0,156.865,1113.133,36150.24196,26518.07571,520.838881,5636.460246,-145.2984,<NA>,<NA>,-0.004019,0.019641,0.155918,10.495439,NaN,77,M,CHE,CHE,93.0786,98.6186,94.382398,2116.17229,38301.889498,28096.420672,551.838997,5971.940043,-153.946501,<NA>,7.657364,10.553255,NaN,2116.17229,7.657364,<NA>,NaN,<NA>
1,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2002-12-01,2002,USD,1419.0,1.060608,1505.002572,7.31655,1,2002,2002.0,1,1,2002.0,1,0.0,1,1,1,210418,2002-12-31,USD,29533.0,18295.0,598.0,5376.0,97.0,139.051,1113.179,31322.932317,19403.821039,634.243508,5701.827926,102.878964,<NA>,<NA>,0.003284,0.032687,0.182034,10.352106,NaN,86,M,CHE,CHE,93.6769,98.6186,94.989079,1584.39537,32975.298425,20427.423041,667.701502,6002.614172,108.306096,<NA>,7.367958,10.403514,NaN,1584.39537,7.367958,1070.153344,3.0,1.406343
2,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2003-12-01,2003,USD,2414.0,0.88509,2136.607095,7.666974,1,2002,NaN,0,0,2002.0,1,1.0,1,1,1,210418,2003-12-31,USD,30413.0,18795.0,656.0,6290.0,86.0,116.464,2028.405,26918.240097,16635.265269,580.618995,5567.215671,76.117734,<NA>,<NA>,0.002828,0.034903,0.206819,10.200559,NaN,86,M,CHE,CHE,94.2748,98.6186,95.595354,2235.053275,28158.523305,17401.750749,607.371561,5823.730365,79.62493,<NA>,7.712020,10.245605,NaN,2235.053275,7.71202,881.992117,4.0,2.422479
3,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2004-12-01,2004,USD,3383.0,0.804927,2723.067443,7.909514,1,2002,NaN,0,0,2002.0,1,2.0,1,1,1,210418,2004-12-31,USD,24677.0,20721.0,1087.0,4901.0,448.0,102.537,2028.405,19863.179216,16678.888704,874.955457,3944.946361,360.607217,<NA>,<NA>,0.018155,0.052459,0.198606,9.896623,NaN,86,M,CHE,CHE,95.0317,98.6186,96.362856,2825.847574,20612.899968,17308.420806,907.979992,4093.845392,374.218065,<NA>,7.946564,9.933672,NaN,2825.847574,7.946564,827.464774,3.0,3.290856
4,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,4611.0,Fred Kindle,President/CEO,2005-01-01,2008-02-13,2005-12-01,2005,USD,1299.0,0.804784,1045.414149,6.952168,1,2005,2005.0,1,1,2002.0,1,3.0,1,1,1,210418,2005-12-31,USD,22276.0,22442.0,1798.0,3933.0,883.0,103.5,2035.111,17927.3638,18060.957909,1447.001262,3165.214663,710.62409,<NA>,<NA>,0.039639,0.080118,0.176558,9.794084,NaN,67,M,CHE,CHE,96.1455,98.6186,97.492258,1072.304786,18388.499926,18525.530406,1484.221712,3246.63181,728.903099,<NA>,6.977566,9.819481,NaN,1072.304786,6.977566,788.688118,5.0,1.32551


#### Winsorize and log-transform pay dispersion

The raw pay ratio can be highly skewed, so we apply the same p1/p99 winsorization idea used for CEO pay. The final pay-dispersion outcome is the log of the winsorized ratio, `log_pay_dispersion`.

In [47]:
# Winsorize pay ratio at p1/p99
ratio_valid = df_panel_final["pay_ratio"].dropna()
ratio_p1, ratio_p99 = ratio_valid.quantile([0.01, 0.99])

df_panel_final["w_pay_ratio"] = df_panel_final["pay_ratio"].clip(
    lower=ratio_p1,
    upper=ratio_p99,
)

# Log-transform winsorized pay ratio
df_panel_final["log_pay_dispersion"] = np.log(
    df_panel_final["w_pay_ratio"].where(
        df_panel_final["w_pay_ratio"] > 0
    )
)

/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/2458286077.py:5: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_panel_final["w_pay_ratio"] = df_panel_final["pay_ratio"].clip(
/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T

#### Validate pay-dispersion coverage

This final check reports the denominator coverage and the distribution of the winsorized CEO-to-executive pay ratio.

In [48]:
# Compact validation output for the pay-dispersion outcome
pay_dispersion_summary = pd.DataFrame({
    "metric": [
        "eligible_role_spells",
        "matched_remuneration_records",
        "firm_year_denominators",
        "valid_pay_ratio",
        "valid_log_pay_dispersion",
        "panel_coverage_pct",
        "ratio_p1",
        "ratio_median",
        "ratio_p75",
        "ratio_p99",
    ],
    "value": [
        len(df_emp_pay_ratio_roles),
        len(df_pay_ratio_remun_matched),
        len(exec_pay_agg),
        df_panel_final["pay_ratio"].notna().sum(),
        df_panel_final["log_pay_dispersion"].notna().sum(),
        100 * df_panel_final["log_pay_dispersion"].notna().mean(),
        ratio_p1,
        df_panel_final["w_pay_ratio"].median(),
        df_panel_final["w_pay_ratio"].quantile(0.75),
        ratio_p99,
    ],
})

display(pay_dispersion_summary)

,metric,value
0,eligible_role_spells,51882.000000
1,matched_remuneration_records,5432.000000
2,firm_year_denominators,1976.000000
3,valid_pay_ratio,1707.000000
4,valid_log_pay_dispersion,1707.000000
5,panel_coverage_pct,75.597874
6,ratio_p1,0.575311
7,ratio_median,2.229508
8,ratio_p75,20.472021
9,ratio_p99,102.333800


**Insights:**

- The role filter keeps 51,882 eligible senior non-CEO executive role spells. After matching these role spells to BoardEx remuneration records within the valid role window, 5,432 compensation records remain usable for constructing the denominator.

- These matched records produce 1,976 firm-year executive-pay denominators. This means many CEO firm-years have at least one eligible non-CEO executive with observed compensation.

- The final pay-dispersion outcome is available for 1,707 firm-year observations, equal to about 75.6% of the panel. Missing values mainly reflect firm-years where no eligible non-CEO executive remuneration record could be matched.

- The median CEO-to-executive pay ratio is 2.23. In a typical covered firm-year, the CEO earns a little more than twice the median eligible non-CEO executive.

- The distribution is highly right-skewed: the 75th percentile is 20.47, and the 99th percentile is 102.33. This justifies winsorizing the ratio before taking logs, so extreme pay gaps do not dominate later analysis.

In [49]:
df_panel_final

,companyid,isin,companyname,hocountryname,sector,directorid,directorname,rolename,datestartrole,dateendrole_clean,annualreportdate,annualreportyear,currency,totalcompensation,fx_rate_avg_to_eur,totalcompensation_eur,log_totalcompensation_eur,ceo_turnover,turnover_year,turnover_year_event,num_ceo_turnovers,ceo_turnover_dummy,first_turnover_year,treated_firm,event_time,post_turnover,event_window_3yr,event_window_5yr,gvkey,datadate,curcd,at,sale,ebit,dltt,nicon,emp,cshoi,at_eur,sale_eur,ebit_eur,dltt_eur,nicon_eur,close_eur,market_cap_eur,roa,ebit_margin,leverage,log_assets_eur_nominal,log_market_cap_eur_nominal,age,gender,econiso,cpi_proxy_econiso,annual_avg_cpi,annual_avg_cpi_2015,annual_avg_cpi_index_2015,real_totalcompensation_eur_2015,real_at_eur_2015,real_sale_eur_2015,real_ebit_eur_2015,real_dltt_eur_2015,real_nicon_eur_2015,real_market_cap_eur_2015,log_real_totalcompensation_eur_2015,log_real_assets_eur_2015,log_real_market_cap_eur_2015,w_real_totalcompensation_eur_2015,log_w_real_pay,median_exec_pay_eur,n_exec_in_ratio,pay_ratio,w_pay_ratio,log_pay_dispersion
0,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,14884.0,Jörgen Centerman,President/CEO,2000-12-31,2002-09-05,2001-12-01,2001,USD,1787.0,1.11768,1997.294162,7.599549,0,2000,NaN,0,0,2002.0,1,-1.0,0,1,1,210418,2001-12-31,USD,32344.0,23726.0,466.0,5043.0,-130.0,156.865,1113.133,36150.24196,26518.07571,520.838881,5636.460246,-145.2984,<NA>,<NA>,-0.004019,0.019641,0.155918,10.495439,NaN,77,M,CHE,CHE,93.0786,98.6186,94.382398,2116.17229,38301.889498,28096.420672,551.838997,5971.940043,-153.946501,<NA>,7.657364,10.553255,NaN,2116.17229,7.657364,<NA>,NaN,<NA>,<NA>,<NA>
1,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2002-12-01,2002,USD,1419.0,1.060608,1505.002572,7.31655,1,2002,2002.0,1,1,2002.0,1,0.0,1,1,1,210418,2002-12-31,USD,29533.0,18295.0,598.0,5376.0,97.0,139.051,1113.179,31322.932317,19403.821039,634.243508,5701.827926,102.878964,<NA>,<NA>,0.003284,0.032687,0.182034,10.352106,NaN,86,M,CHE,CHE,93.6769,98.6186,94.989079,1584.39537,32975.298425,20427.423041,667.701502,6002.614172,108.306096,<NA>,7.367958,10.403514,NaN,1584.39537,7.367958,1070.153344,3.0,1.406343,1.406343,0.340993
2,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2003-12-01,2003,USD,2414.0,0.88509,2136.607095,7.666974,1,2002,NaN,0,0,2002.0,1,1.0,1,1,1,210418,2003-12-31,USD,30413.0,18795.0,656.0,6290.0,86.0,116.464,2028.405,26918.240097,16635.265269,580.618995,5567.215671,76.117734,<NA>,<NA>,0.002828,0.034903,0.206819,10.200559,NaN,86,M,CHE,CHE,94.2748,98.6186,95.595354,2235.053275,28158.523305,17401.750749,607.371561,5823.730365,79.62493,<NA>,7.712020,10.245605,NaN,2235.053275,7.71202,881.992117,4.0,2.422479,2.422479,0.884791
3,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,11152.0,Juergen Dormann,Chairman/President/CEO,2002-09-05,2004-12-31,2004-12-01,2004,USD,3383.0,0.804927,2723.067443,7.909514,1,2002,NaN,0,0,2002.0,1,2.0,1,1,1,210418,2004-12-31,USD,24677.0,20721.0,1087.0,4901.0,448.0,102.537,2028.405,19863.179216,16678.888704,874.955457,3944.946361,360.607217,<NA>,<NA>,0.018155,0.052459,0.198606,9.896623,NaN,86,M,CHE,CHE,95.0317,98.6186,96.362856,2825.847574,20612.899968,17308.420806,907.979992,4093.845392,374.218065,<NA>,7.946564,9.933672,NaN,2825.847574,7.946564,827.464774,3.0,3.290856,3.290856,1.191148
4,422.0,CH0012221716,ABB LTD,Switzerland,Engineering & Machinery,4611.0,Fred Kindle,President/CEO,2005-01-01,2008-02-13,2005-12-01,2005,USD,1299.0,0.804784,1045.414149,6.952168,1,2005,2005.0,1,1,2002.0,1,3.0,1,1,1,210418,2005-12-31,USD,22276.0,22442.0,1798.0,3933.0,883.0,103.5,2035.111,17927.3638,18060.957909,1447.001262,3165.214663,710.62409,<NA>,<NA>,0.039639,0.080118,0.176558,9.794084,NaN,67,M,CHE,CHE,96.1455,98.6186,97.492258,1072.304786,18388.499926,18525.530406,1484.221712,3246.63181,728.903099

<a id="save-panel"></a>
### 5.4.5 Save the Clean Analysis Panel

The final step writes the analysis-ready panel to disk. Before saving, we add small convenience aliases used by later modelling sections: `year` for the annual report year, `log_assets` for real firm size, and `ceo_turnover` as a compact turnover indicator.

Both saved files are written from the same `df_panel_final` object. `panel_clean.csv` is the general cleaned panel checkpoint, while `panel_analysis.csv` is the analysis-facing copy reused by the causal-inference, supervised-learning, and unsupervised-learning sections.

In [50]:
# Persist the final analysis panel for downstream sections

# Convenience aliases used in causal inference, supervised learning, and unsupervised learning
df_panel_final = df_panel_final.assign(
    year=df_panel_final["annualreportyear"],
    log_assets=np.log(df_panel_final["real_at_eur_2015"].where(
        df_panel_final["real_at_eur_2015"] > 0
    )),
)

# Keep a compact turnover alias while preserving the original constructed variable
if "ceo_turnover_dummy" in df_panel_final.columns:
    df_panel_final["ceo_turnover"] = (
        df_panel_final["ceo_turnover_dummy"].fillna(0).astype(int)
    )
else:
    df_panel_final["ceo_turnover"] = 0

PANEL_CLEAN = DATA_DIR / "panel_clean.csv"
PANEL_ANALYSIS = DATA_DIR / "panel_analysis.csv"

# Ensure the data folder exists before writing the final checkpoints
DATA_DIR.mkdir(parents=True, exist_ok=True)

for output_path in [PANEL_CLEAN, PANEL_ANALYSIS]:
    df_panel_final.to_csv(output_path, index=False)

save_summary = pd.DataFrame({
    "metric": [
        "rows",
        "columns",
        "unique_firms",
        "year_min",
        "year_max",
        "valid_log_w_real_pay",
        "valid_log_pay_dispersion",
    ],
    "value": [
        len(df_panel_final),
        df_panel_final.shape[1],
        df_panel_final["companyid"].nunique(),
        df_panel_final["year"].min(),
        df_panel_final["year"].max(),
        df_panel_final["log_w_real_pay"].notna().sum(),
        df_panel_final["log_pay_dispersion"].notna().sum(),
    ],
})

display(save_summary)
print(f"Saved: {PANEL_CLEAN}")
print(f"Saved: {PANEL_ANALYSIS}")

/var/folders/wx/b_h8zt7d5ps2y7dhc2qj9pgc0000gn/T/ipykernel_7730/3171032689.py:13: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_panel_final["ceo_turnover"] = (


,metric,value
0,rows,2258
1,columns,76
2,unique_firms,137
3,year_min,1997
4,year_max,2025
5,valid_log_w_real_pay,2258
6,valid_log_pay_dispersion,1707


Saved: data/panel_clean.csv
Saved: data/panel_analysis.csv


**Insights:**

- `df_panel_final` is now the single checkpoint for the remaining empirical sections. Later sections should read from the saved panel rather than reconstructing earlier merge logic.

- The saved panel keeps the raw nominal variables, the real 2015 EUR variables, the winsorized CEO-pay outcome, and the pay-dispersion outcome. This preserves traceability from source units to modelling variables.

- The compact aliases (`year`, `log_assets`, and `ceo_turnover`) are added only for downstream convenience; they do not replace the original construction columns.

---

---

<a id="data-visualizations"></a>
## 6. Data Visualizations

This section visualizes the revised analysis-ready firm-year panel from Section 5.4. Compensation is shown in **thousands of EUR at 2015 prices**, and the primary CEO-pay outcome is the winsorized log variable `log_w_real_pay`. The section also visualizes the revised pay-dispersion outcome, `log_pay_dispersion`, which is the log of the winsorized CEO-to-median-executive pay ratio.

<a id="target-distribution"></a>
### 6.1 Target Distribution

The first figure checks the two modelling outcomes created in Section 5.4: winsorized real CEO pay and CEO-to-executive pay dispersion. The level plot preserves the economic scale of CEO compensation, while the two log plots show the distributions used by later causal and prediction models.

In [52]:
sns.set_theme(style="whitegrid", context="notebook")

viz_df = df_panel_final.copy().replace([np.inf, -np.inf], np.nan)

target_level = "w_real_totalcompensation_eur_2015"
target_log = "log_w_real_pay"
dispersion_log = "log_pay_dispersion"

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))

# Winsorized real CEO compensation in levels
ceo_pay_plot = viz_df[[target_level, target_log]].dropna()
sns.histplot(
    data=ceo_pay_plot,
    x=target_level,
    bins=35,
    kde=True,
    color="#2A6F97",
    ax=axes[0],
)
axes[0].axvline(
    ceo_pay_plot[target_level].median(),
    color="#C44536",
    linestyle="--",
    label="Median",
)
axes[0].set(
    title="Winsorized real CEO compensation",
    xlabel="Total compensation (EUR thousands, 2015 prices)",
    ylabel="Firm-year observations",
)
axes[0].legend()

# Log of winsorized real CEO compensation
sns.histplot(
    data=ceo_pay_plot,
    x=target_log,
    bins=35,
    kde=True,
    color="#4C956C",
    ax=axes[1],
)
axes[1].axvline(
    ceo_pay_plot[target_log].median(),
    color="#C44536",
    linestyle="--",
    label="Median",
)
axes[1].set(
    title="Log winsorized real CEO pay",
    xlabel="log(winsorized real CEO pay)",
    ylabel="Firm-year observations",
)
axes[1].legend()

# Log CEO-to-executive pay dispersion
dispersion_plot = viz_df[[dispersion_log]].dropna()
sns.histplot(
    data=dispersion_plot,
    x=dispersion_log,
    bins=35,
    kde=True,
    color="#8E6C88",
    ax=axes[2],
)
axes[2].axvline(
    dispersion_plot[dispersion_log].median(),
    color="#C44536",
    linestyle="--",
    label="Median",
)
axes[2].set(
    title="Log CEO-to-executive pay dispersion",
    xlabel="log(winsorized CEO-to-executive pay ratio)",
    ylabel="Firm-year observations",
)
axes[2].legend()

plt.tight_layout()
plt.show()

NameError: name 'sns' is not defined

**Insights:**

- Winsorized real CEO pay remains right-skewed: the median is about **EUR 2.03 million**, the mean is about **EUR 2.28 million**, and the p99 cutoff is about **EUR 7.46 million**.

- `log_w_real_pay` compresses the high-pay tail and is the cleaner CEO-pay outcome for later modelling. The winsorized level variable is still retained because it keeps the economic interpretation in EUR thousands.

- Pay dispersion is available for **1,707 firm-years**, or about **75.6%** of the panel. The median CEO-to-executive ratio is about **2.23**, meaning that the typical covered CEO earns a little more than twice the median eligible non-CEO executive.

- The pay-ratio distribution is much more skewed than CEO pay: the p75 ratio is about **20.31** and the p99 cutoff is about **102.33**. This supports winsorizing and logging `pay_ratio` before using it as an outcome.

<a id="annual-pay-trend"></a>
### 6.2 Annual Pay Trend and Coverage

The next figure separates the evolution of CEO pay from sample coverage. Mean and median compensation are both shown because their gap reveals whether a small high-pay tail is influencing the annual average. The coverage panel compares CEO-pay observations with pay-dispersion observations, because the latter requires a matched non-CEO executive denominator.

In [ ]:
annual_pay = (
    viz_df.dropna(subset=["annualreportyear", target_level])
    .groupby("annualreportyear", as_index=False)
    .agg(
        median_real_pay=(target_level, "median"),
        mean_real_pay=(target_level, "mean"),
        pay_observations=(target_level, "size"),
    )
)

annual_dispersion = (
    viz_df.dropna(subset=["annualreportyear", dispersion_log])
    .groupby("annualreportyear", as_index=False)
    .agg(dispersion_observations=(dispersion_log, "size"))
)

annual_pay = annual_pay.merge(
    annual_dispersion,
    on="annualreportyear",
    how="left",
).fillna({"dispersion_observations": 0})

fig, axes = plt.subplots(
    1, 2, figsize=(15, 4.8),
    gridspec_kw={"width_ratios": [2, 1.15]},
)

sns.lineplot(
    data=annual_pay,
    x="annualreportyear",
    y="median_real_pay",
    marker="o",
    label="Median",
    color="#2A6F97",
    ax=axes[0],
)
sns.lineplot(
    data=annual_pay,
    x="annualreportyear",
    y="mean_real_pay",
    marker="o",
    label="Mean",
    color="#C44536",
    ax=axes[0],
)
axes[0].set(
    title="Winsorized real CEO compensation over time",
    xlabel="Annual report year",
    ylabel="EUR thousands (2015 prices)",
)

coverage_long = annual_pay.melt(
    id_vars="annualreportyear",
    value_vars=["pay_observations", "dispersion_observations"],
    var_name="series",
    value_name="observations",
)
coverage_long["series"] = coverage_long["series"].replace({
    "pay_observations": "CEO pay",
    "dispersion_observations": "Pay dispersion",
})

sns.lineplot(
    data=coverage_long,
    x="annualreportyear",
    y="observations",
    hue="series",
    marker="o",
    palette=["#84A98C", "#8E6C88"],
    ax=axes[1],
)
axes[1].set(
    title="Outcome coverage by year",
    xlabel="Annual report year",
    ylabel="Observations",
)
axes[1].legend(title="")

plt.tight_layout()
plt.show()

**Insights:**

- Median winsorized real CEO pay rises from early-sample levels below **EUR 1 million** to a sample peak of about **EUR 2.64 million in 2017**, before settling around **EUR 1.92 million in 2024** and **EUR 2.16 million in 2025**.

- The mean is above the median in most years, which is consistent with the right-skewed pay distribution even after winsorization.

- CEO-pay coverage is complete for the final panel, with **2,258 firm-years** overall. Pay-dispersion coverage is lower because it requires a matched non-CEO executive denominator, yielding **1,707 valid observations** overall.

<a id="turnover-event-visualization"></a>
### 6.3 Pay and Dispersion Around CEO Turnover

For treated firms, event time equals zero in the first observed CEO-turnover year. The figure shows median winsorized real CEO pay and median winsorized CEO-to-executive pay ratio from five years before through five years after that event. Observation bars show how many firm-years support each CEO-pay point.

In [ ]:
event_pay = (
    viz_df.loc[
        viz_df["treated_firm"].eq(1)
        & viz_df["event_time"].between(-5, 5)
    ]
    .groupby("event_time", as_index=False)
    .agg(
        median_real_pay=(target_level, "median"),
        pay_observations=(target_level, "count"),
        median_pay_ratio=("w_pay_ratio", "median"),
        dispersion_observations=("w_pay_ratio", "count"),
    )
)
event_pay["event_time"] = event_pay["event_time"].astype(int)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax_count = axes[0].twinx()
ax_count.bar(
    event_pay["event_time"],
    event_pay["pay_observations"],
    color="#B8C0CC",
    alpha=0.45,
    label="CEO-pay observations",
)
axes[0].plot(
    event_pay["event_time"],
    event_pay["median_real_pay"],
    color="#2A6F97",
    marker="o",
    linewidth=2.5,
    label="Median real CEO pay",
)
axes[0].axvline(0, color="#C44536", linestyle="--", linewidth=1.5)
axes[0].set(
    title="Median real CEO pay around first turnover",
    xlabel="Years relative to first CEO turnover",
    ylabel="EUR thousands (2015 prices)",
)
ax_count.set_ylabel("Firm-year observations")
axes[0].set_xticks(range(-5, 6))

handles_1, labels_1 = axes[0].get_legend_handles_labels()
handles_2, labels_2 = ax_count.get_legend_handles_labels()
axes[0].legend(handles_1 + handles_2, labels_1 + labels_2, loc="upper left")

axes[1].plot(
    event_pay["event_time"],
    event_pay["median_pay_ratio"],
    color="#8E6C88",
    marker="o",
    linewidth=2.5,
    label="Median pay ratio",
)
axes[1].axvline(0, color="#C44536", linestyle="--", linewidth=1.5)
axes[1].set(
    title="Median CEO-to-executive pay ratio",
    xlabel="Years relative to first CEO turnover",
    ylabel="CEO pay / median eligible executive pay",
)
axes[1].set_xticks(range(-5, 6))
axes[1].legend(loc="upper left")

plt.tight_layout()
plt.show()

**Insights:**

- Median real CEO pay declines from about **EUR 1.82 million in event year -1** to about **EUR 1.22 million in the turnover year**, then rebounds to about **EUR 1.76 million in year +1**.

- Median pay dispersion also dips in the turnover year: the median CEO-to-executive ratio is about **2.34** in event year -1, **1.71** in year 0, and **2.07** in year +1.

- These event-time plots are descriptive and use smaller samples than the full panel, especially for pay dispersion. They motivate the formal event-study models later, but do not by themselves estimate a causal effect.

<a id="exploratory-relationships"></a>
### 6.4 Exploratory Relationships

The correlation heatmap summarizes the two compensation outcomes together with continuous candidate controls. The sector boxplots use the eight sectors with the most usable CEO-pay observations and suppress individual outlier markers so that differences in central distributions remain visible.

In [ ]:
correlation_columns = {
    "log_w_real_pay": "Log winsorized real pay",
    "log_pay_dispersion": "Log pay dispersion",
    "log_real_assets_eur_2015": "Log real assets",
    "log_real_market_cap_eur_2015": "Log real market cap",
    "roa": "ROA",
    "ebit_margin": "EBIT margin",
    "leverage": "Leverage",
    "age": "CEO age",
}

available_corr_columns = [
    col for col in correlation_columns
    if col in viz_df.columns
]

corr_data = (
    viz_df[available_corr_columns]
    .apply(pd.to_numeric, errors="coerce")
    .rename(columns=correlation_columns)
)

sector_plot = viz_df.dropna(subset=["sector", target_log]).copy()
top_sectors = sector_plot["sector"].value_counts().head(8).index
sector_plot = sector_plot[sector_plot["sector"].isin(top_sectors)]
sector_order = (
    sector_plot.groupby("sector")[target_log]
    .median()
    .sort_values(ascending=False)
    .index
)

fig, axes = plt.subplots(
    1, 2, figsize=(15, 6),
    gridspec_kw={"width_ratios": [1, 1.2]},
)

sns.heatmap(
    corr_data.corr(),
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    ax=axes[0],
)
axes[0].set_title("Correlations among outcomes and exploratory variables")

sns.boxplot(
    data=sector_plot,
    x=target_log,
    y="sector",
    order=sector_order,
    color="#84A98C",
    showfliers=False,
    ax=axes[1],
)
axes[1].set(
    title="Log winsorized real CEO pay in represented sectors",
    xlabel="log(winsorized real CEO pay)",
    ylabel="Sector",
)

plt.tight_layout()
plt.show()

**Insights:**

- Log winsorized CEO pay has modest positive correlations with **log market capitalization (0.26)**, **log pay dispersion (0.25)**, and **log assets (0.23)**. Profitability, leverage, and CEO age are weaker in the pooled descriptive sample.

- Log pay dispersion is only modestly related to the observable controls shown here. Its strongest displayed relationship is with log CEO pay, which is expected because CEO pay is the ratio numerator.

- Sector distributions differ visibly. Among the most represented sectors, clothing/personal-products and oil-and-gas observations tend to sit above banking, telecommunications, leisure/hotels, and information-technology hardware. These are pooled associations, not within-firm or causal estimates.

- **Modelling implication:** keep firm size, sector, country, and year controls in later models, and treat pay dispersion as a separate outcome rather than as a simple restatement of CEO pay.

<a id="grouped-pay-comparisons"></a>
### 6.5 Mean and Median Pay Across CEO and Firm Groups

Mean and median winsorized real CEO compensation are plotted together because their distance is itself informative: a mean materially above the median indicates that a small high-pay tail is pulling up the average. Age and gender include every observed group. To keep labels readable, the industry and headquarters-country panels visualize the **12 groups with the most usable compensation observations**; the complete results remain available in `grouped_ceo_pay_summary`. Each marker is based on firm-year observations, not unique CEOs. The age bands use the BoardEx `age` field retained in the panel; because date of birth is not retained here, this may not equal age in each historical report year.

In [ ]:
# Construct age bands and harmonize the gender labels used in the grouped summaries.
grouped_viz_df = viz_df.copy()
grouped_viz_df["age_numeric"] = pd.to_numeric(grouped_viz_df["age"], errors="coerce")
grouped_viz_df["age_group"] = pd.cut(
    grouped_viz_df["age_numeric"],
    bins=[0, 39, 49, 59, 69, np.inf],
    labels=["Under 40", "40–49", "50–59", "60–69", "70+"],
)

grouped_viz_df["gender_group"] = (
    grouped_viz_df["gender"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace({"MALE": "M", "FEMALE": "F"})
)

def summarize_grouped_pay(data, group_column, dimension):
    """Return mean, median, and coverage for one grouping variable."""
    summary = (
        data.dropna(subset=[group_column, target_level])
        .groupby(group_column, observed=True)[target_level]
        .agg(mean_pay="mean", median_pay="median", observations="size")
        .reset_index()
        .rename(columns={group_column: "group"})
    )
    summary.insert(0, "dimension", dimension)
    summary["group"] = summary["group"].astype(str)
    return summary

age_pay_summary = summarize_grouped_pay(grouped_viz_df, "age_group", "Age group")
gender_pay_summary = summarize_grouped_pay(grouped_viz_df, "gender_group", "Gender")
industry_pay_summary = summarize_grouped_pay(grouped_viz_df, "sector", "Industry")
country_pay_summary = summarize_grouped_pay(
    grouped_viz_df, "hocountryname", "Headquarters country"
)

grouped_ceo_pay_summary = pd.concat(
    [
        age_pay_summary,
        gender_pay_summary,
        industry_pay_summary,
        country_pay_summary,
    ],
    ignore_index=True,
)

def most_observed(summary, number=12):
    return summary.nlargest(number, "observations").sort_values(
        "median_pay", ascending=True
    )

plot_summaries = {
    "Age group": age_pay_summary.iloc[::-1],
    "Gender": gender_pay_summary.sort_values("median_pay"),
    "Industry (12 most observed)": most_observed(industry_pay_summary),
    "Country (12 most observed)": most_observed(country_pay_summary),
}

def draw_mean_median_dumbbell(ax, summary, title):
    """Plot group medians and means on the same horizontal scale."""
    positions = np.arange(len(summary))
    lower = summary[["mean_pay", "median_pay"]].min(axis=1)
    upper = summary[["mean_pay", "median_pay"]].max(axis=1)

    ax.hlines(
        positions,
        lower,
        upper,
        color="#AAB2BD",
        linewidth=2,
        zorder=1,
    )
    ax.scatter(
        summary["median_pay"],
        positions,
        color="#2A6F97",
        s=55,
        label="Median",
        zorder=2,
    )
    ax.scatter(
        summary["mean_pay"],
        positions,
        color="#C44536",
        marker="D",
        s=48,
        label="Mean",
        zorder=2,
    )
    ax.set_yticks(positions)
    ax.set_yticklabels(summary["group"])
    ax.set_title(title)
    ax.set_xlabel("Winsorized real CEO compensation (EUR thousands, 2015 prices)")
    ax.set_ylabel("")

fig, axes = plt.subplots(2, 2, figsize=(16, 13))
for ax, (title, summary) in zip(axes.flat, plot_summaries.items()):
    draw_mean_median_dumbbell(ax, summary, title)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.955),
    ncol=2,
    frameon=False,
)
fig.suptitle(
    "Mean and median winsorized real CEO compensation across groups",
    fontsize=15,
    y=0.995,
)
plt.tight_layout(rect=[0, 0, 1, 0.91])
plt.show()

display(
    grouped_ceo_pay_summary.assign(
        mean_pay_million_eur=lambda x: x["mean_pay"] / 1_000,
        median_pay_million_eur=lambda x: x["median_pay"] / 1_000,
    )[
        [
            "dimension",
            "group",
            "mean_pay_million_eur",
            "median_pay_million_eur",
            "observations",
        ]
    ].round(2)
)


**Insights:**

- Median pay is lowest for CEOs aged **40-49** at about **EUR 1.38 million** and highest for the **60-69** group at about **EUR 2.14 million**. The 40-49 group is small, with only **28 firm-years**, so it should be read cautiously.

- Female CEO observations have mean/median pay of approximately **EUR 1.68/1.45 million**, compared with **EUR 2.30/2.07 million** for male observations. This comparison is highly unbalanced (**55 versus 2,203 firm-years**) and is not an adjusted gender pay gap.

- Among the 12 most represented sectors, median pay ranges from about **EUR 1.52 million** in information-technology hardware to about **EUR 3.16 million** in clothing/personal products. Across the displayed countries, it ranges from about **EUR 0.74 million** in Norway to about **EUR 2.83 million** in Italy.

- The gaps between paired mean and median markers reveal within-group skewness. All comparisons are pooled and descriptive: they combine firm size, sector, country, year, and repeated CEO observations. Formal age, gender, industry, or country effects therefore require multivariate modelling and appropriate firm/year controls.

---

<a id="causal-inference"></a>
## 7. Causal Inference

This section estimates whether CEO turnover is followed by changes in compensation outcomes. The causal design uses the analysis-ready panel from Section 8.5, treats CEO turnover as a governance shock, and controls for firm fundamentals, country, sector, and year effects. We estimate the effect for two outcomes:

1. CEO pay level: `log_w_real_pay`.
2. Pay dispersion: `log_pay_dispersion`.

Both outcomes use the same logic: define a complete regression sample, state the backdoor adjustment set, estimate a pre/post difference-in-differences regression, and then inspect event-time coefficients around the turnover year.


<a id="causal-inference-results"></a>
### 7.1 Causal Inference: CEO Pay

The first causal outcome is CEO pay level, measured as `log_w_real_pay`, the log of winsorized real CEO total compensation. The treatment is `ceo_turnover`, defined from the firm-level CEO-spell table.

The identifying idea is that CEO turnover is not randomly assigned: firm performance, firm size, leverage, country, and sector can affect both the probability of turnover and CEO pay. We therefore model these variables as backdoor controls and estimate the post-turnover change in pay after adjustment.

This subsection reports two complementary estimates:

- **DiD / adjusted pre-post regression:** estimates the average post-turnover difference in log CEO pay after controlling for firm fundamentals, country, sector, and year effects.
- **Event study:** estimates event-time coefficients from three years before to three years after turnover, using the year before turnover as the omitted baseline.


#### 7.1.1 Sample, DAG, and Adjusted DiD Estimate

This block loads the saved analysis panel, keeps rows with the CEO-pay outcome and required controls, states the causal graph, identifies the backdoor adjustment set, and estimates the adjusted post-turnover effect.


In [ ]:
# --- 7.1.1 Causal inference: CEO pay sample, DAG, and DiD ---
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning, module="dowhy")
import networkx as nx
if not hasattr(nx.algorithms, "d_separated"):
    from networkx.algorithms.d_separation import is_d_separator
    nx.algorithms.d_separated = is_d_separator
from dowhy import CausalModel
import statsmodels.formula.api as smf

# Load the analysis panel saved in Section 8.5
df_a = pd.read_csv(DATA_DIR / "panel_analysis.csv")
df_a = df_a.assign(
    event_time=pd.to_numeric(df_a["event_time"], errors="coerce").astype("Int64")
)

# Regression sample: rows with pay and at least one Compustat covariate
df_reg = df_a[
    df_a["log_w_real_pay"].notna() &
    df_a["log_assets"].notna() &
    df_a["roa"].notna()
].copy()

print(f"Regression sample : {len(df_reg):,} rows, {df_reg['isin'].nunique()} firms")
print(f"Treated (turnover): {df_reg['treated_firm'].sum()} rows  "
      f"({df_reg.groupby('isin')['treated_firm'].first().sum():.0f} firms)")
print(f"Year range        : {df_reg['year'].min()} \u2013 {df_reg['year'].max()}")

# --- Causal DAG (DoWhy) ---
causal_graph = """
digraph {
    ceo_turnover  -> log_w_real_pay;
    roa           -> ceo_turnover;
    roa           -> log_w_real_pay;
    log_assets    -> ceo_turnover;
    log_assets    -> log_w_real_pay;
    leverage      -> ceo_turnover;
    leverage      -> log_w_real_pay;
    hocountryname -> ceo_turnover;
    hocountryname -> log_w_real_pay;
    sector        -> ceo_turnover;
    sector        -> log_w_real_pay;
}
"""

model = CausalModel(
    data=df_reg,
    treatment="ceo_turnover",
    outcome="log_w_real_pay",
    graph=causal_graph,
)
model.view_model()

identified = model.identify_effect(proceed_when_unidentifiable=True)
adj_set = identified.get_backdoor_variables()
print(f"Backdoor adjustment set: {adj_set}")

# --- Adjusted DiD regression: post-turnover effect on CEO pay ---
formula = (
    "log_w_real_pay ~ treated_firm + post_turnover "
    "+ roa + log_assets + leverage "
    "+ C(year) + C(hocountryname) + C(sector)"
)
res_did = smf.ols(formula, data=df_reg).fit(cov_type="HC3")

print("\u2500\u2500 DiD: effect of post-turnover period on log CEO pay \u2500\u2500")
print(f"  post_turnover coef : {res_did.params['post_turnover']:.4f}")
print(f"  95% CI             : [{res_did.conf_int().loc['post_turnover', 0]:.4f}, "
      f"{res_did.conf_int().loc['post_turnover', 1]:.4f}]")
print(f"  p-value            : {res_did.pvalues['post_turnover']:.4f}")
print(f"  N                  : {int(res_did.nobs)},  R\u00b2 = {res_did.rsquared:.3f}")

**Insights:**

- The adjusted CEO-pay regression uses 1,908 firm-year observations across 134 firms from 2000 to 2024.

- The backdoor adjustment set identified by the DAG is economically intuitive: profitability (`roa`), firm size (`log_assets`), leverage, headquarters country, and sector all plausibly affect both CEO turnover and CEO pay.

- The post-turnover coefficient is negative (`-0.0595` log points), suggesting lower CEO pay after turnover on average, but the confidence interval includes zero and the p-value is 0.1734.

- This means the adjusted DiD estimate does not provide statistically strong evidence that CEO turnover changes average CEO pay levels in the post-turnover period.

- The model explains a meaningful share of variation in CEO pay (`R² = 0.373`), so firm fundamentals, country, sector, and year effects are important controls even though the turnover effect itself is not statistically significant.


#### 7.1.2 CEO-Pay Event Study

The event study checks whether pay was already moving before turnover and whether the effect is concentrated in the turnover year or persists afterward. The omitted baseline is event time `-1`, the year before turnover.


In [ ]:
# --- 7.1.2 Event-study regression: CEO pay around turnover ---
# Regress log pay on event-time dummies (\u03c4 = -3\u2026+3) for treated firms,
# omitting \u03c4 = -1 as the baseline. Control firms contribute to year FE.
# treated_firm absorbs the pre-period level difference between treated/control.
# Columns named et_m3\u2026et_p3 (\u201cm\u201d=minus, \u201cp\u201d=plus) to avoid patsy parsing
# hyphens in et_-3 as subtraction.

es_sample = df_reg[
    (df_reg["treated_firm"].eq(1) & df_reg["event_time"].between(-3, 3))
    | df_reg["treated_firm"].eq(0)
].copy()

def tau_col(t):
    return f"et_m{abs(t)}" if t < 0 else f"et_p{t}"

for tau in range(-3, 4):
    if tau == -1:
        continue
    es_sample[tau_col(tau)] = (
        (es_sample["treated_firm"].eq(1)) &
        (es_sample["event_time"] == tau)
    ).fillna(False).astype(int)

et_terms = " + ".join(tau_col(t) for t in range(-3, 4) if t != -1)
formula_es = (
    f"log_w_real_pay ~ {et_terms} + treated_firm "
    "+ roa + log_assets + leverage "
    "+ C(year) + C(hocountryname) + C(sector)"
)
res_es = smf.ols(formula_es, data=es_sample).fit(cov_type="HC3")

# Collect \u03c4 coefficients and CIs (\u03c4 = -1 is the omitted baseline \u2192 0)
taus_full = list(range(-3, 4))
coef_dict = {
    t: (0.0, 0.0, 0.0) if t == -1
    else (
        res_es.params.get(tau_col(t), np.nan),
        res_es.conf_int().loc[tau_col(t), 0] if tau_col(t) in res_es.params else np.nan,
        res_es.conf_int().loc[tau_col(t), 1] if tau_col(t) in res_es.params else np.nan,
    )
    for t in taus_full
}
coefs_full = [coef_dict[t][0] for t in taus_full]
ci_lo_full = [coef_dict[t][1] for t in taus_full]
ci_hi_full = [coef_dict[t][2] for t in taus_full]

fig, ax = plt.subplots(figsize=(9, 5))
ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(-0.5, color="red", linestyle="--", linewidth=1, label="Turnover (\u03c4=0)")
ax.fill_between(taus_full, ci_lo_full, ci_hi_full, alpha=0.2, color="steelblue")
ax.plot(taus_full, coefs_full, marker="o", color="steelblue", linewidth=1.8, markersize=6)
ax.set_xticks(taus_full)
ax.set_xlabel("Event time (years relative to CEO turnover)")
ax.set_ylabel("\u0394 log CEO real pay vs \u03c4 = \u22121")
ax.set_title("Event-Study: CEO Pay Around Turnover (baseline = year before turnover, HC3 CI)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Event-study sample: {len(es_sample):,} rows  "
      f"(treated in window: {es_sample[es_sample['treated_firm']==1].shape[0]},  "
      f"control rows: {(es_sample['treated_firm']==0).sum()})")

print("\u2500\u2500 Event-time coefficients (log pay vs baseline \u03c4=\u22121):")
for t, c, lo, hi in zip(taus_full, coefs_full, ci_lo_full, ci_hi_full):
    marker = "\u2190 baseline" if t == -1 else ""
    print(f"  \u03c4 = {t:+d}:  \u03b2 = {c:+.4f}  [{lo:+.4f}, {hi:+.4f}]  {marker}")

**Insights:**

- The event-study sample contains 400 firm-year observations: 345 treated observations inside the event window and 55 control observations.

- The pre-turnover coefficients at `τ = -3` and `τ = -2` are close to zero and their confidence intervals include zero, which supports the parallel-trends interpretation before the turnover year.

- CEO pay falls sharply in the turnover year: the coefficient at `τ = 0` is `-0.5318` log points and its confidence interval stays clearly below zero.

- The post-turnover coefficients at `τ = +1`, `τ = +2`, and `τ = +3` remain negative, but their confidence intervals include or nearly include zero, so the strongest evidence is concentrated in the turnover year itself.

- Taken together, the event study suggests a short-run CEO-pay reset around leadership change rather than a clearly persistent long-run reduction in CEO pay.


<a id="causal-inference-pay-dispersion-results"></a>
### 7.2 Causal Inference: Pay Dispersion

The second causal outcome is `log_pay_dispersion`, the log of the winsorized ratio between CEO total compensation and median non-CEO executive compensation within the same firm-year.

The same causal structure is used because the same firm conditions that can trigger CEO turnover may also shape internal pay hierarchies. The sample is smaller than the CEO-pay sample because pay-dispersion construction requires usable compensation records for other senior executives in the same firm-year.

This subsection repeats the adjusted DiD and event-study design with pay dispersion as the outcome.


#### 7.2.1 Pay-Dispersion Sample, DAG, and Adjusted DiD Estimate

This block keeps rows with usable pay-dispersion outcomes and controls, identifies the same backdoor adjustment set, and estimates whether the post-turnover period is associated with a change in internal pay dispersion.


In [ ]:
# --- 7.2.1 Adjusted DiD regression: pay dispersion ---

df_reg_disp = df_a[
    df_a["log_pay_dispersion"].notna() &
    df_a["log_assets"].notna() &
    df_a["roa"].notna()
].copy()

print(f"Dispersion regression sample : {len(df_reg_disp):,} rows, {df_reg_disp['isin'].nunique()} firms")
print(f"Treated (turnover)           : {df_reg_disp['treated_firm'].sum()} rows  "
      f"({df_reg_disp.groupby('isin')['treated_firm'].first().sum():.0f} firms)")

causal_graph_disp = """
digraph {
    ceo_turnover  -> log_pay_dispersion;
    roa           -> ceo_turnover;
    roa           -> log_pay_dispersion;
    log_assets    -> ceo_turnover;
    log_assets    -> log_pay_dispersion;
    leverage      -> ceo_turnover;
    leverage      -> log_pay_dispersion;
    hocountryname -> ceo_turnover;
    hocountryname -> log_pay_dispersion;
    sector        -> ceo_turnover;
    sector        -> log_pay_dispersion;
}
"""

model_disp = CausalModel(
    data=df_reg_disp,
    treatment="ceo_turnover",
    outcome="log_pay_dispersion",
    graph=causal_graph_disp,
)
model_disp.view_model()

identified_disp = model_disp.identify_effect(proceed_when_unidentifiable=True)
print(f"Backdoor adjustment set: {identified_disp.get_backdoor_variables()}")

formula_disp = (
    "log_pay_dispersion ~ treated_firm + post_turnover "
    "+ roa + log_assets + leverage "
    "+ C(year) + C(hocountryname) + C(sector)"
)
res_did_disp = smf.ols(formula_disp, data=df_reg_disp).fit(cov_type="HC3")

print("\u2500\u2500 DiD: effect of post-turnover period on log pay dispersion \u2500\u2500")
print(f"  post_turnover coef : {res_did_disp.params['post_turnover']:.4f}")
print(f"  95% CI             : [{res_did_disp.conf_int().loc['post_turnover', 0]:.4f}, "
      f"{res_did_disp.conf_int().loc['post_turnover', 1]:.4f}]")
print(f"  p-value            : {res_did_disp.pvalues['post_turnover']:.4f}")
print(f"  N                  : {int(res_did_disp.nobs)},  R\u00b2 = {res_did_disp.rsquared:.3f}")

**Insights:**

- The pay-dispersion regression uses 1,310 firm-year observations across 124 firms, so the sample is smaller than the CEO-pay regression because it requires usable non-CEO executive pay in the same firm-year.

- The DAG uses the same backdoor controls as the CEO-pay model: profitability (`roa`), firm size (`log_assets`), leverage, headquarters country, and sector.

- The post-turnover coefficient is `-0.3386` log points, and the 95% confidence interval `[-0.5124, -0.1647]` is entirely below zero.

- This provides statistically strong evidence that pay dispersion is lower after CEO turnover, conditional on the controls in the model.

- In substantive terms, the result suggests that CEO turnover compresses the internal pay hierarchy: the CEO-to-median-executive pay ratio becomes smaller after leadership change.


#### 7.2.2 Pay-Dispersion Event Study

The event study repeats the turnover-window analysis for pay dispersion. This shows whether internal pay inequality changes at the turnover year itself or evolves before and after the CEO transition.


In [ ]:
# --- 7.2.2 Event-study: pay dispersion around CEO turnover ---

es_sample_disp = df_reg_disp[
    (df_reg_disp["treated_firm"].eq(1) & df_reg_disp["event_time"].between(-3, 3))
    | df_reg_disp["treated_firm"].eq(0)
].copy()

for tau in range(-3, 4):
    if tau == -1:
        continue
    es_sample_disp[tau_col(tau)] = (
        (es_sample_disp["treated_firm"].eq(1)) &
        (es_sample_disp["event_time"] == tau)
    ).fillna(False).astype(int)

et_terms_disp = " + ".join(tau_col(t) for t in range(-3, 4) if t != -1)
formula_es_disp = (
    f"log_pay_dispersion ~ {et_terms_disp} + treated_firm "
    "+ roa + log_assets + leverage "
    "+ C(year) + C(hocountryname) + C(sector)"
)
res_es_disp = smf.ols(formula_es_disp, data=es_sample_disp).fit(cov_type="HC3")

coef_dict_disp = {
    t: (0.0, 0.0, 0.0) if t == -1
    else (
        res_es_disp.params.get(tau_col(t), np.nan),
        res_es_disp.conf_int().loc[tau_col(t), 0] if tau_col(t) in res_es_disp.params else np.nan,
        res_es_disp.conf_int().loc[tau_col(t), 1] if tau_col(t) in res_es_disp.params else np.nan,
    )
    for t in range(-3, 4)
}
coefs_disp = [coef_dict_disp[t][0] for t in range(-3, 4)]
ci_lo_disp = [coef_dict_disp[t][1] for t in range(-3, 4)]
ci_hi_disp = [coef_dict_disp[t][2] for t in range(-3, 4)]

fig, ax = plt.subplots(figsize=(9, 5))
ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(-0.5, color="red", linestyle="--", linewidth=1, label="Turnover (\u03c4=0)")
ax.fill_between(range(-3, 4), ci_lo_disp, ci_hi_disp, alpha=0.2, color="darkorange")
ax.plot(range(-3, 4), coefs_disp, marker="o", color="darkorange", linewidth=1.8, markersize=6)
ax.set_xticks(range(-3, 4))
ax.set_xlabel("Event time (years relative to CEO turnover)")
ax.set_ylabel("\u0394 log pay dispersion vs \u03c4 = \u22121")
ax.set_title("Event-Study: Pay Dispersion Around CEO Turnover (baseline = year before turnover, HC3 CI)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Event-study sample: {len(es_sample_disp):,} rows  "
      f"(treated in window: {es_sample_disp[es_sample_disp['treated_firm']==1].shape[0]},  "
      f"control rows: {(es_sample_disp['treated_firm']==0).sum()})")

print("\u2500\u2500 Event-time coefficients (log pay dispersion vs baseline \u03c4=\u22121):")
for t in range(-3, 4):
    c, lo, hi = coef_dict_disp[t]
    marker = "\u2190 baseline" if t == -1 else ""
    print(f"  \u03c4 = {t:+d}:  \u03b2 = {c:+.4f}  [{lo:+.4f}, {hi:+.4f}]  {marker}")

**Insights:**

- The pay-dispersion event-study sample contains 312 firm-year observations: 279 treated observations inside the event window and 33 control observations.

- The pre-turnover coefficients at `τ = -3` and `τ = -2` are close to zero relative to their wide confidence intervals, so there is no clear evidence of a strong pre-trend before turnover.

- The post-turnover coefficients from `τ = 0` to `τ = +3` are all negative, which is consistent with the adjusted DiD result that pay dispersion falls after CEO turnover.

- However, every event-time confidence interval includes zero, so the event-study estimates are not individually statistically significant.

- The cautious conclusion is that the event study points in the same direction as the DiD estimate, but the smaller event-window sample makes the timing of the pay-dispersion effect less precise.


---

<a id="supervised-learning"></a>
## 8. Supervised Learning

<a id="supervised-learning-target"></a>
### 8.1 Prediction Target

This section treats the cleaned panel as a forecasting dataset. The goal is not to estimate a causal effect; it is to ask how much of CEO pay and pay dispersion can be explained by observable firm, executive, governance, country, sector, and year characteristics.

The first task is to define the CEO-pay outcome carefully. For CEO pay, the prediction target is `log_w_real_pay`, the logarithm of winsorized real total CEO compensation in 2015 EUR. The log transformation reduces the influence of very large compensation packages, while winsorization limits the pull of extreme observations.

To avoid stale in-memory objects, this section reloads the saved clean panel from Section 5 when the checkpoint is available. The cell then verifies that the target columns exist, counts usable observations, checks the observed year range, and visualizes the target distribution.

In [ ]:
# -- 8.1 Prediction target -------------------------------------------------
# Reload the saved clean panel when available so Section 8 uses the same
# checkpoint produced at the end of Section 5.

PANEL_ANALYSIS_PATH = DATA_DIR / "panel_analysis.csv"

if PANEL_ANALYSIS_PATH.exists():
    df_panel_final = pd.read_csv(PANEL_ANALYSIS_PATH)
    print(f"Loaded analysis panel from: {PANEL_ANALYSIS_PATH}")
elif "df_panel_final" not in globals():
    raise NameError(
        "df_panel_final is not available and the saved panel checkpoint was not found. "
        "Run Section 5.4.5 first."
    )
else:
    print("Using in-memory df_panel_final; saved panel checkpoint was not found.")

SUPERVISED_TARGET = "log_w_real_pay"
TARGET_LEVEL_COL = "w_real_totalcompensation_eur_2015"
YEAR_COL = "annualreportyear"

required_target_cols = [SUPERVISED_TARGET, TARGET_LEVEL_COL, YEAR_COL]
missing_target_cols = [
    col for col in required_target_cols
    if col not in df_panel_final.columns
]

if missing_target_cols:
    raise KeyError(
        "Missing supervised-learning target/setup column(s): "
        + ", ".join(missing_target_cols)
    )

# Make the year and target numeric for diagnostics and downstream splitting.
df_panel_final[YEAR_COL] = pd.to_numeric(df_panel_final[YEAR_COL], errors="coerce")
df_panel_final[SUPERVISED_TARGET] = pd.to_numeric(
    df_panel_final[SUPERVISED_TARGET],
    errors="coerce",
)
df_panel_final[TARGET_LEVEL_COL] = pd.to_numeric(
    df_panel_final[TARGET_LEVEL_COL],
    errors="coerce",
)

target_df = df_panel_final.loc[
    df_panel_final[SUPERVISED_TARGET].notna(),
    [YEAR_COL, SUPERVISED_TARGET, TARGET_LEVEL_COL],
].copy()

if target_df.empty:
    raise ValueError(f"No non-missing observations found for {SUPERVISED_TARGET}.")

target_summary = target_df[SUPERVISED_TARGET].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).to_frame("log winsorized real pay")

target_coverage = pd.DataFrame({
    "metric": [
        "panel_rows",
        "usable_target_rows",
        "target_missing_rows",
        "first_year",
        "last_year",
    ],
    "value": [
        len(df_panel_final),
        len(target_df),
        df_panel_final[SUPERVISED_TARGET].isna().sum(),
        int(target_df[YEAR_COL].min()),
        int(target_df[YEAR_COL].max()),
    ],
})

print(f"Prediction target: {SUPERVISED_TARGET}")
print("Target definition: log of winsorized real total CEO compensation in 2015 EUR.")
display(target_coverage)
display(target_summary.round(3))

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.histplot(target_df[SUPERVISED_TARGET], bins=30, kde=True, ax=ax)
ax.set_title("Supervised-Learning Target: Log Winsorized Real CEO Pay")
ax.set_xlabel("Log winsorized real total compensation, 2015 EUR")
ax.set_ylabel("Firm-year observations")
plt.tight_layout()
plt.show()

<a id="supervised-learning-sample"></a>
### 8.2 Model Sample and Features

After defining the target, we construct the supervised-learning sample. The target/outcome is `log_w_real_pay`; it is not used as an input feature. The predictors are limited to observable CEO, firm, governance, country, sector, and year characteristics.

The feature-building workflow is split into smaller blocks. First, we define diagnostic identifiers and numeric candidate variables. Second, we inspect numeric distributions with boxplots. Third, we inspect correlations among candidate numeric predictors. Fourth, we define the final leakage-safe model features. Finally, we build the supervised-learning sample and check missingness.

#### Define Diagnostic Identifiers and Numeric Candidates

Identifier columns such as `companyid`, `isin`, and `directorid` are useful for checking coverage, firm counts, and examples, but they are not model predictors. Numeric candidate variables include CEO age, profitability, leverage, firm size, employee count, and turnover indicators. Because employee count is strongly scale-dependent, we create `log_employees` as the model-ready version while keeping raw `emp` for diagnostics.

In [ ]:
# -- 8.2.1 Diagnostic identifiers and numeric candidates ---------------
# Identifiers are retained for diagnostics only; they are excluded from X.

id_columns = [
    "companyid",
    "companyname",
    "isin",
    "directorid",
    "directorname",
]

# Keep raw employee count for diagnostics and create a log version for modeling.
df_panel_final["emp"] = pd.to_numeric(df_panel_final["emp"], errors="coerce")
df_panel_final["log_employees"] = np.log1p(df_panel_final["emp"])

numeric_candidate_features = [
    "age",
    "roa",
    "ebit_margin",
    "leverage",
    "log_real_assets_eur_2015",
    "log_real_market_cap_eur_2015",
    "emp",
    "log_employees",
    "ceo_turnover",
    "post_turnover",
]

available_numeric_candidate_features = [
    col for col in numeric_candidate_features
    if col in df_panel_final.columns
]

numeric_candidate_coverage = pd.DataFrame({
    "non_missing": df_panel_final[available_numeric_candidate_features].notna().sum(),
    "missing": df_panel_final[available_numeric_candidate_features].isna().sum(),
    "missing_pct": (
        df_panel_final[available_numeric_candidate_features].isna().mean() * 100
    ).round(2),
})

print(f"Available identifier columns: {sum(col in df_panel_final.columns for col in id_columns)} of {len(id_columns)}")
display(numeric_candidate_coverage.sort_values("missing_pct", ascending=False))

**Insights:**

- The diagnostic identifiers are used only for checking firms, directors, and examples; they are not included as model predictors.

- Most numeric candidate features have strong coverage. `ceo_turnover` and `post_turnover` are complete, while `age`, `leverage`, and `log_real_assets_eur_2015` have only minimal missingness.

- The main coverage limitation is `log_real_market_cap_eur_2015`, which is missing for about one quarter of the panel. This supports not relying on market capitalization as the main firm-size feature.

- `emp` has moderate missingness, but enough coverage to be useful. The model uses `log_employees` as the transformed employee-size feature while keeping raw `emp` for diagnostics.

- Missing values are not dropped at this stage; they are documented here and handled later in the preprocessing pipeline.

#### Inspect Numeric Feature Distributions

The boxplots below compare the numeric candidate variables before final feature selection. Because the variables have very different units, the first plot uses standardized values. The second plot focuses on raw `emp` versus `log_employees`, making the reason for using the log version visible.

In [ ]:
import math
# -- 8.2.2 Boxplots for numeric candidate features ------------------------

boxplot_features = [
    "age",
    "roa",
    "ebit_margin",
    "leverage",
    "log_real_assets_eur_2015",
    "log_real_market_cap_eur_2015",
    "emp",
    "log_employees",
]

# Keep only features that exist in the current panel.
boxplot_features = df_panel_final.columns.intersection(boxplot_features).tolist()

ncols = 4
nrows = math.ceil(len(boxplot_features) / ncols)

plt.figure(figsize=(14, 2.2 * nrows), facecolor="white")
sns.set(style="whitegrid")

for i, feature in enumerate(boxplot_features, start=1):
    ax = plt.subplot(nrows, ncols, i)
    sns.boxplot(
        x=pd.to_numeric(df_panel_final[feature], errors="coerce"),
        ax=ax,
        color="lightsteelblue",
    )
    ax.set_title(feature, fontsize=11)
    ax.set_xlabel("")

plt.tight_layout()
plt.show()

**Insights:**

- The numeric features have very different scales, so separate boxplots are easier to interpret than one combined plot.

- Raw employee count (`emp`) is strongly right-skewed: most firms are concentrated below roughly 200 thousand employees, while several very large firms appear as high-end outliers. This supports using `log_employees` for modelling.

- The `log_employees` boxplot shows a much more compact distribution than raw `emp`, meaning the log transformation reduces the dominance of very large employers while preserving firm-size information.

- Profitability variables (`roa` and `ebit_margin`) contain both positive and negative outliers, which is expected in firm financial data and supports robust preprocessing rather than dropping observations.

- `log_real_assets_eur_2015` and `log_real_market_cap_eur_2015` are less extreme than their raw scale versions would be, but still show some tail observations. Keeping them in log form is therefore appropriate.

- `age` and `leverage` are comparatively well-behaved, with fewer extreme observations than the firm-size and profitability variables.

#### Inspect Correlations Among Candidate Numeric Features

Before finalising the numeric feature set, we inspect pairwise correlations among plausible numeric predictors. This helps identify variables that carry very similar information. Multicollinearity is less damaging for pure prediction than for coefficient interpretation, but redundant variables can still add noise and make the model harder to explain.

In [ ]:
# -- 8.2.3 Correlation check for numeric candidates --------------------

preselection_numeric_features = [
    "age",
    "roa",
    "ebit_margin",
    "leverage",
    "log_real_assets_eur_2015",
    "log_real_market_cap_eur_2015",
    "log_employees",
    "ceo_turnover",
    "post_turnover",
]

available_preselection_numeric_features = [
    col for col in preselection_numeric_features
    if col in df_panel_final.columns
]

preselection_corr = (
    df_panel_final.loc[
        df_panel_final[SUPERVISED_TARGET].notna(),
        available_preselection_numeric_features,
    ]
    .apply(pd.to_numeric, errors="coerce")
    .corr()
)

display(preselection_corr.round(2))

fig, ax = plt.subplots(figsize=(8.5, 6.5))
sns.heatmap(
    preselection_corr,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    ax=ax,
)
ax.set_title("Pre-selection Correlations Among Numeric Candidate Features")
plt.tight_layout()
plt.show()

**Insights:**

- The correlation matrix shows no severe multicollinearity among the final retained predictors, so the feature set is suitable for supervised learning.

- `roa` and `ebit_margin` are strongly correlated (**0.68**), which supports keeping only `roa` as the main profitability measure in the final feature set.

- The firm-size variables are moderately correlated: `log_real_assets_eur_2015` correlates with `log_real_market_cap_eur_2015` (**0.53**) and `log_employees` (**0.53**). This means they capture related but not identical size information.

- `log_employees` is also moderately correlated with market capitalization (**0.46**), but not so strongly that it becomes a duplicate feature. It can add a useful operational-size dimension.

- `ceo_turnover` has very low correlations with the financial and size variables.

- `post_turnover` is modestly correlated with firm size and employee count.

#### Select the Final Leakage-Safe Feature Set

The final feature set keeps CEO age, profitability, leverage, firm size, employee size, CEO-turnover status, country, sector, and report year. Direct compensation variables and the prediction target are excluded because they would leak the answer into the input features.

In [ ]:
# -- 8.2.3 Final leakage-safe feature set ------------------------------

numeric_features = [
    "age",
    "roa",
    "leverage",
    "log_real_assets_eur_2015",
    "log_employees",
    "ceo_turnover",
    "post_turnover",
]

categorical_features = [
    "gender",
    "hocountryname",
    "sector",
    YEAR_COL,
]

pay_leakage_columns = {
    SUPERVISED_TARGET,
    TARGET_LEVEL_COL,
    "totalcompensation",
    "totalcompensation_eur",
    "log_totalcompensation_eur",
    "real_totalcompensation_eur_2015",
    "log_real_totalcompensation_eur_2015",
    "salary",
    "salary_eur",
    "bonus",
    "bonus_eur",
    "totaldirectcomp",
    "totaldirectcomp_eur",
    "perftotal",
    "real_salary_eur_2015",
    "real_bonus_eur_2015",
    "real_totaldirectcomp_eur_2015",
}

candidate_columns = id_columns + numeric_features + categorical_features + [SUPERVISED_TARGET]
available_columns = [
    col for col in dict.fromkeys(candidate_columns)
    if col in df_panel_final.columns
]
missing_candidate_columns = [
    col for col in dict.fromkeys(candidate_columns)
    if col not in df_panel_final.columns
]

# Keep only feature names that actually exist in the current panel.
id_columns = [col for col in id_columns if col in df_panel_final.columns]
numeric_features = [col for col in numeric_features if col in df_panel_final.columns]
categorical_features = [col for col in categorical_features if col in df_panel_final.columns]
model_features = numeric_features + categorical_features

for forbidden_group in [set(id_columns), pay_leakage_columns]:
    overlap = sorted(set(model_features).intersection(forbidden_group))
    if overlap:
        raise ValueError(f"Model feature list contains forbidden column(s): {overlap}")

feature_role_table = pd.DataFrame({
    "role": ["identifier", "numeric feature", "categorical feature"],
    "columns": [
        ", ".join(id_columns),
        ", ".join(numeric_features),
        ", ".join(categorical_features),
    ],
})

print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")
if missing_candidate_columns:
    print("Columns requested but unavailable:", missing_candidate_columns)

display(feature_role_table)

**Insights:**

- The model feature set is intentionally narrow and explainable instead of using every available column.
- Direct compensation variables are excluded to avoid target leakage.
- `companyid`, `isin`, `directorid`, and names are kept only for checks and examples, not for prediction.

#### Build the Supervised-Learning Sample

This block creates `ml_sample`, converts numeric columns to numeric dtype, and reports feature missingness. The output is the cleaned sample used by the train/test split in Section 8.3.

In [ ]:
# -- 8.2.4 Build supervised-learning sample ---------------------------

ml_sample = df_panel_final.loc[
    df_panel_final[SUPERVISED_TARGET].notna(),
    available_columns,
].copy()

# Convert numeric model columns to numeric dtype.
for col in numeric_features + [SUPERVISED_TARGET]:
    ml_sample[col] = pd.to_numeric(ml_sample[col], errors="coerce")

# Year is later treated as categorical, but it should still be a clean year value.
if YEAR_COL in ml_sample.columns:
    ml_sample[YEAR_COL] = pd.to_numeric(
        ml_sample[YEAR_COL],
        errors="coerce",
    ).astype("Int64")

feature_missingness = ml_sample[model_features].isna().sum().to_frame("missing")
feature_missingness["missing_pct"] = (
    feature_missingness["missing"] / len(ml_sample) * 100
).round(2)
feature_missingness = feature_missingness.sort_values("missing_pct", ascending=False)

print(f"ML sample rows with non-missing target: {len(ml_sample):,}")
print(f"Model features: {len(model_features)}")
display(feature_missingness)
display(ml_sample[id_columns + model_features + [SUPERVISED_TARGET]].head())

**Insights:**

- The supervised-learning sample keeps all **2,258 observations** with a valid CEO-pay target.

- The final feature matrix contains **11 raw predictors**: 7 numeric features and 4 categorical features.

- Missingness is modest and concentrated in `roa` (**5.71%**) and `log_employees` (**1.95%**). These rows are not dropped; missing values are handled later by the preprocessing pipeline.

- The preview confirms the distinction between diagnostic identifiers and model features before the train/test split.

<a id="supervised-learning-split"></a>
### 8.3 Train/Test Split

#### Create Time-Based Train/Test Split

The supervised-learning models are evaluated with a time-based split. Observations before 2020 are used for training, while observations from 2020 onward are held out for testing. This is closer to a forecasting setup than a random split because the model learns from earlier firm-years and is evaluated on later firm-years.

Preprocessing is not fitted before this split. Imputation, scaling, and one-hot encoding are learned only from the training period, then applied to the test period. The split cell also checks that identifiers and target columns are not accidentally included in the feature matrix.

In [ ]:
# -- 8.3 Time-based train/test split --------------------------------------

TEST_START_YEAR = 2020

ml_model_df = ml_sample.copy()

# Ensure the report year is numeric before applying the time-based cutoff.
ml_model_df[YEAR_COL] = pd.to_numeric(ml_model_df[YEAR_COL], errors="coerce")

# Drop rows without a valid year because they cannot be assigned to train or test.
ml_model_df = ml_model_df.loc[ml_model_df[YEAR_COL].notna()].copy()

# Convert year to integer for clean sorting and comparison.
ml_model_df[YEAR_COL] = ml_model_df[YEAR_COL].astype(int)

# Sort by time and identifiers to make the split deterministic and easy to inspect.
ml_model_df = ml_model_df.sort_values([YEAR_COL] + id_columns).reset_index(drop=True)

# Train on observations before 2020; test on observations from 2020 onward.
train_df = ml_model_df.loc[ml_model_df[YEAR_COL] < TEST_START_YEAR].copy()
test_df = ml_model_df.loc[ml_model_df[YEAR_COL] >= TEST_START_YEAR].copy()

# Stop early if the cutoff year does not create both train and test samples.
if train_df.empty or test_df.empty:
    raise ValueError(
        "The selected TEST_START_YEAR does not create both train and test samples. "
        "Choose a cutoff year inside the observed year range."
    )

# Make sure identifiers and target variables are not accidentally used as predictors.
for forbidden in [SUPERVISED_TARGET, TARGET_LEVEL_COL, *id_columns]:
    if forbidden in model_features:
        raise ValueError(f"Forbidden column found in model_features: {forbidden}")

# Build the feature matrices for train and test.
# scikit-learn imputers expect missing values as np.nan, not pandas pd.NA.
X_train = train_df[model_features].copy().astype(object)
X_test = test_df[model_features].copy().astype(object)
X_train = X_train.where(pd.notna(X_train), np.nan)
X_test = X_test.where(pd.notna(X_test), np.nan)

# Build the target vectors for train and test.
y_train = train_df[SUPERVISED_TARGET].astype(float).copy()
y_test = test_df[SUPERVISED_TARGET].astype(float).copy()

# Summarize the split to check sample size, firm coverage, years, and target stability.
split_summary = pd.DataFrame({
    "rows": [len(train_df), len(test_df)],
    "firms": [train_df["companyid"].nunique(), test_df["companyid"].nunique()],
    "first_year": [train_df[YEAR_COL].min(), test_df[YEAR_COL].min()],
    "last_year": [train_df[YEAR_COL].max(), test_df[YEAR_COL].max()],
    "turnover_rate": [
        train_df["ceo_turnover"].mean() if "ceo_turnover" in train_df else np.nan,
        test_df["ceo_turnover"].mean() if "ceo_turnover" in test_df else np.nan,
    ],
    "target_mean": [y_train.mean(), y_test.mean()],
    "target_std": [y_train.std(), y_test.std()],
}, index=["train", "test"])

print(f"Test period starts in {TEST_START_YEAR}.")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape : {X_test.shape}")

display(split_summary.round(3))

**Insights:**

- The split uses observations before **2020** for training and observations from **2020 onward** for testing.

- The training set contains **1,701 observations** and the test set contains **557 observations**, so the holdout period remains large enough for evaluation.

- Both periods cover many firms: **129 firms** in training and **122 firms** in testing. This means the test set is not driven by only a few companies.

- The target mean is similar across periods (**7.513** in training vs. **7.579** in testing), and the target standard deviation is also close (**0.715** vs. **0.679**). This suggests that the CEO-pay target distribution is reasonably stable across the split.

- The test set is kept separate until final evaluation, so preprocessing and model selection should be learned from the training period only.

<a id="supervised-learning-preprocessing"></a>
### 8.4 Preprocessing Pipeline: Imputation, Encoding, and Scaling

This subsection prepares the model inputs after the time-based train/test split. The target has already been separated, so the task here is only to make the feature matrix usable for scikit-learn models.

The preprocessing has three parts. First, missing numeric values are median-imputed, missingness indicators are added, and numeric values are scaled. Second, missing categorical values are imputed and categories are one-hot encoded. Third, both branches are combined with a `ColumnTransformer`, so each column type receives the right transformation.

The transformer is fitted on `X_train` only and then applied to both `X_train` and `X_test`. This preserves the forecasting logic of the split and avoids data leakage from the test period.

##### Encoding Categorical Features

The categorical predictors are `gender`, `hocountryname`, `sector`, and `annualreportyear`. These variables are labels rather than continuous measurements, so we use one-hot encoding rather than assigning arbitrary numeric codes.

Before encoding, missing categories are filled with the most frequent training-set value. The encoder ignores unknown categories, which is important for a time-based split because a country, sector, or year may appear in the test period even if it was not present in the training period.

In [ ]:
# -- 8.4.1 Categorical preprocessing --------------------------------------

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

categorical_transformer

##### Imputing and Scaling Numeric Features

The numeric predictors are median-imputed and then standardized. Median imputation is a conservative choice for financial variables because firm size, profitability, and leverage can be skewed.

The imputer also adds missingness indicators for numeric columns. This keeps the imputation simple while allowing the model to learn whether missing accounting or executive fields are themselves informative.

In [ ]:
# -- 8.4.2 Numeric preprocessing ------------------------------------------

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])

numeric_transformer

##### Combining Steps with ColumnTransformer

The `ColumnTransformer` keeps the preprocessing rules explicit: numeric features go through the numeric pipeline, and categorical features go through the categorical pipeline.

This makes the workflow easier to inspect and safer to reuse in modelling. Later model pipelines can call the same `preprocessor` object, ensuring that every model receives the same cleaned and encoded feature matrix.

In [ ]:
# -- 8.4.3 Combined preprocessing pipeline --------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

preprocessor

##### Fitting the Preprocessor

We fit the preprocessing transformer before modelling so we can inspect the transformed design matrix. The fit is done on `X_train` only.

The fitted transformer is then used to transform both train and test features. The displayed shape summary shows how many columns the preprocessing creates, and the preview helps verify that scaling, missingness indicators, and one-hot encoding worked as expected.

In [ ]:
# -- 8.4.4 Fit preprocessing transformer ----------------------------------

fitted_preprocessor = preprocessor.fit(X_train)

X_train_preprocessed = fitted_preprocessor.transform(X_train)
X_test_preprocessed = fitted_preprocessor.transform(X_test)
preprocessed_feature_names = fitted_preprocessor.get_feature_names_out()

if hasattr(X_train_preprocessed, "toarray"):
    X_train_preprocessed = X_train_preprocessed.toarray()
    X_test_preprocessed = X_test_preprocessed.toarray()

preprocessed_shape_summary = pd.DataFrame({
    "rows": [X_train_preprocessed.shape[0], X_test_preprocessed.shape[0]],
    "features_after_preprocessing": [
        X_train_preprocessed.shape[1],
        X_test_preprocessed.shape[1],
    ],
}, index=["train", "test"])

missing_indicator_features = [
    name for name in preprocessed_feature_names
    if "missingindicator" in name
]

print(f"Transformed feature count: {len(preprocessed_feature_names):,}")
print(f"Numeric missingness indicators added: {len(missing_indicator_features)}")
display(preprocessed_shape_summary)

preprocessed_preview = pd.DataFrame(
    X_train_preprocessed[:5],
    columns=preprocessed_feature_names,
    index=X_train.index[:5],
)

display(preprocessed_preview.round(3))



**Insights:**

- The preprocessing step expands the **11 raw model predictors** into **83 transformed features**.

- This expansion comes mainly from one-hot encoding the categorical variables: `gender`, `hocountryname`, `sector`, and `annualreportyear`.

- Four numeric missingness indicators are added because some numeric predictors contain missing values. This lets the model learn whether missingness itself carries predictive information.

- Numeric features are median-imputed and standardized, which is appropriate because variables such as profitability, leverage, assets, and employee size have different scales and outliers.

- The preprocessor is fitted only on `X_train` and then applied to `X_test`, which avoids leaking information from the test period into the training process.

<a id="supervised-learning-tuning-cache"></a>
### 8.5 Tuning Cache

The Optuna searches below are the slowest part of the supervised-learning section. The notebook therefore uses a **local, optional cache** under `../data/model_cache`. On a fresh clone, this folder may not exist; the notebook creates it automatically and reruns the missing tuning trials.

The cache is meant to speed up local reruns, not to be a required project input. Generated Optuna databases and fitted model files should be ignored by Git. Whenever the feature set, preprocessing, target, or sample definition changes, `STUDY_VERSION` should be increased so old tuning results are not silently reused.

In [ ]:
# -- 8.5 Persistent tuning cache ------------------------------------------
# The cache is optional: if it is missing on a fresh clone, Optuna reruns the search.
# Increase STUDY_VERSION whenever features, preprocessing, or sample rules change.

TUNING_CACHE_DIR = DATA_DIR / "model_cache"
TUNING_CACHE_DIR.mkdir(parents=True, exist_ok=True)

OPTUNA_DB_PATH = TUNING_CACHE_DIR / "optuna_studies.db"
OPTUNA_STORAGE_URL = f"sqlite:///{OPTUNA_DB_PATH.as_posix()}"

N_OPTUNA_TRIALS = 50
STUDY_VERSION = "v4"

study_context = {
    "target": SUPERVISED_TARGET,
    "study_version": STUDY_VERSION,
    "train_rows": int(len(y_train)),
    "test_rows": int(len(y_test)),
    "raw_features": int(len(model_features)),
    "features_after_preprocessing": int(X_train_preprocessed.shape[1]),
    "test_start_year": int(TEST_START_YEAR),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "cv_preprocessing": "inside_pipeline",
}


def run_cached_optuna_study(
    study_name,
    objective,
    n_trials=N_OPTUNA_TRIALS,
    storage_url=OPTUNA_STORAGE_URL,
    sampler_seed=42,
    context=None,
):
    "Load an existing Optuna study or run the missing number of trials."
    sampler = optuna.samplers.TPESampler(seed=sampler_seed)
    study = optuna.create_study(
        study_name=study_name,
        direction="minimize",
        storage=storage_url,
        load_if_exists=True,
        sampler=sampler,
    )

    if context is not None:
        study.set_user_attr("context", context)

    completed_trials = sum(
        trial.state == optuna.trial.TrialState.COMPLETE
        for trial in study.trials
    )
    remaining_trials = max(0, n_trials - completed_trials)

    if remaining_trials > 0:
        print(
            f"Running {remaining_trials} new trial(s) for {study_name} "
            f"({completed_trials}/{n_trials} already complete)."
        )
        study.optimize(objective, n_trials=remaining_trials, show_progress_bar=True)
    else:
        print(
            f"Loaded cached Optuna study {study_name} "
            f"with {completed_trials} completed trial(s)."
        )

    return study


def save_best_params(study, filename, context=None):
    "Save the best Optuna parameters with setup metadata."
    payload = {
        "study_name": study.study_name,
        "best_value": float(study.best_value),
        "best_params": study.best_params,
        "completed_trials": sum(
            trial.state == optuna.trial.TrialState.COMPLETE
            for trial in study.trials
        ),
        "context": context or {},
    }
    path = TUNING_CACHE_DIR / filename
    path.write_text(json.dumps(payload, indent=2))
    print(f"Saved best parameters: {path}")
    return path


print(f"Tuning cache directory: {TUNING_CACHE_DIR}")
print(f"Optuna storage: {OPTUNA_STORAGE_URL}")
print(f"Study version: {STUDY_VERSION}")
print("Study context:")
display(pd.Series(study_context).to_frame("value"))

<a id="supervised-learning-pay-regression"></a>
### 8.6 CEO Pay Regression Models

The CEO-pay regression block compares simple linear baselines with tree-based and boosting models. The workflow is intentionally staged: first screen model families with cross-validation, then tune the two strongest candidates, and finally evaluate the tuned models once on the held-out post-2020 test period.

All cross-validation in this section uses full scikit-learn pipelines. This means imputation, missingness indicators, scaling, and one-hot encoding are fitted inside each training fold rather than once on the full training period.

#### 8.6.1 Initial Model Screening

We begin with a broad but lightweight comparison of regression models using cross-validation on the training period only. Each model is wrapped in a pipeline with the preprocessing step, so preprocessing is re-learned inside each fold. MSE is used because the target is logged CEO pay and larger errors should receive more penalty.

In [ ]:
# -- 8.6.1 Initial model screening via pipeline CV ------------------------

from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit

RANDOM_STATE = 654321
PAY_CV = TimeSeriesSplit(n_splits=5)

screen_models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01, max_iter=10000),
    "SGDRegressor": SGDRegressor(max_iter=1000, tol=1e-3, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "Light Gradient Boosting": lgb.LGBMRegressor(
        objective="regression",
        verbose=-1,
        random_state=RANDOM_STATE,
    ),
    "XGBoost": xgb.XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        random_state=RANDOM_STATE,
        verbosity=0,
        n_jobs=-1,
    ),
    "AdaBoost": AdaBoostRegressor(random_state=RANDOM_STATE),
}


def make_model_pipeline(model):
    "Build a fresh preprocessing + model pipeline for CV or final fitting."
    return Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", model),
    ])


def cross_validated_mse(model, X, y, cv=PAY_CV):
    pipeline = make_model_pipeline(model)
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="X does not have valid feature names.*",
            category=UserWarning,
        )
        scores = cross_val_score(
            pipeline,
            X,
            np.ravel(y),
            scoring="neg_mean_squared_error",
            cv=cv,
            error_score="raise",
        )
    mse_scores = -scores
    return mse_scores.mean(), mse_scores.std(ddof=1)


screen_results = {}
for name, model in screen_models.items():
    mean_mse, std_mse = cross_validated_mse(model, X_train, y_train)
    screen_results[name] = {"CV MSE (mean)": mean_mse, "CV MSE (std)": std_mse}
    print(f"{name:30s}  CV MSE: {mean_mse:.4f} +/- {std_mse:.4f}")

screen_df = (
    pd.DataFrame(screen_results).T
    .rename_axis("Model")
    .sort_values("CV MSE (mean)")
)

display(screen_df.round(4))

#### 8.6.2 XGBoost Hyperparameter Search (Optuna)

XGBoost is tuned with Optuna over learning rate, number of trees, tree depth, subsampling, and regularization. The study is saved in the local optional cache with the current `STUDY_VERSION`, so rerunning the notebook continues or reloads the matching search instead of starting from zero.

In [ ]:
# -- 8.6.2 XGBoost hyperparameter search with pipeline CV -----------------

def xgb_objective(trial):
    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "booster": "gbtree",
        "verbosity": 0,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }
    pipeline = make_model_pipeline(xgb.XGBRegressor(**params))
    scores = cross_val_score(
        pipeline,
        X_train,
        np.ravel(y_train),
        scoring="neg_mean_squared_error",
        cv=PAY_CV,
        error_score="raise",
    )
    return -scores.mean()


xgb_study = run_cached_optuna_study(
    study_name=f"ceo_pay_xgboost_{STUDY_VERSION}",
    objective=xgb_objective,
    sampler_seed=RANDOM_STATE,
    context=study_context,
)
save_best_params(xgb_study, f"ceo_pay_xgb_best_params_{STUDY_VERSION}.json", context=study_context)

print("Best XGBoost hyperparameters:")
for k, v in xgb_study.best_params.items():
    print(f"  {k}: {v}")
print(f"Best CV MSE: {xgb_study.best_value:.4f}")

#### 8.6.3 Random Forest Hyperparameter Search (Optuna)

Random Forest is tuned over tree count, depth, split size, leaf size, and feature subsampling. The search uses the same pipeline-CV pattern as XGBoost, with a cache name tied to the current `STUDY_VERSION`.

In [ ]:
# -- 8.6.3 Random Forest hyperparameter search with pipeline CV -----------

def rf_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "max_depth": trial.suggest_int("max_depth", 3, 24),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.5, 0.7, 1.0]),
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    }
    pipeline = make_model_pipeline(RandomForestRegressor(**params))
    scores = cross_val_score(
        pipeline,
        X_train,
        np.ravel(y_train),
        scoring="neg_mean_squared_error",
        cv=PAY_CV,
        error_score="raise",
    )
    return -scores.mean()


rf_study = run_cached_optuna_study(
    study_name=f"ceo_pay_random_forest_{STUDY_VERSION}",
    objective=rf_objective,
    sampler_seed=RANDOM_STATE,
    context=study_context,
)
save_best_params(rf_study, f"ceo_pay_rf_best_params_{STUDY_VERSION}.json", context=study_context)

print("Best Random Forest hyperparameters:")
for k, v in rf_study.best_params.items():
    print(f"  {k}: {v}")
print(f"Best CV MSE: {rf_study.best_value:.4f}")

#### 8.6.4 Final Evaluation on Test Set

The tuned XGBoost and Random Forest pipelines are retrained on the full training period and evaluated once on the 2020-onward test period. RidgeCV remains the transparent linear benchmark. Final fitted pipelines and the test-set metric table are saved to the optional local model cache.

In [ ]:
# -- 8.6.4 Final test-set evaluation with full pipelines ------------------

ridge_pipe = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=PAY_CV, scoring="r2")),
])
ridge_pipe.fit(X_train, y_train)
ridge_pred = ridge_pipe.predict(X_test)

best_xgb_pipe = make_model_pipeline(
    xgb.XGBRegressor(
        **xgb_study.best_params,
        objective="reg:squarederror",
        eval_metric="rmse",
        verbosity=0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
)
best_xgb_pipe.fit(X_train, y_train)
xgb_pred = best_xgb_pipe.predict(X_test)

best_rf_pipe = make_model_pipeline(
    RandomForestRegressor(
        **rf_study.best_params,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
)
best_rf_pipe.fit(X_train, y_train)
rf_pred = best_rf_pipe.predict(X_test)


def regression_metrics(name, y_true, y_pred):
    return {
        "Model": name,
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MSE": mean_squared_error(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
    }


final_results = pd.DataFrame([
    regression_metrics("Ridge (baseline)", y_test, ridge_pred),
    regression_metrics("XGBoost (tuned)", y_test, xgb_pred),
    regression_metrics("Random Forest (tuned)", y_test, rf_pred),
]).set_index("Model").sort_values("R2", ascending=False)

joblib.dump(ridge_pipe, TUNING_CACHE_DIR / f"ceo_pay_ridge_pipeline_{STUDY_VERSION}.joblib")
joblib.dump(best_xgb_pipe, TUNING_CACHE_DIR / f"ceo_pay_xgb_pipeline_{STUDY_VERSION}.joblib")
joblib.dump(best_rf_pipe, TUNING_CACHE_DIR / f"ceo_pay_rf_pipeline_{STUDY_VERSION}.joblib")
final_results.to_csv(TUNING_CACHE_DIR / f"ceo_pay_test_metrics_{STUDY_VERSION}.csv")

print("Test-set performance:")
display(final_results.round(4))
print(f"Saved CEO-pay pipelines and metrics to: {TUNING_CACHE_DIR}")

In [ ]:
# -- 8.6.5 Predicted-vs-actual and residual diagnostics -------------------

preds_map = {
    "Ridge (baseline)": ridge_pred,
    "XGBoost (tuned)": xgb_pred,
    "Random Forest (tuned)": rf_pred,
}
best_name = final_results.index[0]
best_pred = preds_map[best_name]
residuals = y_test.to_numpy() - best_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.scatter(y_test, best_pred, alpha=0.45, s=18, color="steelblue")
lo = min(float(y_test.min()), float(best_pred.min()))
hi = max(float(y_test.max()), float(best_pred.max()))
ax.plot([lo, hi], [lo, hi], "r--", linewidth=1)
ax.set_xlabel("Actual log real CEO pay")
ax.set_ylabel("Predicted log real CEO pay")
ax.set_title(f"Predicted vs. Actual\n{best_name}")

ax = axes[1]
ax.scatter(best_pred, residuals, alpha=0.45, s=18, color="steelblue")
ax.axhline(0, color="red", linestyle="--", linewidth=1)
ax.set_xlabel("Predicted log real CEO pay")
ax.set_ylabel("Residual: actual minus predicted")
ax.set_title(f"Residuals\n{best_name}")

plt.tight_layout()
plt.show()

In [ ]:
# -- 8.6.6 Feature importance ---------------------------------------------

xgb_preprocessor = best_xgb_pipe.named_steps["preprocessor"]
rf_preprocessor = best_rf_pipe.named_steps["preprocessor"]
best_xgb = best_xgb_pipe.named_steps["model"]
best_rf = best_rf_pipe.named_steps["model"]

xgb_feature_names = xgb_preprocessor.get_feature_names_out()
rf_feature_names = rf_preprocessor.get_feature_names_out()

xgb_imp = pd.Series(best_xgb.feature_importances_, index=xgb_feature_names, name="importance")
rf_imp = pd.Series(best_rf.feature_importances_, index=rf_feature_names, name="importance")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

xgb_imp.nlargest(15).sort_values().plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("XGBoost (tuned): Top 15 Feature Importances")
axes[0].set_xlabel("Relative importance")

rf_imp.nlargest(15).sort_values().plot(kind="barh", ax=axes[1], color="darkorange")
axes[1].set_title("Random Forest (tuned): Top 15 Feature Importances")
axes[1].set_xlabel("Mean decrease in impurity")

plt.tight_layout()
plt.show()

top_importance = pd.concat(
    [
        xgb_imp.nlargest(10).rename("XGBoost"),
        rf_imp.nlargest(10).rename("Random Forest"),
    ],
    axis=1,
).fillna(0)

display(top_importance.round(4))

#### 8.6.5 CEO-Pay Regression Takeaways

- **Initial screening** compares linear baselines, regularized linear models, single trees, random forests, boosting models, and SGD on the training period only.
- **Pipeline cross-validation** fits preprocessing inside each fold, so imputation, scaling, missingness indicators, and one-hot encoding are not learned from validation-fold observations.
- **Optuna tuning** uses the current `STUDY_VERSION`, so saved searches are not mixed with older feature or preprocessing definitions.
- **Final test-set comparison** reports R2, RMSE, MSE, and MAE for RidgeCV, tuned XGBoost, and tuned Random Forest on the 2020-onward holdout period.
- **Feature importance** should be read descriptively: it shows which transformed predictors help prediction, not causal effects on CEO pay.

<a id="supervised-learning-turnover-classification"></a>
### 8.7 CEO Turnover Classification

Beyond predicting pay levels, we ask whether observable firm and executive characteristics predict whether a CEO turnover occurs in a firm-year. This is a binary classification task with `ceo_turnover` as the outcome.

The feature set excludes `ceo_turnover` and `post_turnover` to avoid leakage. It reuses the same time-based train/test split as the CEO-pay regression, so the classification task is evaluated on the same 2020-onward holdout period.

Two models are compared: a balanced Logistic Regression baseline and an XGBoost classifier with class-imbalance weighting. Both models use a decision threshold selected on the training set with Youden J rather than the default 0.5. Evaluation emphasizes imbalance-aware measures: ROC-AUC, Average Precision, Balanced Accuracy, and MCC.

In [ ]:
# -- 8.7 CEO-turnover classification --------------------------------------

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    roc_curve,
    PrecisionRecallDisplay,
    ConfusionMatrixDisplay,
)

CLF_TARGET = "ceo_turnover"
if CLF_TARGET not in train_df.columns or CLF_TARGET not in test_df.columns:
    raise KeyError(f"{CLF_TARGET} is required for the turnover-classification task.")

clf_numeric_features = [
    feature for feature in numeric_features
    if feature not in {CLF_TARGET, "post_turnover"}
]
clf_categorical_features = categorical_features.copy()
clf_features = clf_numeric_features + clf_categorical_features

for forbidden in {CLF_TARGET, "post_turnover", SUPERVISED_TARGET, TARGET_LEVEL_COL}:
    if forbidden in clf_features:
        raise ValueError(f"Leakage column found in classification features: {forbidden}")

clf_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, clf_numeric_features),
        ("cat", categorical_transformer, clf_categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

X_train_clf = train_df[clf_features].copy().astype(object)
X_test_clf = test_df[clf_features].copy().astype(object)
X_train_clf = X_train_clf.where(pd.notna(X_train_clf), np.nan)
X_test_clf = X_test_clf.where(pd.notna(X_test_clf), np.nan)

y_train_clf = train_df[CLF_TARGET].fillna(0).astype(int)
y_test_clf = test_df[CLF_TARGET].fillna(0).astype(int)

if y_train_clf.nunique() < 2 or y_test_clf.nunique() < 2:
    raise ValueError("Train and test sets must each contain both turnover classes.")

X_train_clf_prep = clf_preprocessor.fit_transform(X_train_clf)
X_test_clf_prep = clf_preprocessor.transform(X_test_clf)

if hasattr(X_train_clf_prep, "toarray"):
    X_train_clf_prep = X_train_clf_prep.toarray()
    X_test_clf_prep = X_test_clf_prep.toarray()

train_neg, train_pos = (y_train_clf == 0).sum(), (y_train_clf == 1).sum()
test_neg, test_pos = (y_test_clf == 0).sum(), (y_test_clf == 1).sum()
scale_pos_weight = train_neg / train_pos
chance_ap = test_pos / (test_neg + test_pos)

class_balance = pd.DataFrame({
    "rows": [len(y_train_clf), len(y_test_clf)],
    "turnover_events": [train_pos, test_pos],
    "turnover_rate": [y_train_clf.mean(), y_test_clf.mean()],
}, index=["train", "test"])

display(class_balance.round(3))


def youden_threshold(model, X, y):
    "Find the finite decision threshold that maximizes Youden J on training data."
    fpr, tpr, thresholds = roc_curve(y, model.predict_proba(X)[:, 1])
    finite = np.isfinite(thresholds)
    if not finite.any():
        return 0.5
    j_scores = tpr[finite] - fpr[finite]
    return thresholds[finite][np.argmax(j_scores)]


lr_clf = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=RANDOM_STATE,
)
lr_clf.fit(X_train_clf_prep, y_train_clf)
lr_threshold = youden_threshold(lr_clf, X_train_clf_prep, y_train_clf)
lr_prob = lr_clf.predict_proba(X_test_clf_prep)[:, 1]
lr_pred = (lr_prob >= lr_threshold).astype(int)

xgb_clf = xgb.XGBClassifier(
    objective="binary:logistic",
    scale_pos_weight=scale_pos_weight,
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    verbosity=0,
    eval_metric="aucpr",
    n_jobs=-1,
)
xgb_clf.fit(X_train_clf_prep, y_train_clf)
xgb_threshold = youden_threshold(xgb_clf, X_train_clf_prep, y_train_clf)
xgb_prob = xgb_clf.predict_proba(X_test_clf_prep)[:, 1]
xgb_pred = (xgb_prob >= xgb_threshold).astype(int)


def clf_metrics(y_true, y_pred, y_prob):
    return {
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "Avg Precision": average_precision_score(y_true, y_prob),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "MCC": matthews_corrcoef(y_true, y_pred),
    }


results_clf = pd.DataFrame({
    "Logistic Regression": clf_metrics(y_test_clf, lr_pred, lr_prob),
    "XGBoost": clf_metrics(y_test_clf, xgb_pred, xgb_prob),
}).T.rename_axis("Model")

joblib.dump(clf_preprocessor, TUNING_CACHE_DIR / f"turnover_clf_preprocessor_{STUDY_VERSION}.joblib")
joblib.dump(lr_clf, TUNING_CACHE_DIR / f"turnover_logistic_model_{STUDY_VERSION}.joblib")
joblib.dump(xgb_clf, TUNING_CACHE_DIR / f"turnover_xgb_model_{STUDY_VERSION}.joblib")
results_clf.to_csv(TUNING_CACHE_DIR / f"turnover_classification_metrics_{STUDY_VERSION}.csv")

print(
    f"Chance baseline on test set: Avg Precision ~= {chance_ap:.3f}; "
    "Balanced Accuracy = 0.500; MCC = 0.000"
)
display(results_clf.round(3))

print(f"Logistic Regression threshold = {lr_threshold:.3f}")
print(classification_report(
    y_test_clf,
    lr_pred,
    target_names=["No Turnover", "Turnover"],
    zero_division=0,
))
print(f"XGBoost threshold = {xgb_threshold:.3f}")
print(classification_report(
    y_test_clf,
    xgb_pred,
    target_names=["No Turnover", "Turnover"],
    zero_division=0,
))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

PrecisionRecallDisplay.from_predictions(
    y_test_clf,
    lr_prob,
    ax=axes[0],
    name="Logistic Regression",
)
PrecisionRecallDisplay.from_predictions(
    y_test_clf,
    xgb_prob,
    ax=axes[0],
    name="XGBoost",
)
axes[0].axhline(chance_ap, color="grey", linestyle="--", label=f"Chance ({chance_ap:.2f})")
axes[0].legend(fontsize=9)
axes[0].set_title("Precision-Recall Curves")

ConfusionMatrixDisplay.from_predictions(
    y_test_clf,
    lr_pred,
    display_labels=["No Turnover", "Turnover"],
    ax=axes[1],
    colorbar=False,
)
axes[1].set_title(f"Logistic Regression (thr={lr_threshold:.2f})")

ConfusionMatrixDisplay.from_predictions(
    y_test_clf,
    xgb_pred,
    display_labels=["No Turnover", "Turnover"],
    ax=axes[2],
    colorbar=False,
)
axes[2].set_title(f"XGBoost (thr={xgb_threshold:.2f})")

plt.tight_layout()
plt.show()

<a id="supervised-learning-dispersion-regression"></a>
### 8.8 Pay Dispersion Regression

The second supervised-learning outcome is `log_pay_dispersion`: the log of the winsorized CEO-to-executive pay ratio constructed in Section 5.4.4. A high ratio means the CEO earns substantially more than the firm's median named executive; a low ratio means compressed internal pay inequality.

This task uses an explicit, leakage-safe feature list rather than all available columns. Direct pay variables, the CEO-pay target, pay-ratio components, turnover indicators, and identifiers are excluded. The same time-based train/test logic applies, but the sample is smaller because pay-dispersion coverage is available for only part of the panel.

<a id="supervised-learning-dispersion-setup"></a>
#### 8.8.1 Pay-Dispersion Sample Setup

The pay-dispersion task switches the outcome to `log_pay_dispersion` and rebuilds a dedicated model sample. It keeps the same broad predictor families as the CEO-pay model: CEO age, firm profitability, leverage, firm size, country, sector, and year. CEO turnover indicators are excluded because this block asks how much ordinary observable structure predicts dispersion, not whether the treatment variable mechanically separates the outcome.

In [ ]:
# -- 8.8.1 Pay-dispersion sample and preprocessor -------------------------

DISP_TARGET = "log_pay_dispersion"

if DISP_TARGET not in df_panel_final.columns:
    raise KeyError(f"{DISP_TARGET} not found. Re-run Section 5.4.4 before Section 8.8.")

disp_numeric_features = [
    feature for feature in numeric_features
    if feature not in {"ceo_turnover", "post_turnover"}
]
disp_categorical_features = categorical_features.copy()
disp_features = disp_numeric_features + disp_categorical_features

for forbidden in {
    DISP_TARGET,
    SUPERVISED_TARGET,
    TARGET_LEVEL_COL,
    "pay_ratio",
    "w_pay_ratio",
    "median_exec_pay_eur",
    "n_exec_in_ratio",
    "ceo_turnover",
    "post_turnover",
    *id_columns,
}:
    if forbidden in disp_features:
        raise ValueError(f"Leakage or identifier column found in dispersion features: {forbidden}")

disp_required_cols = disp_features + [DISP_TARGET, YEAR_COL, "companyid"]
disp_missing_cols = [col for col in disp_required_cols if col not in df_panel_final.columns]
if disp_missing_cols:
    raise KeyError("Missing pay-dispersion modeling column(s): " + ", ".join(disp_missing_cols))

disp_model_df = df_panel_final.loc[
    df_panel_final[DISP_TARGET].notna(),
    list(dict.fromkeys(disp_required_cols)),
].copy()

for col in disp_numeric_features + [DISP_TARGET, YEAR_COL]:
    disp_model_df[col] = pd.to_numeric(disp_model_df[col], errors="coerce")

disp_model_df = disp_model_df.loc[disp_model_df[YEAR_COL].notna()].copy()
disp_model_df[YEAR_COL] = disp_model_df[YEAR_COL].astype(int)
disp_model_df = disp_model_df.sort_values([YEAR_COL, "companyid"]).reset_index(drop=True)

disp_train_df = disp_model_df.loc[disp_model_df[YEAR_COL] < TEST_START_YEAR].copy()
disp_test_df = disp_model_df.loc[disp_model_df[YEAR_COL] >= TEST_START_YEAR].copy()

if disp_train_df.empty or disp_test_df.empty:
    raise ValueError("The pay-dispersion split must contain both train and test observations.")

X_disp_train = disp_train_df[disp_features].copy().astype(object)
X_disp_test = disp_test_df[disp_features].copy().astype(object)
X_disp_train = X_disp_train.where(pd.notna(X_disp_train), np.nan)
X_disp_test = X_disp_test.where(pd.notna(X_disp_test), np.nan)
y_disp_train = disp_train_df[DISP_TARGET].astype(float)
y_disp_test = disp_test_df[DISP_TARGET].astype(float)

disp_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, disp_numeric_features),
        ("cat", categorical_transformer, disp_categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

disp_sample_summary = pd.DataFrame({
    "rows": [len(disp_train_df), len(disp_test_df), len(disp_model_df)],
    "firms": [
        disp_train_df["companyid"].nunique(),
        disp_test_df["companyid"].nunique(),
        disp_model_df["companyid"].nunique(),
    ],
    "first_year": [
        disp_train_df[YEAR_COL].min(),
        disp_test_df[YEAR_COL].min(),
        disp_model_df[YEAR_COL].min(),
    ],
    "last_year": [
        disp_train_df[YEAR_COL].max(),
        disp_test_df[YEAR_COL].max(),
        disp_model_df[YEAR_COL].max(),
    ],
    "target_mean": [y_disp_train.mean(), y_disp_test.mean(), disp_model_df[DISP_TARGET].mean()],
}, index=["train", "test", "all"])

disp_feature_missingness = X_disp_train.isna().sum().to_frame("train_missing")
disp_feature_missingness["train_missing_pct"] = (
    disp_feature_missingness["train_missing"] / len(X_disp_train) * 100
).round(2)

print(f"Pay-dispersion usable rows: {len(disp_model_df):,} of {len(df_panel_final):,} panel rows")
print(f"Dispersion numeric features: {len(disp_numeric_features)}")
print(f"Dispersion categorical features: {len(disp_categorical_features)}")
display(disp_sample_summary.round(3))
display(disp_feature_missingness.sort_values("train_missing_pct", ascending=False))

In [ ]:
# -- 8.8.2 Pay-dispersion model screening ---------------------------------

DISP_RANDOM_STATE = RANDOM_STATE
DISP_CV = TimeSeriesSplit(n_splits=5)

disp_screen_models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01, max_iter=10000),
    "SGDRegressor": SGDRegressor(max_iter=1000, tol=1e-3, random_state=DISP_RANDOM_STATE),
    "Decision Tree": DecisionTreeRegressor(random_state=DISP_RANDOM_STATE),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=DISP_RANDOM_STATE,
        n_jobs=-1,
    ),
    "XGBoost": xgb.XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        random_state=DISP_RANDOM_STATE,
        verbosity=0,
        n_jobs=-1,
    ),
    "AdaBoost": AdaBoostRegressor(random_state=DISP_RANDOM_STATE),
}

rows = []
for name, model in disp_screen_models.items():
    pipe = Pipeline([("pre", disp_preprocessor), ("model", model)])
    scores = cross_val_score(
        pipe,
        X_disp_train,
        np.ravel(y_disp_train),
        cv=DISP_CV,
        scoring="neg_mean_squared_error",
        error_score="raise",
    )
    mse_scores = -scores
    rows.append({
        "Model": name,
        "CV MSE (mean)": mse_scores.mean(),
        "CV MSE (std)": mse_scores.std(ddof=1),
    })

disp_screen_df = (
    pd.DataFrame(rows)
    .set_index("Model")
    .sort_values("CV MSE (mean)")
)

display(disp_screen_df.round(4))

best_disp_screen_mse = disp_screen_df["CV MSE (mean)"].iloc[0]
print(f"Best screening CV MSE: {best_disp_screen_mse:.4f} ({disp_screen_df.index[0]})")

#### 8.8.2 Pay Dispersion - Optuna Tuning (Random Forest & XGBoost)

The screening table identifies the strongest model families on the training period. We tune Random Forest and XGBoost with cache names tied to the current `STUDY_VERSION`, then leave the final comparison to the held-out 2020-onward test set.

In [ ]:
# -- 8.8.3 Optuna tuning for pay-dispersion top models --------------------

from sklearn.base import clone


def disp_xgb_objective(trial):
    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "booster": "gbtree",
        "verbosity": 0,
        "random_state": DISP_RANDOM_STATE,
        "n_jobs": -1,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }
    pipe = Pipeline([
        ("pre", clone(disp_preprocessor)),
        ("model", xgb.XGBRegressor(**params)),
    ])
    scores = cross_val_score(
        pipe,
        X_disp_train,
        np.ravel(y_disp_train),
        scoring="neg_mean_squared_error",
        cv=DISP_CV,
        error_score="raise",
    )
    return -scores.mean()


def disp_rf_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "max_depth": trial.suggest_int("max_depth", 3, 24),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.5, 0.7, 1.0]),
        "random_state": DISP_RANDOM_STATE,
        "n_jobs": -1,
    }
    pipe = Pipeline([
        ("pre", clone(disp_preprocessor)),
        ("model", RandomForestRegressor(**params)),
    ])
    scores = cross_val_score(
        pipe,
        X_disp_train,
        np.ravel(y_disp_train),
        scoring="neg_mean_squared_error",
        cv=DISP_CV,
        error_score="raise",
    )
    return -scores.mean()


disp_study_context = {
    "target": DISP_TARGET,
    "train_rows": int(len(y_disp_train)),
    "test_rows": int(len(y_disp_test)),
    "raw_features": int(len(disp_features)),
    "test_start_year": int(TEST_START_YEAR),
}

disp_xgb_study = run_cached_optuna_study(
    study_name=f"pay_dispersion_xgboost_{STUDY_VERSION}",
    objective=disp_xgb_objective,
    sampler_seed=DISP_RANDOM_STATE,
    context=disp_study_context,
)
save_best_params(disp_xgb_study, f"pay_dispersion_xgb_best_params_{STUDY_VERSION}.json", context=disp_study_context)

print("Best XGBoost params (dispersion):")
for k, v in disp_xgb_study.best_params.items():
    print(f"  {k}: {v}")
print(f"Best CV MSE: {disp_xgb_study.best_value:.4f}")

disp_rf_study = run_cached_optuna_study(
    study_name=f"pay_dispersion_random_forest_{STUDY_VERSION}",
    objective=disp_rf_objective,
    sampler_seed=DISP_RANDOM_STATE,
    context=disp_study_context,
)
save_best_params(disp_rf_study, f"pay_dispersion_rf_best_params_{STUDY_VERSION}.json", context=disp_study_context)

print()
print("Best Random Forest params (dispersion):")
for k, v in disp_rf_study.best_params.items():
    print(f"  {k}: {v}")
print(f"Best CV MSE: {disp_rf_study.best_value:.4f}")

#### 8.8.3 Final Evaluation on Test Set

The tuned pay-dispersion models are retrained on the full training period and compared with RidgeCV on the held-out test years. The fitted dispersion preprocessor, final models, and metric table are saved to the model cache.

In [ ]:
# -- 8.8.4 Final test-set evaluation for pay dispersion -------------------

disp_pre_fitted = disp_preprocessor.fit(X_disp_train)
X_disp_train_pp = disp_pre_fitted.transform(X_disp_train)
X_disp_test_pp = disp_pre_fitted.transform(X_disp_test)

if hasattr(X_disp_train_pp, "toarray"):
    X_disp_train_pp = X_disp_train_pp.toarray()
    X_disp_test_pp = X_disp_test_pp.toarray()

disp_feature_names = disp_pre_fitted.get_feature_names_out()

print(f"Dispersion transformed feature count: {len(disp_feature_names):,}")

disp_ridge = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=DISP_CV, scoring="r2")
disp_ridge.fit(X_disp_train_pp, y_disp_train)

best_disp_xgb = xgb.XGBRegressor(
    **disp_xgb_study.best_params,
    objective="reg:squarederror",
    eval_metric="rmse",
    verbosity=0,
    random_state=DISP_RANDOM_STATE,
    n_jobs=-1,
)
best_disp_xgb.fit(X_disp_train_pp, y_disp_train)

best_disp_rf = RandomForestRegressor(
    **disp_rf_study.best_params,
    random_state=DISP_RANDOM_STATE,
    n_jobs=-1,
)
best_disp_rf.fit(X_disp_train_pp, y_disp_train)

disp_final = pd.DataFrame([
    regression_metrics("Ridge (baseline)", y_disp_test, disp_ridge.predict(X_disp_test_pp)),
    regression_metrics("XGBoost (tuned)", y_disp_test, best_disp_xgb.predict(X_disp_test_pp)),
    regression_metrics("Random Forest (tuned)", y_disp_test, best_disp_rf.predict(X_disp_test_pp)),
]).set_index("Model").sort_values("R2", ascending=False)

joblib.dump(disp_pre_fitted, TUNING_CACHE_DIR / f"pay_dispersion_preprocessor_{STUDY_VERSION}.joblib")
joblib.dump(disp_ridge, TUNING_CACHE_DIR / f"pay_dispersion_ridge_model_{STUDY_VERSION}.joblib")
joblib.dump(best_disp_xgb, TUNING_CACHE_DIR / f"pay_dispersion_xgb_model_{STUDY_VERSION}.joblib")
joblib.dump(best_disp_rf, TUNING_CACHE_DIR / f"pay_dispersion_rf_model_{STUDY_VERSION}.joblib")
disp_final.to_csv(TUNING_CACHE_DIR / f"pay_dispersion_test_metrics_{STUDY_VERSION}.csv")

print("Test-set performance: pay dispersion")
display(disp_final.round(4))
print(f"Saved pay-dispersion models and metrics to: {TUNING_CACHE_DIR}")

---

<a id="unsupervised-learning"></a>
## 9. Unsupervised Learning

The causal-inference section asks whether CEO turnover changes compensation, and the supervised-learning section asks how well observable features predict compensation outcomes. This section asks a different, exploratory question: do firm-year observations naturally form recognizable compensation profiles?

In plain language, a compensation profile is a group of firm-years that look similar in how CEOs are paid and how pay is distributed inside the executive team. For example, one group may contain large firms with high CEO pay and moderate pay dispersion, while another may contain firms with compressed executive pay after CEO turnover. These groups are descriptive patterns, not causal effects.

<a id="unsupervised-learning-objective"></a>
### 9.1 Objective

The objective is to explore whether European firms in the final analysis panel form interpretable compensation types. Unlike supervised learning, there is no target variable to predict. Variables that were outcomes in the supervised section, such as `log_w_real_pay` and `log_pay_dispersion`, are intentionally included here because they help define the compensation profile itself.

The first two steps are therefore simple and transparent: define the clustering question, then build the firm-year sample and feature list used for PCA and clustering.

<a id="unsupervised-learning-sample"></a>
### 9.2 Clustering Sample

The clustering sample starts from `df_panel_final`, the same analysis-ready firm-year table used in the causal and supervised-learning sections. We keep identifiers only for interpretation, not as clustering inputs. The feature set combines compensation outcomes, firm fundamentals, CEO demographics, and turnover indicators.

This section deliberately builds a separate unsupervised feature set instead of reusing the fitted supervised-learning preprocessor. The supervised pipeline excluded compensation outcomes to avoid target leakage. Here, compensation outcomes are part of the profile we want to describe. Preprocessing for PCA and clustering will be fitted separately in the next step.

In [ ]:
# -- 9.2  Build the unsupervised-learning sample --------------------------
unsup_id_columns = [
    "companyid", "companyname", "isin", "directorid", "directorname_emp",
    "annualreportyear",
]

unsup_numeric_features = [
    "log_w_real_pay",
    "log_pay_dispersion",
    "log_real_assets_eur_2015",
    "roa",
    "leverage",
    "age",
    "ceo_turnover",
    "post_turnover",
]

unsup_categorical_features = [
    "gender",
    "hocountryname",
    "sector",
]

unsup_profile_required = [
    "log_w_real_pay",
    "log_pay_dispersion",
]

unsup_candidate_columns = (
    unsup_id_columns
    + unsup_numeric_features
    + unsup_categorical_features
)
unsup_candidate_columns = list(dict.fromkeys(unsup_candidate_columns))

unsup_available_columns = [
    col for col in unsup_candidate_columns
    if col in df_panel_final.columns
]
unsup_missing_columns = [
    col for col in unsup_candidate_columns
    if col not in df_panel_final.columns
]

unsup_required_available = [
    col for col in unsup_profile_required
    if col in df_panel_final.columns
]

if len(unsup_required_available) != len(unsup_profile_required):
    missing_required = sorted(set(unsup_profile_required) - set(unsup_required_available))
    raise KeyError(
        "Missing required unsupervised-learning profile column(s): "
        + ", ".join(missing_required)
    )

unsup_sample = df_panel_final.loc[:, unsup_available_columns].copy()
unsup_sample = unsup_sample.dropna(subset=unsup_required_available).copy()

unsup_numeric_features = [
    col for col in unsup_numeric_features
    if col in unsup_sample.columns
]
unsup_categorical_features = [
    col for col in unsup_categorical_features
    if col in unsup_sample.columns
]
unsup_model_features = unsup_numeric_features + unsup_categorical_features

unsup_feature_coverage = (
    unsup_sample[unsup_model_features]
    .isna()
    .sum()
    .to_frame("missing")
)
unsup_feature_coverage["missing_pct"] = (
    unsup_feature_coverage["missing"] / len(unsup_sample) * 100
).round(2)
unsup_feature_coverage["dtype"] = unsup_sample[unsup_model_features].dtypes.astype(str)
unsup_feature_coverage = unsup_feature_coverage.sort_values(
    ["missing_pct", "missing"],
    ascending=False,
)

unsup_sample_summary = pd.DataFrame({
    "value": {
        "rows": len(unsup_sample),
        "unique_firms": unsup_sample["companyid"].nunique() if "companyid" in unsup_sample else np.nan,
        "first_year": unsup_sample["annualreportyear"].min() if "annualreportyear" in unsup_sample else np.nan,
        "last_year": unsup_sample["annualreportyear"].max() if "annualreportyear" in unsup_sample else np.nan,
        "numeric_features": len(unsup_numeric_features),
        "categorical_features": len(unsup_categorical_features),
        "missing_candidate_columns": len(unsup_missing_columns),
    }
})

print("Unsupervised-learning sample prepared.")
print(f"Rows retained: {len(unsup_sample):,}")
print(f"Features selected: {len(unsup_model_features)}")

if unsup_missing_columns:
    print("Candidate columns not available in df_panel_final:")
    print(unsup_missing_columns)

display(unsup_sample_summary)
display(unsup_feature_coverage)
display(unsup_sample[unsup_id_columns + unsup_model_features].head(5))

<a id="unsupervised-learning-preprocessing"></a>
### 9.3 Preprocessing for PCA and Clustering

This step applies the same preprocessing principles used in supervised learning, but with a new unsupervised pipeline and a different feature set. Numeric variables are median-imputed and scaled so that firm size, profitability, leverage, pay, and age are comparable in distance-based methods. Categorical variables are imputed and one-hot encoded so country, sector, and gender can contribute to the clustering representation without imposing arbitrary numeric order.

Because this section is exploratory rather than predictive, the transformer is fitted on the full clustering sample. The transformed matrix produced here is the input for PCA and clustering.

In [ ]:
# -- 9.3  Preprocess features for PCA and clustering ----------------------
unsup_X = unsup_sample[unsup_model_features].copy().astype(object)
unsup_X = unsup_X.where(pd.notna(unsup_X), np.nan)

unsup_numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

unsup_categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

unsup_preprocessor = ColumnTransformer(
    transformers=[
        ("num", unsup_numeric_transformer, unsup_numeric_features),
        ("cat", unsup_categorical_transformer, unsup_categorical_features),
    ],
    remainder="drop",
)

X_unsup_preprocessed = unsup_preprocessor.fit_transform(unsup_X)
unsup_feature_names = unsup_preprocessor.get_feature_names_out()

# PCA and plotting are easier with a dense matrix. The current feature count is modest.
if hasattr(X_unsup_preprocessed, "toarray"):
    X_unsup_preprocessed_dense = X_unsup_preprocessed.toarray()
else:
    X_unsup_preprocessed_dense = np.asarray(X_unsup_preprocessed)

unsup_preprocess_summary = pd.DataFrame({
    "value": {
        "rows": X_unsup_preprocessed_dense.shape[0],
        "features_after_preprocessing": X_unsup_preprocessed_dense.shape[1],
        "numeric_input_features": len(unsup_numeric_features),
        "categorical_input_features": len(unsup_categorical_features),
        "missing_after_preprocessing": int(np.isnan(X_unsup_preprocessed_dense).sum()),
    }
})

unsup_preprocessed_preview = pd.DataFrame(
    X_unsup_preprocessed_dense[:5, : min(15, X_unsup_preprocessed_dense.shape[1])],
    columns=unsup_feature_names[: min(15, len(unsup_feature_names))],
)

print("Unsupervised preprocessing fitted on the full clustering sample.")
display(unsup_preprocess_summary)
display(unsup_preprocessed_preview.round(3))

<a id="unsupervised-learning-pca"></a>
### 9.4 PCA Dimensionality Reduction

The preprocessed clustering matrix has many columns because categorical variables are one-hot encoded. PCA reduces this matrix into a smaller set of orthogonal components that summarize the main directions of variation in the compensation profiles.

PCA is not the clustering method itself. It is an interpretability and visualization step. The explained-variance table shows how much information each component captures, the loading table shows which original features contribute most to the first two components, and the PC1/PC2 plot gives a two-dimensional view of the compensation-profile space before clustering.

In [ ]:
# -- 9.4  PCA dimensionality reduction ------------------------------------
from sklearn.decomposition import PCA

unsup_pca = PCA()
unsup_pca_scores = unsup_pca.fit_transform(X_unsup_preprocessed_dense)

unsup_pca_columns = [
    f"PC{i}"
    for i in range(1, unsup_pca_scores.shape[1] + 1)
]
unsup_pca_df = pd.DataFrame(
    unsup_pca_scores,
    columns=unsup_pca_columns,
    index=unsup_sample.index,
)

unsup_pca_plot_df = pd.concat(
    [
        unsup_sample[[col for col in unsup_id_columns if col in unsup_sample.columns]].reset_index(drop=True),
        unsup_sample[["log_w_real_pay", "log_pay_dispersion"]].reset_index(drop=True),
        unsup_pca_df[["PC1", "PC2"]].reset_index(drop=True),
    ],
    axis=1,
)

unsup_pca_variance = pd.DataFrame({
    "component": unsup_pca_columns,
    "explained_variance_ratio": unsup_pca.explained_variance_ratio_,
    "cumulative_explained_variance": np.cumsum(unsup_pca.explained_variance_ratio_),
})

unsup_pca_loadings = pd.DataFrame(
    unsup_pca.components_.T,
    index=unsup_feature_names,
    columns=unsup_pca_columns,
)

pc1_top_features = unsup_pca_loadings["PC1"].abs().sort_values(ascending=False).head(10).index
pc2_top_features = unsup_pca_loadings["PC2"].abs().sort_values(ascending=False).head(10).index

pc_loading_summary = pd.DataFrame({
    "PC1_feature": pc1_top_features,
    "PC1_loading": unsup_pca_loadings.loc[pc1_top_features, "PC1"].values,
    "PC2_feature": pc2_top_features,
    "PC2_loading": unsup_pca_loadings.loc[pc2_top_features, "PC2"].values,
})

print("PCA fitted on the preprocessed unsupervised feature matrix.")
print(
    "Variance explained by PC1 and PC2: "
    f"{unsup_pca_variance.loc[1, 'cumulative_explained_variance']:.1%}"
)

display(unsup_pca_variance.head(10).round(4))
display(pc_loading_summary.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

n_show = min(10, len(unsup_pca_variance))
axes[0].bar(
    unsup_pca_variance["component"].head(n_show),
    unsup_pca_variance["explained_variance_ratio"].head(n_show),
    color="steelblue",
    alpha=0.8,
)
axes[0].plot(
    unsup_pca_variance["component"].head(n_show),
    unsup_pca_variance["cumulative_explained_variance"].head(n_show),
    color="darkorange",
    marker="o",
)
axes[0].set_title("PCA Explained Variance")
axes[0].set_xlabel("Principal component")
axes[0].set_ylabel("Share of variance")
axes[0].tick_params(axis="x", rotation=45)
axes[0].set_ylim(0, min(1.05, max(0.2, unsup_pca_variance["cumulative_explained_variance"].head(n_show).max() + 0.1)))

scatter = axes[1].scatter(
    unsup_pca_plot_df["PC1"],
    unsup_pca_plot_df["PC2"],
    c=unsup_pca_plot_df["log_pay_dispersion"],
    cmap="viridis",
    alpha=0.65,
    s=22,
)
axes[1].axhline(0, color="black", linewidth=0.8, alpha=0.4)
axes[1].axvline(0, color="black", linewidth=0.8, alpha=0.4)
axes[1].set_title("Firm-Year Compensation Profiles in PC Space")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
colorbar = fig.colorbar(scatter, ax=axes[1])
colorbar.set_label("log pay dispersion")

plt.tight_layout()
plt.show()

<a id="unsupervised-learning-cluster-selection"></a>
### 9.5 Choosing the Number of Clusters

Before assigning firms to clusters, we need a defensible number of groups. This subsection compares K-Means solutions for several values of `k` using two diagnostics.

The **elbow curve** reports inertia, the within-cluster sum of squared distances. Lower inertia is better, but it always decreases as more clusters are added, so we look for the point where the improvement starts to flatten. The **silhouette score** measures how separated the clusters are, with higher values indicating better-defined groups. These diagnostics guide the choice of `k` used for the final clustering model and profile interpretation.

In [ ]:
# -- 9.5  Choose the number of K-Means clusters --------------------------
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

KMEANS_RANDOM_STATE = 654321
cluster_k_values = range(2, 9)
cluster_selection_rows = []

for k in cluster_k_values:
    kmeans_model = KMeans(
        n_clusters=k,
        random_state=KMEANS_RANDOM_STATE,
        n_init=25,
    )
    cluster_labels = kmeans_model.fit_predict(X_unsup_preprocessed_dense)
    silhouette = silhouette_score(X_unsup_preprocessed_dense, cluster_labels)
    cluster_selection_rows.append({
        "k": k,
        "inertia": kmeans_model.inertia_,
        "silhouette_score": silhouette,
    })

unsup_cluster_selection = pd.DataFrame(cluster_selection_rows)
unsup_suggested_k = int(
    unsup_cluster_selection.sort_values(
        ["silhouette_score", "k"],
        ascending=[False, True],
    )["k"].iloc[0]
)

print("K-Means cluster-count diagnostics:")
display(unsup_cluster_selection.round(4))
print(f"Highest silhouette score suggests k = {unsup_suggested_k}.")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(
    unsup_cluster_selection["k"],
    unsup_cluster_selection["inertia"],
    marker="o",
    color="steelblue",
)
axes[0].set_title("Elbow Diagnostic")
axes[0].set_xlabel("Number of clusters (k)")
axes[0].set_ylabel("Inertia")
axes[0].set_xticks(list(cluster_k_values))

axes[1].plot(
    unsup_cluster_selection["k"],
    unsup_cluster_selection["silhouette_score"],
    marker="o",
    color="darkorange",
)
axes[1].axvline(
    unsup_suggested_k,
    color="black",
    linestyle="--",
    linewidth=1,
    alpha=0.6,
)
axes[1].set_title("Silhouette Diagnostic")
axes[1].set_xlabel("Number of clusters (k)")
axes[1].set_ylabel("Silhouette score")
axes[1].set_xticks(list(cluster_k_values))

plt.tight_layout()
plt.show()

<a id="unsupervised-learning-final-clusters"></a>
### 9.6 Final Clusters and Compensation Profiles

Using the diagnostic choice from the previous subsection, we now fit the final K-Means model and attach each firm-year observation to a compensation-profile cluster. The goal is descriptive: each cluster should be read as a group of observations with similar pay level, pay dispersion, firm fundamentals, executive demographics, and turnover status.

The summary tables below translate the cluster labels into interpretable profiles. The first table shows cluster size and key averages. The second compares each cluster with the full unsupervised sample, so positive values mean the cluster is above the sample average and negative values mean it is below. The final plot shows the clusters in the first two PCA dimensions.

In [ ]:
# -- 9.6  Fit final K-Means model and profile clusters --------------------
FINAL_CLUSTER_K = unsup_suggested_k

unsup_final_kmeans = KMeans(
    n_clusters=FINAL_CLUSTER_K,
    random_state=KMEANS_RANDOM_STATE,
    n_init=50,
)
unsup_cluster_labels = unsup_final_kmeans.fit_predict(X_unsup_preprocessed_dense)

unsup_clustered = unsup_sample.copy()
unsup_clustered["cluster"] = unsup_cluster_labels
unsup_clustered["cluster_label"] = "Cluster " + unsup_clustered["cluster"].astype(str)
unsup_clustered["PC1"] = unsup_pca_df.loc[unsup_sample.index, "PC1"].values
unsup_clustered["PC2"] = unsup_pca_df.loc[unsup_sample.index, "PC2"].values

cluster_count_summary = (
    unsup_clustered
    .groupby("cluster_label")
    .agg(
        rows=("cluster", "size"),
        unique_firms=("companyid", "nunique"),
        first_year=("annualreportyear", "min"),
        last_year=("annualreportyear", "max"),
    )
)
cluster_count_summary["row_share"] = (
    cluster_count_summary["rows"] / len(unsup_clustered)
).round(3)

profile_numeric_cols = [
    "log_w_real_pay",
    "log_pay_dispersion",
    "log_real_assets_eur_2015",
    "roa",
    "leverage",
    "age",
    "ceo_turnover",
    "post_turnover",
]
profile_numeric_cols = [
    col for col in profile_numeric_cols
    if col in unsup_clustered.columns
]

# Some notebook runs keep BoardEx/financial columns as pandas string dtype.
# Coerce the profiling columns here so group means are calculated safely.
unsup_profile_numeric = unsup_clustered[profile_numeric_cols].apply(
    pd.to_numeric,
    errors="coerce",
)
unsup_profile_numeric["cluster_label"] = unsup_clustered["cluster_label"]

cluster_profile = (
    unsup_profile_numeric
    .groupby("cluster_label")[profile_numeric_cols]
    .mean()
    .round(3)
)

sample_profile_mean = unsup_profile_numeric[profile_numeric_cols].mean()
cluster_profile_vs_sample = (
    cluster_profile - sample_profile_mean
).round(3)

cluster_top_categories = []
for cluster_label, cluster_df in unsup_clustered.groupby("cluster_label"):
    row = {"cluster_label": cluster_label}
    for col in unsup_categorical_features:
        if col in cluster_df.columns:
            top_value = cluster_df[col].mode(dropna=True)
            row[f"top_{col}"] = top_value.iloc[0] if not top_value.empty else np.nan
            row[f"top_{col}_share"] = (
                cluster_df[col].eq(row[f"top_{col}"]).mean().round(3)
                if not pd.isna(row[f"top_{col}"])
                else np.nan
            )
    cluster_top_categories.append(row)
cluster_top_categories = pd.DataFrame(cluster_top_categories).set_index("cluster_label")

print(f"Final K-Means model fitted with k = {FINAL_CLUSTER_K} clusters.")
display(cluster_count_summary)
display(cluster_profile)
display(cluster_profile_vs_sample)
display(cluster_top_categories)

fig, ax = plt.subplots(figsize=(8, 6))
for cluster_label, cluster_df in unsup_clustered.groupby("cluster_label"):
    ax.scatter(
        cluster_df["PC1"],
        cluster_df["PC2"],
        label=cluster_label,
        alpha=0.65,
        s=24,
    )
ax.axhline(0, color="black", linewidth=0.8, alpha=0.4)
ax.axvline(0, color="black", linewidth=0.8, alpha=0.4)
ax.set_title("K-Means Compensation Profiles in PCA Space")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend(title="Cluster", frameon=False)
plt.tight_layout()
plt.show()

print("Interpretation guide:")
print("- Use cluster_profile to describe the level of each cluster.")
print("- Use cluster_profile_vs_sample to describe whether a cluster is above or below the sample average.")
print("- These clusters are descriptive compensation profiles, not causal estimates.")

<a id="unsupervised-learning-interpretation"></a>
### 9.7 Cluster Interpretation

The three-cluster solution separates the panel into distinct compensation profiles rather than causal treatment effects.

- **Cluster 0: established post-turnover, high-pay profile.** This is the largest group, covering about two thirds of the unsupervised sample. It has above-average real CEO pay, slightly above-average pay dispersion, larger firm size, and `post_turnover` equal to 1 on average. These observations look like mature post-turnover firm-years where compensation has settled into a relatively high-pay regime.
- **Cluster 1: smaller, pre-turnover profile.** This group has below-average firm size, slightly below-average CEO pay, higher leverage, and `post_turnover` equal to 0 on average. It captures smaller or less mature compensation settings before the first observed turnover event.
- **Cluster 2: turnover-year compression profile.** This group has `ceo_turnover` equal to 1 on average, lower CEO pay, and lower pay dispersion than the full sample. It is the most directly aligned with the causal evidence: turnover years appear as a separate compensation regime with compressed CEO-to-executive pay gaps.

Overall, the first clustering solution is useful as a descriptive view of turnover stages, not as independent causal evidence. It shows that observations marked by CEO turnover can be separated from other firm-years when the turnover indicators are included, but the robustness check below is needed to see whether compensation and firm fundamentals alone produce the same structure.

*Caveat (see Section 9.8).* Because `ceo_turnover` and `post_turnover` are part of the clustering feature set, these three clusters largely coincide with the three turnover stages. The robustness check in Section 9.8 re-runs the clustering without those two indicators to test whether the profiles survive on compensation and firm fundamentals alone — and finds that the clusters then reorganise around firm size and pay dispersion.

<a id="unsupervised-learning-robustness"></a>
### 9.8 Robustness: Clustering Without Turnover Indicators

In Section 9.6 the three clusters separate almost perfectly on `ceo_turnover` and `post_turnover`: their per-cluster means are essentially 0 or 1, so each cluster is one turnover *stage* (pre-turnover, turnover year, established post-turnover). Because those two variables are engineered indicators of a firm-year's position relative to the turnover event, K-Means can latch onto them and recover the turnover stages rather than independent *compensation* types.

This robustness check refits the same recipe (preprocessing, k-selection, final K-Means) on a feature set that **excludes the two turnover indicators**, leaving only compensation outcomes, firm fundamentals, and executive demographics. It then cross-tabulates the new clusters against the original turnover state to test whether turnover timing re-emerges on its own:

- If each new cluster stays concentrated in a single turnover state (high *modal-state share*), turnover timing dominates the compensation structure even when it is not a clustering feature, and the 9.6 profiles are not merely an artefact of including the dummies.
- If the new clusters mix across turnover states, then genuinely independent compensation profiles exist once the turnover dummies are removed.

The cell prints the no-turnover k-selection diagnostics, the cluster profiles, the cluster-by-turnover-state cross-tab with a per-cluster purity score, and an automated verdict. The side-by-side PCA plot shows the new clusters next to the same points colored by turnover state.

In [ ]:
# -- 9.8  Robustness: cluster without the turnover indicators -------------
# Same recipe as 9.3-9.6, but the engineered turnover dummies are dropped
# from the feature set so the grouping reflects compensation outcomes, firm
# fundamentals, and demographics only. Reuses objects defined in 9.2-9.6.

noturn_numeric_features = [
    col for col in unsup_numeric_features
    if col not in ("ceo_turnover", "post_turnover")
]
noturn_model_features = noturn_numeric_features + unsup_categorical_features

noturn_X = unsup_sample[noturn_model_features].copy().astype(object)
noturn_X = noturn_X.where(pd.notna(noturn_X), np.nan)

noturn_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), noturn_numeric_features),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), unsup_categorical_features),
    ],
    remainder="drop",
)

X_noturn = noturn_preprocessor.fit_transform(noturn_X)
if hasattr(X_noturn, "toarray"):
    X_noturn = X_noturn.toarray()
else:
    X_noturn = np.asarray(X_noturn)

# k-selection on the same grid and diagnostics as 9.5
noturn_selection_rows = []
for k in cluster_k_values:
    km = KMeans(n_clusters=k, random_state=KMEANS_RANDOM_STATE, n_init=25)
    labels_k = km.fit_predict(X_noturn)
    noturn_selection_rows.append({
        "k": k,
        "inertia": km.inertia_,
        "silhouette_score": silhouette_score(X_noturn, labels_k),
    })
noturn_selection = pd.DataFrame(noturn_selection_rows)
noturn_k = int(
    noturn_selection
    .sort_values(["silhouette_score", "k"], ascending=[False, True])["k"]
    .iloc[0]
)

# Final K-Means on the no-turnover feature set
noturn_kmeans = KMeans(n_clusters=noturn_k, random_state=KMEANS_RANDOM_STATE, n_init=50)
noturn_labels = noturn_kmeans.fit_predict(X_noturn)

noturn_clustered = unsup_sample.copy()
noturn_clustered["cluster_label"] = ["Cluster " + str(lab) for lab in noturn_labels]

# Profile WITH the turnover columns included, to see whether they still separate
noturn_profile_numeric = noturn_clustered[profile_numeric_cols].apply(pd.to_numeric, errors="coerce")
noturn_profile_numeric["cluster_label"] = noturn_clustered["cluster_label"]
noturn_profile = (
    noturn_profile_numeric
    .groupby("cluster_label")[profile_numeric_cols]
    .mean()
    .round(3)
)

# Reconstruct the original turnover state and cross-tab it against the new clusters
ceo_state = pd.to_numeric(noturn_clustered["ceo_turnover"], errors="coerce").fillna(-1).astype(int)
post_state = pd.to_numeric(noturn_clustered["post_turnover"], errors="coerce").fillna(-1).astype(int)
noturn_clustered["turnover_state"] = (
    "ceo=" + ceo_state.astype(str) + " / post=" + post_state.astype(str)
).to_numpy()

state_crosstab = pd.crosstab(
    noturn_clustered["cluster_label"],
    noturn_clustered["turnover_state"],
)
cluster_purity = (state_crosstab.max(axis=1) / state_crosstab.sum(axis=1)).round(3)
mean_purity = float(cluster_purity.mean())

print(f"Clustering WITHOUT turnover indicators: best silhouette at k = {noturn_k}.")
print(
    f"Reference 9.5 (WITH turnover dummies): k = {FINAL_CLUSTER_K}, "
    f"best silhouette = {unsup_cluster_selection['silhouette_score'].max():.3f}."
)
print(f"No-turnover best silhouette = {noturn_selection['silhouette_score'].max():.3f}.")
print()

display(noturn_selection.round(4))
display(noturn_profile)
print("New clusters vs. original turnover state (row counts):")
display(state_crosstab)
display(cluster_purity.to_frame("modal_state_share"))

share_txt = f"mean modal-state share {mean_purity:.2f}"
if mean_purity > 0.80:
    verdict = (
        f"Clusters stay highly concentrated in single turnover states ({share_txt}). "
        "Turnover timing still dominates the compensation structure even when it is "
        "not a clustering feature, so the 9.6 profiles are not merely an artefact of "
        "including the dummies."
    )
elif mean_purity < 0.55:
    verdict = (
        f"Clusters mix across turnover states ({share_txt}). Independent compensation "
        "profiles emerge once the turnover dummies are removed, which means the 9.6 "
        "solution was driven by the engineered indicators."
    )
else:
    verdict = (
        f"Clusters are only partly aligned with turnover states ({share_txt}). Both "
        "compensation fundamentals and turnover timing shape the grouping."
    )
print(verdict)

# Side-by-side PCA view: new clusters vs. the same points colored by turnover state
noturn_pca = PCA(n_components=2)
noturn_scores = noturn_pca.fit_transform(X_noturn)
noturn_plot = pd.DataFrame({
    "PC1": noturn_scores[:, 0],
    "PC2": noturn_scores[:, 1],
    "cluster_label": noturn_clustered["cluster_label"].values,
    "turnover_state": noturn_clustered["turnover_state"].values,
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)
for label, grp in noturn_plot.groupby("cluster_label"):
    axes[0].scatter(grp["PC1"], grp["PC2"], label=label, alpha=0.65, s=22)
axes[0].set_title("No-Turnover K-Means Clusters (PCA space)")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].legend(title="Cluster", frameon=False)
for state, grp in noturn_plot.groupby("turnover_state"):
    axes[1].scatter(grp["PC1"], grp["PC2"], label=state, alpha=0.65, s=22)
axes[1].set_title("Same Points Colored by Turnover State")
axes[1].set_xlabel("PC1")
axes[1].legend(title="Turnover state", frameon=False)
plt.tight_layout()
plt.show()


**Reading the robustness check (9.8).** Dropping the two turnover indicators changes the clustering substantially. The best silhouette shifts from `k = 3` with the dummies to `k = 6` without them, and the best silhouette falls from 0.196 to 0.147. This means the turnover indicators had been inflating the apparent separation in the original three-cluster solution.

The six no-turnover clusters are organised mainly by firm size, pay level, pay dispersion, leverage, and CEO age rather than by one clean turnover stage:

- **Cluster 0:** moderate-size, low-dispersion firms with relatively high leverage.
- **Cluster 1:** lower-pay, low-dispersion firms with the highest turnover-year share among the robustness clusters.
- **Cluster 2:** very large firms with high CEO pay and moderate pay dispersion.
- **Cluster 3:** older-CEO firm-years with mid-level pay and dispersion.
- **Cluster 4:** high-dispersion firm-years, where the CEO-to-executive pay gap is the defining feature.
- **Cluster 5:** smaller, profitable firms with low leverage and moderate pay dispersion.

Turnover timing is now only **partly aligned** with the clusters. The per-cluster `ceo_turnover` and `post_turnover` means are fractional rather than exact 0/1 stage labels, and the cross-tab has a mean modal-state share of about 0.68. The automated verdict therefore reads: both compensation fundamentals and turnover timing shape the grouping.

**Implication.** The clean "turnover-stage" profiles in Section 9.6 depend strongly on including the engineered turnover indicators in the feature set. Once those indicators are removed, the structure is more diffuse and is better described by gradients in **firm size, pay level, and internal pay dispersion**. This does not weaken the causal evidence in Section 7: it clarifies that the unsupervised view characterises broad firm-year profiles, whereas the compression of pay dispersion around turnover is a within-firm effect that pooled cluster analysis is not designed to isolate.

---

<a id="results-and-discussion"></a>
## 10. Results and Discussion

The preceding sections build the analysis panel, visualize the main outcomes, estimate causal effects, benchmark supervised-learning models, and identify unsupervised compensation profiles. This final section draws the evidence together around the research question.

<a id="discussion-conclusion"></a>
### 10.1 Discussion & Conclusion

#### Research question
*Does CEO turnover causally change CEO compensation levels and pay dispersion within European firms?*

---

#### 10.1.1 Causal Inference

**CEO pay level (Section 7.1)**

The difference-in-differences regression yields a post-turnover coefficient of **−0.0595** (95 % CI [−0.1452, +0.0262], p = 0.1734), which is not statistically significant at the 5 % level. On average, a firm that experienced CEO turnover does not pay its replacement materially differently once firm fundamentals (ROA, log assets, leverage, country, sector) and year fixed effects are controlled for.

The event study refines this picture. The coefficient at τ = 0 is **−0.5318** (95 % CI [−0.7852, −0.2784]), a large and significant drop concentrated in the turnover year itself. Coefficients at τ = −3 and τ = −2 are small and span zero, supporting the parallel-trends assumption. The effect fades rapidly: τ = +1 (−0.1673) and τ = +2 (−0.1506) are both insignificant. The pattern is consistent with incoming CEOs accepting a transitional discount in the year they take the role — or with boards temporarily constraining pay during leadership transitions — after which compensation returns to a level driven by firm fundamentals.

**Pay dispersion (Section 7.2)**

The parallel DiD for log pay dispersion (CEO-to-median-executive pay ratio) tells a different story. The post-turnover coefficient is **−0.3386** (95 % CI [−0.5124, −0.1647], p = 0.0001), with N = 1,310 firm-years across 124 firms. This is a large and highly significant effect: CEO turnover is associated with a sustained compression of the within-firm pay hierarchy. The event-study coefficients are all negative after τ = 0, though wide confidence intervals (driven by the smaller dispersion sample) prevent individual significance. Pre-trend coefficients at τ = −3 and τ = −2 are close to zero, consistent with the identifying assumption.

Together the two causal estimates answer the research question asymmetrically: turnover produces a transitory and statistically weak change in *pay levels* but a significant *compression of pay dispersion*. New CEOs are not paid radically differently from their predecessors in steady state, but the internal pay hierarchy appears to reset after a leadership change.

---

#### 10.1.2 Supervised Learning

**CEO pay level — regression (Section 8.6)**

The supervised CEO-pay models benchmark how predictable compensation is from observable firm, executive, governance, country, sector, and year features. In the final tuned comparison, XGBoost performs best on the held-out test period with **R² = 0.4133**, RMSE = 0.5034, ahead of tuned Random Forest (R² = 0.3923) and RidgeCV (R² = 0.3064). The result shows that observable features explain a meaningful but incomplete share of CEO-pay variation. Roughly 59 % remains outside the best model, consistent with board discretion, contract details, and private governance information that are not observed in the dataset.

**CEO turnover — classification (Section 8.7)**

Predicting which firm-years experience CEO turnover is substantially harder. The best classification results are only modestly above chance: XGBoost reaches ROC-AUC = 0.629, average precision = 0.172, balanced accuracy = 0.598, and MCC = 0.124; Logistic Regression is very similar but slightly weaker. This suggests that CEO turnover is not easily predicted from the available financial and executive features. That is plausible because succession decisions often depend on board dynamics, personal factors, strategy disputes, activist pressure, or private performance assessments.

**Pay dispersion — regression (Section 8.8)**

For `log_pay_dispersion`, tuned Random Forest leads the final test-set comparison with **R² = 0.4037**, RMSE = 0.7883, ahead of tuned XGBoost (R² = 0.3199) and RidgeCV (R² = 0.1273). Pay dispersion is therefore roughly as predictable as pay levels, but the stronger performance of tree ensembles indicates non-linear structure: firm size, sector, profitability, and turnover-related variables interact in ways that a linear model cannot fully capture.

---

#### 10.1.3 Unsupervised Learning

The unsupervised-learning analysis provides a descriptive view of compensation profiles in the final panel. With the turnover indicators included, the highest silhouette score points to **k = 3**, and the clusters largely separate pre-turnover, turnover-year, and established post-turnover firm-years. These groups are useful for interpretation, but they should not be read as treatment-effect estimates.

**Cluster 0** is the largest group, covering about 68 % of the unsupervised sample. It has above-average real CEO pay, slightly above-average pay dispersion, larger firm size, and post-turnover observations. This profile looks like established post-turnover firm-years in which compensation has settled into a relatively high-pay regime.

**Cluster 1** contains about 19 % of the sample. It is characterized by smaller firms, slightly below-average CEO pay, higher leverage, and pre-turnover observations. This group captures lower-scale compensation settings before the first observed leadership change.

**Cluster 2** contains about 12 % of the sample and is the most directly related to the research question. It consists of turnover-year observations with lower CEO pay and lower pay dispersion than the full sample. The robustness check in Section 9.8 shows, however, that this clean turnover-stage structure depends strongly on including the engineered turnover indicators. Once those indicators are removed, the clusters become more diffuse and are organized mainly by firm size, pay level, and pay dispersion.

---

#### 10.1.4 What each lens reveals

| Lens | Key finding | What it adds |
|---|---|---|
| **Causal inference** | Turnover significantly compresses pay dispersion; the average pay-level effect is transitory and statistically weak | Estimates the treatment relationship while adjusting for firm characteristics and fixed effects |
| **Supervised learning** | Observable features explain about 40–41 % of pay and pay-dispersion variation, but turnover itself is difficult to predict | Quantifies predictability and shows that much compensation variation remains outside observable firm-year features |
| **Unsupervised learning** | Clusters with turnover indicators recover turnover stages; robustness without them shifts the structure toward firm size and pay dispersion | Shows that clustering is useful descriptively, but it is not independent causal evidence |

The three approaches tell a coherent but carefully layered story. Causal inference provides the strongest evidence that CEO turnover compresses internal pay dispersion. Supervised learning shows that pay and dispersion are only partly predictable from steady-state firm and executive features, leaving room for governance events and board-level decisions. Unsupervised learning adds descriptive context: turnover indicators help recover turnover-stage profiles, but without those indicators the natural cluster structure is driven more by firm size and pay dispersion. Together, the evidence suggests that CEO turnover matters less as a permanent shock to average CEO pay and more as a reset of the internal executive pay hierarchy.

---

#### 10.1.5 Limitations

- **Small treated sample**: roughly 127 firms experienced at least one CEO turnover in the main causal sample, limiting power for event-study estimates.
- **Incomplete dispersion data**: the pay-ratio variable requires at least two named executives with remuneration records in the same firm-year, reducing the dispersion sample to around 1,300 rows.
- **European disclosure gap**: European executive pay disclosure is less standardised than in the US. Compensation records are likely incomplete for smaller firms and earlier years, introducing measurement error that can attenuate estimates.
- **Endogeneity of turnover timing**: even with backdoor adjustment, firms that dismiss a CEO may differ from control firms on unobserved dimensions, such as board culture, activist pressure, or private performance information.
- **Clustering is descriptive**: the unsupervised profiles depend on feature choices, scaling, and the selected number of clusters. The turnover-stage profiles are especially sensitive to whether `ceo_turnover` and `post_turnover` are included as clustering features, so they help interpret patterns but do not identify causal effects.
- **Currency and inflation adjustment**: converting compensation and firm values to EUR and 2015 prices introduces FX and inflation uncertainty for non-Eurozone firms.

---

<a id="evaluation-strategy"></a>
## 11. Evaluation Strategy

The mission succeeds if the final notebook produces a transparent, reproducible panel and uses it to answer the research question from three angles:

- **Causal inference:** estimate whether CEO turnover is associated with changes in real 2015 EUR CEO pay after adjusting for firm characteristics.
- **Prediction:** benchmark models of real 2015 EUR compensation using RMSE/MAE and classification metrics where turnover or high-pay outcomes are modeled.
- **Unsupervised learning:** identify firm or executive clusters that reveal heterogeneous compensation patterns.
- **Robustness:** report missing-data coverage, sensitivity to winsorization, and whether results change across event windows.


---

<a id="work-plan"></a>
## 12. Work Plan

| Step | Owner | Description | Output | Status |
|---:|---|---|---|---|
| 1 | Achmad | Define the STOXX 600 universe and retrieve BoardEx/Compustat inputs | Raw source tables | Complete |
| 2 | Kajetan | Clean identifiers, dates, compensation variables, and missing values | Harmonized source tables | Complete |
| 3 | Achmad | Identify firm-level CEO spells and turnover events | `ceo_turnover_firm_year` | Complete |
| 4 | Kajetan | Match CEO remuneration to valid CEO spells | `ceo_pay_panel` | Complete |
| 5 | Achmad | Convert monetary inputs to EUR; add market capitalization and financial ratios | Nominal-EUR source tables and `df_funda` | Complete |
| 6 | Achmad + Kajetan | Merge inputs, apply CPI, and validate firm-year uniqueness | Real-2015-EUR `df_panel_final` | Complete |
| 7 | Kajetan | Implement causal and event-window analysis | Estimates and robustness checks | Complete |
| 8 | Achmad + Kajetan | Implement supervised and unsupervised learning | Model metrics, clusters, and heterogeneity | Complete |
| 9 | Achmad + Kajetan | Synthesize findings and limitations | Final discussion and conclusion | Complete |


---